# Kaggriculture X540: idle-fertilizer liquidation overlay

This notebook publishes the exact, self-contained X540 agent on top of the Nah I'd Win X492 base so other Kaggriculture participants can inspect and copy the technique.

## What changed

X540 preserves the X492 field route, crop mix, animal plan, expert selection, and existing market policy. Its only change is at hour 23: when fertilizer is at or above its base price and market capacity remains, it sells up to 12 idle fertilizer units after X492's planned sales and pickups. It does not use private opponent state, copy a replay tape, or alter field actions.

The released source is byte-pinned at SHA-256 `db2707d6a349521dd165de280038440cfcfdafc50a9739ffe1bb70d5b62dfdf7`. The notebook emits the executable entrypoint as `/kaggle/working/main.py` and validates that exact file before the competition submission path is used.

## Evidence and limits

- Fresh paired screen: 24 games, both seats, seeds 2260--2262; Starter was neutral, while X540 won all six paired cells against Adaptive V42, Rayk, and Salem, with mean margins `+$52`, `+$34`, and `+$6`.
- Disjoint confirmation: 80 games, both seats, seeds 2270--2279; Starter was `0/20/0`, Adaptive `20/0/0` at `+$65.6` mean, Rayk `20/0/0` at `+$57.1`, and Salem `20/0/0` at `+$14.4`, with identical terminal stock, animals, and weeds.
- These are controlled local results, not a guarantee of leaderboard improvement. The public rating is matchmaking-dependent and should be interpreted separately from the mechanism evidence.

The policy is intentionally copyable: download the notebook output `main.py`, or copy the displayed source cell, and run it with the Kaggriculture engine. Keep the source hash and the final `agent` entrypoint intact when submitting.


In [1]:
from __future__ import annotations

import base64
import hashlib
from pathlib import Path

EXPECTED_SHA256 = "db2707d6a349521dd165de280038440cfcfdafc50a9739ffe1bb70d5b62dfdf7"
SOURCE_B85 = """A|fJ7VP|J@X=8P4bairNAVo4bG%gAXLSb`dIv{jtWgv5PaBysCWn>_JGcYhPEpuaUa%CWGVRCC_bS-0VZe(e6X>V?2WFT#6cyx7gWi4-JEoFFcWpZ?LAaHMNX=8aV3PEIKWn>_1VR>b8Iv`;nV{dMAWpZ|5bZK^FAaiA9WG!NKWM^e`AZK-9a%3Q8Z*m}EbZ>2GV<1pWK~7X4c42I3WMOn^Z*DFM3PE&qa%p09bZKvHAZBuJZ6I`LWgv5Jb#h~6AZ~ATWnyn{YanTJAaHVJb7gXNWn?Z2A|fIRX>D+Ca&#bKVRL0RGzw{LaBp&SAY*TEc?xN5aBp&SAZl}OZVG8_aBp&SAZ=lEXbNd<aBp&SAbM<RVhRchUqM4uNl#8wAUz;zb8l`gY;R#?b0~UjX<{y9Wn*t`aB^jHb0}hAb7eL(E@C(}WMyM-WMwEPV=Z!PJWoDaGdOErLp)zzP;hxvUP?H4b#rVrU@JH@G&odfLriodF-tIgBWPDSSxk3*bUk(@G(#{mDRL@UVkLS}JTWnCFjPWPKutG$ZdFopUQ2CQb0KCza7%tmZCPYVBqStDPhK)MHgHIJKWSlQMRP|%J}Gf}Bs+0FN-=tQIekrhOf^GVdP*{2SaoJlG<#Zpc_B1jRc|OkOHz3-V_<wCX>dzzR46qmGC@#1S|ubkHZ5{<Nn}t{GFVD_Yf39^N?>JmdRKQ^N;E%YBt>^7M{+cCK0;a}XhAY2Ln=#RKv8@&ZclzzV{9u{C18DCVs1%9WJqQzSZ8Tya7Jh|Ej&tEa(zf4R7-kbeRL>IU`2LmDk*(TJZC>`b3Y_WC?hL)Mr3U>Hg7R9UPDx3Nh@YQYD!p2Ha$Ifbaz^QH$HP%dR}*VcQ7+YDs_4`XgOa^H7h=6C_Pj>Sa?`_bx2c2FhFZbRYiMzMKv@|cv3iLV{J%FOm97MPbzwNC{H;zDMfomCN@nYZ(~|CUqUHCbbD$fO?g^CT5f%HGc7wucxF^7N^X2jW<q*SK6X+gWNCd(W_B%CYB)keV^TmWDkLURY-UF^K}<hFM0!CjPklLcJ$OoXRCp?GQX@P#T3L22NGfAkQ%y%kd3$C!dN@BZSA1`5WoJNsIVx`^cS%Q3Au?t$eN|Xma&%}mYG^x4Dr7<}H(_#KBQ--aDs41EZ#6VYJYZ06Yb$0uPhWm0L^WqLEhRubZag7<HdrxMXLop3T46mzUQtOkbURc(cO)%eM>k15bVf&EYF=w&JwRc8b0sJ#CQC9<Nh3K<KSgb9N>4m%S#xD{Gh!rANOn{}R!&7JQdu%sHgjH9Ni%dfa%ntjEodQnJ4{w?ba6FtC_HR?U{q!(URrn~aZ6WuCUHw?dLbhvY)U3yM?7XuYcXX}ad0_TC46#wKRZKxBuacTDNSHnM>0GxKw2^}R5VmOdNo%=IZ91PHFQ8XJ#S}KC~-(LJu^{dcr<onS4d$sNML0~Xgq#CdSGT_Za-BpBzIsTO;}SZM@KRxW^Q>VaZXcJXm?9lFeYhMeN`=QG+<;SN?2z<P)AWsJYZKcXDK)-BtTbsKQ&`hQe<E@awSD3M{GAKLoF?HLsU&_ayEKjK5##FPFg%ACPhVVX(LK9Qa?9qT5u_IbxJ{QZbwdLJ#u|xY(!;5G(>V)PGoCQWI|<QK{qlpZE7?%OJy~6J#~0NKsj-HO-N8xXC`$sM>lIFEjeR(cPl$UXkUGISwng-BxO=HMkRHAVpB_GN_S9VK{9D0IaXLJK2t#<Bx6@oL3d1MG;>i;YjAcmJS|UDMmSX~bUju_M=g0iVpCseSyWj{KP_QkR%LWRXGA1NBq2v*Y-3J%Auvl;Kq_!qR$n1!Zbma=az9pnOlW*+aX5HRO=4kwX(n-AV{R*QPAyqALsNHjLQ_H^DPDIgMKNVWQduKactBSqK}30VQ%^!Ec06r*WoJiwNO5&GO?5PMD?oK9QE_u9Hg8#OW-B;FKr~NjB`7OjKt*{wUsi2kD{y2hOMFIUbUQO%HAh58by_JYHeqK*S3Nv(cWP`hZBt<;VrzOTb4Vd0aWgZ0D0W70Ay`vqOm1g*IZH%kVt9F9a8qV$MnN$&S#(V_aC>n^OM7`!ODaY<S5rtuHF|w#LnA{fLPvR5VQW2jcr+njb$2UsQEX{VPHaC_Fg9i>Dpzz-G)s9fL}MgoX+306S7TseVRw5=BvV8yD|szaNo_%KdSP={ZgxRxY)@uqJxqFVJV{YxB}IKSen@vcW-&-kF+WjhH#Bl^G;?NZEo&rsV^L5?J3LcOT1Po^GIL38PgG)dWl~;rS3_h$b6;U8YF|ZRN;6+;C1y@3Q7C0(HD6RwGkQ`=Lpdf?J|RtaJ!D`@OlVAFM{8v@b1*|nY9n+=Dspd6a%3YUK1wNQML>E^AyQv-b0#o+UM5FhH(+WjDP%=BURh6iJ1bLHR(f7jNKSZlV`eZvM_Dl_C~-F-MruhwNFyOZQbc}kQ*l~QKp`Y&Qf@g#K1XUYJR@{pb8c{VFiuNGc`8gKc5`uLQ#MjCQbk&JKwx@&Bxq7HB`9=jb#6>@S~+4WC?;1aI7l^bJ8xD`dt`kzQ&BijElEx&W;-@PQG8!PYiD3Xc{6xGBuaHoV_IfJN>fO2dOs;XVtGI&JZ^1ObXa6Nc|2)WCNf!9O;#j#J6C2QR5UF(Wm7XiD}HHubzVzrR!uojC2~C~ay=+BIc#ckXh3T@Rcl&rBvdeBNm@ZkKsaS1BWON5L?mxgRd*yNL|7qCS8h&IZCF@PS~yL8Ra#?4VNgv!a&KC2SV$&jbWTuuDQswSVO2msMSgrnKTuCbY&&3Ucs@*CH%c^nOe<z!dOIslBSv^aC{R;zUP?YOJ6TXqM?FqaT3}0KGb>ptIcR=uOeuMND`_TXIX*E%RX#yKPC-g$Dm-3oST{UIP&aZbGB{dDK0q{gUs7>CNK|-7V`E=mBt%I-Yi~kJUO`o9SSdqSeQ|3hEkrz2JVRnIMSe*kSw2%ubueN#NlqqhHcNg>d3;M)WkoYPP<&2iLv2q*P$OtOab9gzEg?rXFfCPbMMP(AW?)$}EkS5$IC@4XH9;vRLuNBYJVJYKJbZp_F-vGPP-jw3H*hU`NO^m4QBp;BG=3v)MJ079MtDysJ$6J<Kv6s-Q+QTZMr%P)Fk^K_N_i$fD|kU*OKB!db9g~WH%)VLdv+#0er{evV^t(zL}f=ab7pH~SVLNQYb`)%R(e7=OLc8jHD*L!G)G7(JZ(pHMQK`KO=o(1NKaNuNl`vxS5qxxVL@3oPF`{`F<&q=F(f!9AxvdoOiFk*VQ55FH+xBSCLu^nQ+IBDFm7IQP)ux6X?Rpcc1T4fdt*LjElMa^K}ly=Kqyp8RBv-bdo+0{RB2FjRaawJXHP|NC~{CZUS>FVN=HafPkMG)eJ~_HJ2-75Qa@rhbvZPBQeHh{YBDl5SZ!uGJV8%YcP2|hb17;=BX($5D{XpbJ9d61LUk=ENLL|2Z!2d$JV;+-D>gV&Nkw>iVK#0zbWKSnI4x;zJ7{ikQX^|&a5Ys+LvJ~FGIUlsbx=n(DkgI!YhyMrKUQRYS37(>VP!OSDQ7V-QATHSS|)E;ep4xHcx!b{V`*(pNF!!cSzl#&DrO}>cz0z>SuIc{cv&PQV@GT^dVFwSJ9k%JOmjOUP+n;%V{UY2R5(gYWi>Q8V{uJiW=u*rDP}EUdoxE#N;h^raX4jEJv=ovbSpVuYeFk=aXUReWGPEYY;j_DC~9zXY+-t4CSGf4W=<qCOe1JOSYSaUHg9NDLVR&CJUDo2bWLSpBY13MJ7;=xT4`%FSu%J|D_C_qFf(^pZ&M+2ekpT7bXQ?dPj^&3URXzLGb?;RJ!*4DPi1IzJ8O7;NozzSR9aJXV{ln>Pjp8%GAMFQG$>j(H*iEKeI{R2OE7*}S8`uZHD648eR6kbM|eR*Dl~aOD>7nncQkuEQ9?96StTiTb$UU4c{pJ;SbixjXlZ6*cWQHbY(HXsa!x5=c5X3bbxU(uLtcF$D<nQWMst2`KTUH-W-@zLD?e9QOm$2&MP_3mduC8WQ7BM3BUx;IW>rQ(N<&XQXIW=eQ&w42K_NywElebBer#uHLVI6nOKdeYZ!~*APi8|*b7M$aL@iWJPG2xuZ6rA%bzy8IHza6%K`U%fZ*yNwDS9MoJ6~#SSv`1ZS2rmwVqR-^Fi18;K|4kyc|CVQLr7*vEkJW-JYsTBDOgV=Qg%mAYkW9iJ1J~7XlpYxX=!bHR(nZyJ4;0>L1S=OSuk`rWm<hIUtu;!cS=1?Q!r^uSzjwuSTs^-Q%G%fNpe~$U^93=c34p+Mp0TxO(<_eK~_#EXC*B$S8Gi(CPXVaS88!VS6XaqQekgLMkzfvVn<D9DQ|0TNMBEDMR8AGQcWpWMMh^lZa*k-H)L^cNF`NmRw-0CQ#?&=HdaVLN^5g%bTd^>eq&%!GAL<JcqwLeR!C+^P&ILNO+9!vM|?6rFeWoJSyg&$OLIacH%Dh-CRs~AW<x$<Ic{-zWF%-jR&i)*PjpjfS3+t`O>!+&Ge~t+PAG3maz#{WCUz@IPi}fgPI7iJV?A0)US2p&RZVPsa4I`BRenu7XF_~-GGtVHHds+DEg@nwEhuJ9R!CYaZe?_OcSmPhM^GhxXjnyfO*e65T6<F_c~)3AOfzgPF=#_^GI42CT4X9JDQkOEIDBC-VQVr#P&|85Pjp&VSZp^|Ms6WuR$yx?LN+-jKXNj3Z&D#eNJ>E^OEXSraU@n~aztS>WHm^5M<qy1Whg{8R&Y6ZLNp~lb7yE{J$_a-CL?S?P<dcRRV!pkP$o4cFhf0hKXxcUQb}YZMQk`RN@-qxA$MX&SWG}FOh!y+Jtj+QNIO|oS1V&$X*pL!M@DdHIAT{MYi&(mAyRc^M^Z>%cp-Q<bW&1UN?3GabtNHIOIa&nbs<b-JVs$}S3N~_W;bOrF@1AxMNva~RbnN0O;TfWY%?`-J6L^ePb6S!C^cqkGBtfNZzUvqPH18_UUN%rO?!1ybVf6MRBlOSVLLV^A#o!;B_mjFRU;`vMtDdzXh3E*dp%xRJW*JAd|yo_N?v(lLpCN+GEz)cb3|_@OHNNWQA98|JX1VXF<EG1JV;Z1cWNnWF=|77Eq+-cPFiR;M^hnpDMWi=V=6sbPCQ{mYHCzzK1N0(XGnBxT5&W<Kqh!)aZWLDby`+!NNh48bW=xtEoXNqHb+WGQCMI%Y)U3&UOhl@GJHlTFn(BKC`VRST1rJzSx9d`Qg&8BIY(h;PDXxhT5n-SN@{0OX*pLoP*E`>btpn&VRTP7eP>g8I50+Fej|BbR#rPyH(z^fPGUAtdvP{oML|qQF;z%vQb{>8Rcd)zJ~K&mK0|9MHF#ldeItHwZbc(mY+zwFMnO3!HFHZ=Sz}-^Z7EGzQ&Czzct9gcYhPY;V`FGFDLg2CW^q(uF+D*|BvCYSStThXbVXxIFf=$<OjJNVV`?;fZ#aBXd^SBnYd2v+Kv!*HO<E~eMOsNrSZzU5C`2J`F>6I{M0O@PK{iBhGcit0P<&2%WjRq)B{6JHP(yu8HYzD%MNUXCZ#jHIZY5!6BX%iSF=<Y0Kv6$pJT*ZhPANDxGB-1JPHbm3P+&qsVsv$ND`_-LUS@MaY+^H4L^~mIeRWrBBWG|eHf&U6Uu!W@YDrEvOLkRHaU?lQLm^Q;QDSs)I9gJ4L3}uCWHWg*R47e)U}8v0Uq)~<OIC1kIWcc$Ky61Yd{attMleh!OE@($Svg`RSY~`XB{e&4LUVUFYhh|WOfx2Id_y=WeLX9DKyNfuCM8yUDOEXTX?A=yS422tNkv3LNh@w<H#B~4aVBa}COA%OB|afjYglM^SVDL)Zc{imDLqe7DtlpUSAHusb!I_8GDl!VUuQLDWL7+MC3k#zVNiTFMlgCcO;#;bS5HtdXHZQhAyi^%C`3VMc`a%!HZ5ZzJx^*+Xfio5PhWRmJ0x;VA#8OuGiNqRLp*y%Xjo8jD>7d&V{C6KdoyrKXIM&aPd;`eEqZKICMJ7vdtYWkc6=>QWLkYlPC!UKBz0P5GHGK$BT92-H*{Y(UM4tlcVlcjL{C9NQbSaJbwet8b5BruenK=$O?4ziJ$zVfBQz>ubACTyFla<GUV3vjcy=>Wcv3%pZYg+qJ5GE@ZBcx2Yf)b^SVL7~PfcD~XFyCuGeRwOIYE9YU{rl@V`@)RekxfgOKet2PBKJDS3W>XLnL!)R(eoDY-&JOG;T(1Yg$fTPjy9KJ~CESVN-1}J2N#*U|~f-Ol4GGR4H;nS3GJdJ4<GKMm<v_Gks?<S3_81HZ5~;M0{R6URq2-HYh=JGH`l9VKHV!Rx)u>Yjt*bb4nvxMM``$R6lGyHC0JFMLTvzGhT2tQebLWeK$jIR3<-XDPnX&D=9uIKuuI%D?2}WR%cFoIV)>$T0>D-R5?g+C2dAHQZ{cuWO7q(Mrl$)Sa?8JKzcnSb230xRd!`*I8s6|XjDdKWmPD2L_K;!bX7!Ca(H(%YF~FzU{xhIHX$}ORCF>zPe@E*etB9}axHjtIVfUED^EKlC2U1maAZGGV`*0?DQ9RvM`vefS#m@$Q)Dw~S4Jj#MKwTILO(`lds#gzZ97(RLMudPU~nx^S93Q=R8d(bDl=(BNHi)|QaOHSa6VyeU|36WPbxt)OlxjtD11CSOLksmSwmw+Ss`mgX>~SORC9S+WkyS6VpUZzM0YJENPc=&bX6;6D=2U^Sv+7cS~F8+J|TTZDoZwPV@X17IW<o{cq&SIDQ;RqX**d<GH82Lbx1x+L~uTFRyZUoVKif8NO?YNQFB6CdTCx_a#cc3c56^hac@^JXH#V|C00-~WOr3oVoFbRWH)UzKuuaWbw5#hBq={gbaiqwW_2V#Lv=whUo&ZRVlg&KR6#9fT18e?SX5wNZZ&#1Y;R{cD>GklZ#-{SYBD7@MK@A2Qgt|2U}tM4OjS5HBYk98Vs}zJJA6uJXed%_WGil9Q)fS9Og}PUP<v%&By&qCZ%#)hPkB5cKs;DNQ+{J*L}Wg9IZ!_>PfKJ^J5FgTcPd9cWlBw7YbZE=Vqs58Y&&35Mq_bNMN2$YL_s`1NJ}|GG;2R?M?P9MLswy9dP8q=U`0$gReEJ%Nm5jAG$A`lV|#seEj&tnMI}o&MQBk|cYaiDKs75$elT7*Xgg6%YI{;rO?xUodTdT|CO=O~UqLxIO*BF{G+;<UHcx10BzJXgK5TJ#AxTMRAtr4<bw5ZpQ+hB<Sv-A2K{I(VVkR?7Jw-NhCNML1ZZv&gT0$!*WoTbzKus-Zbv95>Jx*SGVRl+lL{>OTHF;GeaV>XzFmg{~Y&C3sdVXMjdqgusU^OUuZDuV^Q%rnVVm~-<epgpHKR#4LF=9tNaWFtadPh+zba7%)J9kG)HZe(YZAVd9LsfGlSS@x>WJ-8bU~42YPiT8UY*=J<SZpCfH+OR_M=)x8O;T%pBVut`H91~xerGUhY$HWOCOcwbH$@|PDNr^cMss2`JabA<Y-m0)ElhPIMo>&-STbu>B{EcGYi&kLd0|B=c}+_;P;f{=Mny7kG-zLLacfIJR55NUH!FH+Ib?HRCUse0LU2t)PeDR^Zg)0dFmGc`Yf5%NLqBC-KTSVVQbu_+GI=pLRCPc=K}sV(Ia*e5JZWoLMkF*^XiI&1SbZitdPYffZA~*bDJD5uBPmKtRAn-5a%UkeVQ6S>bxBrNadk2#Gd)gIZYV%VNj+?BS0Q9qb9PEKP-;VQGc-$NP&-mldpk2WOg~moDNR9nZCN)tZYC%rK`J{lIb%I&LUl+xdw4)FS87sLQ*lRGEm3<dP+w1Zd2C;0Hga)IIB-f?N+T**DO5E(c49w5OM59qJy3HyLwZX!ZhLb}Flr?>Y&~IZaWHIgA!9;SUQj(%Gcs;xLr!NSdr?zyZbUy>bU7(+U}9rpS43=aL|-9BSwLknUUxN0Xmdj)USDu<V=+5uS1^1;GEpfbHcn?>eOFLlGj}6Hdvq;VdVXv*B}!>?BUW=tRZDOrcOznHXfk|Sdog);V=ypgN@6!XZhlBAQD=2oVox_rW??=*H&#|pEq6dpb!H(~a4BFaC{Z$5QZ_hyEqZfBDl%{@FlK5tH!(PJKTd8|NjNoFBQ!K(DP>+*UVT?_JW+EvWJ@!CY(i&zVm~o;aCu%?F)(;_RCaP-RX;vAJY+q1W?oQmX*o@FU}8Rcdr&iJI5T`qdvjiMV<A&gGB<c;L}func3L%6PkS*(Sy^6sWqMF>ZcRf$JTOBvPH1d7c3@U_Kp{g#M1Fc?QbKe;HZ55zB`827Ibd-yWKCH!Fmzr+J!UmNGb&A0N@XfbRaq%dUpGEXIC(o@M?h*(DJxTcV=+WFZB9#KdvbC%W?&^rVmN3<Dr`(tL3l}gcPS)HH855_Uq)&)R&GfpCQ%_KU^gUECPqX}J3ve+A$T!qS!{PGOiwv^FfvplGd@UjWh*u-Pgfx}a3oW5b2U73I3X}LUNCfXUMM9}d1^{?L1{TMWk5)GcxORRMQBMeYhX7aa%fpdHefk0OJPAtRakL+S}P%JZg)OzBWh)EV0K7vO*~LdHbYlQJ4RzvJVRe-GBR#(Pf$xZD>8ONM?5)FZhdEIcqmFxCMk1ZPI6jjElN2yLLo~iI3a2?S#wZ4D10hfeO^W|SR-<5ZfJK>L?n7QdQx+Bb16%2Pa`c=GD%Nqa3n=Tbu?BfY$``NVLLW6M?ro*R5xX4L_1zSS2s6aP(L|tVs>p#R3U0}bX0sxeR(oabX7!BJZD6FcXe+`H(^*~Ib?Q8Ku<AEQhjM^Gb3<CR8nDTQ(rVmNl#}xS5`z#CL?xaVNZ5MI5AO8cylsKX<BzZT1<5?D|2^3LuO}pZ6tMNBymwIP-tRVbxnI^H+(#HAuV=FbZ{YZa3yMCb9YZ#JR@ItXnjv?b67J;STJyTc{e#(d~#MvD>6STb8#qoBt=C^aCRkXR9_=2UU+sXQaf>ESu;r{Y(-UGNi<q%Ep8@RUPU=YN-<S8KVerxH(p0SU~Mo-X+dQ@a5ywKL11ZOUp*ylQD<s?SaxV=WiV`6SSf5)S4}cMByBriRZm_vC3YxASzby+Dt%BxcVb^@Bw}xOR#RAHS7L4`XhnN&PD(*dHcn19BuQs`C{S5bJ78*eKy^JnZZ~&DFiLnWOiy_vR8nG0U}!v5NpUq#bwOfEV0C#bRy#s@HB~TTRZviDX-#!gC1-C}Ku2bCB|L3DLo_QjZcRd2V{j^cbw5!?JAPzAG+$&*C{;saX+c9`MSW~`Rc3T{X(dKcW;ZlWN_SLDIeb=Qab8MwNqT2DBT`XBOgJqtOL1yiJYrX4IAU*7KWu9#Lqv8#QCcxsWNmUjVRczORy}7wIXQ1BLoIwoQDk>GPgX)iCSpxuZF_BXOeRf7D|$#fVm~8odSy;dRy!eXSa)zzbt5uKctKPuW+Y!RcR+MdW^R5@RXbrlNJvz5Pdh|bL})p0S4mD&b7wdsctkBrNmO@cc6cdtePt^_Vl;R~Wpp!NIa+TmV|^u2L?}OPOEN(vYIs67C}(ghb3AT4PBlexPJL7|VnJ?wG-Gc|P%2VrO-65TRCYEqdSz=oGE7TMD{4o3VLf{)F=2OYLq9z|B}Yv+S4MYNVr^t9F*IK^K6h|2C3rGlPAO7fZB%zKSvzKBGE+-#G-gdeadcW@V0v*idu2;-UUykjer8d1W=Co;dPFr;Nls`eDI-99NIiaiOe8!vRXin1Pa`#Eb9F>nBxrq5c|ciiHcn$`L`O(#NlkrzFl<CHVQVvQDR4?=Y&};nEhbuhM_F|}b#Fg?CO$(}H7aZ^EigMNLT`C?MJ8WrD?3hMOgwxoI7f4GC`x@Ta6WN%I5JB~S8GglGhaekKUruybyO=+T5M5TX-X|cQcQ1Bd2df=dT}dHDsU+{V0?5&DtAaKFf(vUQ$S*MUo&5Kb$m}zH9c84WmQ)#IAM1vaZp)QPgrC#PG~YTKPxmvL{VWXRdjqoeL*u;YBNN6YE)luNMAc`d_f~lK0rxoF-UV?Mk84=W^`>gM@mFkD?lk!Q9*1aL@0A-HFQKzAtX(0Jbpw>F<CiRFeoTqQbsU7IVmw`cz8QwZc-_9LSAz<Y<@N)BVKuBSV~!LPAz<8I5}rMWI%E_OIk%obRj)rFlJ&kd@*ZbaaKDrXk$2YT1QfRQg&H3V?bY7XiIT&L`FPMcsE5cbwhDrG(b#2T3&cPOIcQLRAM<tA!9ppcQ9xxBx5`(dR1~(AxJGdOIlQWDl%4IdQ@jXI7DhxC1i1UV^%9xF)C4Wb7gQPb1N%CRwhd@dMz?ob3%PoK5{ioGe|HvR3l+LPc$}SY->1SB}!>xN<VEVR6AKmW@K7lSur+fV_{fTHBMA=DR_G@S!#SSGjnTJPBc$)ZZ=<IUq@CeDM~w4M`~q0S}1K{T6TL=Rx5fcct|LCa7}Q0W^r*wT0|;eRycknL|#;SG)GE6BrQX9Hby*JZ7NAOLrpz<BV$QkHbg>oBw}$jDQ-n6COdvpV^@A8YJ6}=Oj2e>MSM6oOL;pzStU(wF>OdPI6ZYZQYbcVXkU9KCM`02HhncVK~y(Gc6l;<b7vtlEjVpFMmR(<WpPhZUu`saVR?IAYCUshG-)MGEqHo+eO6{oL{&#~aV9oWWHcmbd1XvUVLo_CSYARcbSXGHLrF#>Kt?iWL1$A*Qd((jU}->LDN`|fF=#hUNMv$McXUKfW<6AROmIa>R8ufRLo#h{D^qqeGDu#2b9OgDW;uI#L}N5nWoRRPHB@&oRCZNaV?{_>NO4U*XLwm_LU~m@Ha1Q_PE=-lF;y^eQ8r9CT3$6bd1q{AYJ6UEVK8MfO<^&3GFVZ5RaR+nPEIgRc}aOuDs6o~a$iX@JT)?ABtvO+L1kY}bbfL^dvH5@L?dovIdw@XBtLgkEk!{vSzsYGUwchuQ*bk6Wj#eGEowhrG%8v#dq`3yGFC-oNl7v*BT;BUdTK{wCTU`GVPhk0US)esH%L?|G&f*VLn$gzL2-0AIYD<mWK=>XWNJq~Hb*%(SujH;VNi2*Kq)gQSZPg3emGY!RbnbyL2D~;L}*ZISwwzSd`Cz@Sw=!eLqv6FC0cS%Fl}B{d0|l{M_+VyLse#WYh-#oCNn5YUwcO+CPGL}Z+lZUF>N~~WhQlEO)5}nGG{AyWkXUqUpX~bM=(K3c`IsBZbNJ%XiQL4M=DKhSW!VcZ*OC0dN?L$dMS5WUusKjNP2KfKT<Jbc0Mz8S#o1>MQb}jGjK6UBSJA=a!P$rVRU<HVPiZ?bUs;gKvXa`dqz_wJ7-C3c5XsGBvCzXH)lL(OKTxlc0)K(GIm%iJwa(!awKGJbb4z-BUn3QYkqZBZc=V+UrSO_Vr+anaxqJMJ4<d&Zb)W4ZXqE*Gj@4-NM1mCczjVeMQuS|JVRL}HaKS?YkMtwP$VsKG<!u^HC0f2OMQAyR(&C7D|012Kw}|3Atre>Z&ouSOHwFMXLfLQcSCqletKATAv;i0UprbYC1pEuJXth;S$1(eO*ddsDmg-CIc`@nHDgR{Xl_wuc2zfQb!bUXX?#d*BzbfpLU2+jVQW)xIdWwqJYzveXmDsvURXs&Aw_6Sa62h7QFdl;RU{!uL}o&9JU44)Ay<2IY(;EkKSgv?QGHNNW;9e`OHngBDLgGJOeH{LWotlpb|iOYcvV1qR#I?dPGfClcTQ$|LvKoWLOC!`Y<NW`VQ?WRJ9IT!ayv(7P+u}Rcw%jARViUndQ&nsS3YZRK0iflDrqw^FhM*_b8|dYL_}&+Ay`N%YJDkgJ$g<#XJcnFL~eL8Do!deVn`%*QYCIZRBcK%Bwt4+IZjV<K|5?ROFeQ_aA#psJ4k9PZ(v$!RC+x<XCzQ%J|=Q=A%0<QRbfFWKvr&7Y*tcHZ81u2XGn5dNnvtXQEyLYM^rK~Ku<hrMm$tlC1_zuQc6!FUs^FcT1inzaZ*NlPHAOMV^KIZczGdnKXzwjOJGe)Jv(?<H%fkdWnoZXbyGb(bR~0Dc0VIpC|Yq(Yd~&FS37J?EhcwtF<Ea?No+SqcS%cWaanC7NlI06Oi4C#SS4CQDqm7#cQqs<BX>l4RZmHLBql#hNkMi^BP3cWNnccADMwK^MM5@lNKtf5YjRRSG-W7BUMhN1YI!hccW7QUZ6Q1@Av7i^N;g9-bUrXjQ(9VIPi}NoaalHcBtlkkGE`x1P%33cU}8uubx}NUH)di-V|jBjL}+(AXJ2eRPF^`xNklt(Qz24nN=Zv~VQ*M^QDAp@NMChwOnN0lLo$9NF+g8VbZ;#!dU`o4L~}%NWqEO6ZGCG^RzFiJWF%@}J|R(cDPJ~xX>(;uOf5!Ybvb!{LPd5vbY6ZoLQOYtKWsrzYFR5xGdn|XM>tAxC}l)XBVs6CLu5EkR4rskR3SSgSxO}&L0?s4PEsj#T2wJNOCer0CV5UsC~9bBOFe5qbTo8%DJCUBRdZfiYi?FaR!ciJML;NSM_5v9J9&0rD>5cIBrPi@HccyMd{i@1P&qv)e0en~MQ~qDD<x5LeMTczB~~<9N+Cu#B{E<|Sxz!pNo;X?O=DkPV=-E3W=}MGGF~<?J1ZzqH+DE{YA{ShGGiz?JX0nqF<5;}XId*`a!^)PIW{;vJwGifR#QPuSV19ka7kb|J}G@BOm<C8NI5u7YjG<kL_9W9dnPtXZ6QEYMo~*`G(t0eOK*8KEme9*P(?pMRc=o}W_Dp!N;^|cGG;h_Mm0A>Y%4HhKwdLTWMnxtCMG#jUMeYfT1q%+J9JYkVlg%?J2EI|NK!W|WlT*dPhnOubw6)XElFB1Qam<iLThtSVK!weUVKq_Q+aG;Sy@y&H%xR=W-3oXR5LYJct1l}QC56WSb0)BJv%vlC2mDQXD}*XPB1w)A#y%UaW-HnZ%I*4b1FtWYk6xxc}!(ZZA&e1GCy`XBuqwnZzOwndR{?ycX=yeOiw;$SVB~1OK(p_B{E+vRCrZiJvC7=G$}zLDkyYeMk94fb~!UwBOz8+BYZ?rayv>bL^*b0PAP0}T7F(PP*!AFMno}VP;@J2LVia<Wp7P3X?tEzGG%>8U{*^$WNl|kM0<O5Urk<gF??4lcSdM;St>P5b9O&TaX3(Hd2(4tU@bFNEng)qQ(;~(Lt$r4MO8*sd38TbFknx5RV`O_Pf2Q0dLbq_C~q)KN<K+SQFl-=Ds($!Dp@^6CTLDIXGJJ-J3>PxD|L8wFkmf7Ya>E8ZBRK{VMAI&W-v5RdP8SWa#U1QKT9@aPGV<eRX$@hQBWgvDo0gpaYuVbJu+)BZhS#*Ibbj&Lq2OtekM>$SXx>tW=ugLMo%GYdLd6_MpZLZC2D+TVM%LdK1p$XOFVpTYgSTyRB&u@dT?4-RcK9gV_|+PbUS-<G9h4Obt!vSMrvMmBSmO(D=B+4C~ZVgJS%f<Qcz_|b!I?4S5tX&QaM9eZcBJyIA}9BDN|E2Z%HjBKT>-neOM$wa%*sTKWa2RYI{OADOgKYS7K{nN>e~nNJm3!Z8>aXbapXTYhhJXDm8OnbT(u~J0x#FY+7)3C{#{)VIyOEQ9^JrSwL+qF;P}dXHy|HOeQI6Au?}qNKGkYWo}wLC_FrBVRT|qSxPZvO+{E?Y)ni&MqojFQfo0bYfgGUKT2<JczSvxWk*<NYiv(>b6GTSFfB_<OLA;SQz<=0Q+HTbWKc9|A#Pw_QZjZ_d{T2(bSQFeDMu-LGkPshZ!;xEZE871I5tauXghscGC?qYXlN^FdU`#6FjQtCQE4}HH#jI`I3;#$P*X@zaeHG)FhxsbRxvqpFi2HPbSq#xBXu-BD^f{3bT>*dV_`c)Vl!5BYI`|)O=M3tR(4K%PkJM5ads_0W-EC(N+BUVM|o*fR(K|3U}S4CFjpyaMRP(=C08pvR3uDhBvd$Zc4S{kF<x_1eI;vgNK!&JCVDg|O*l|=Q8hDfK0rJqG9@{7ZeBZXab!Pib7*onQYl(hR%2*8GiE9zQ&dW0YC(KwUMM+dD>hAKbT@oHJ2!Z7KX6ehGeI~~ZB<7_Lt{KPH!WZ$P)s{LLUm_+NO43zDs?e8C`Ns0R4QUEKR`=MBqTX|dtpjTV@Gx;DJd>wWn*t-Whf_gbY?9$Cn+fkUte}*a&u{KZeL#@Js>ASOf6P1H!V^zEmA`=CkhH*P*O=lMPEitPft=HJs>d(UqnSsK~6+pK}=9cK_ERKFfKAR3SUh@QcFctUr<3(K}}O2Js^7uARr(hB3DR7K~y3-ASg05EFdv3FfcGIAT%&AEFdCtadLDbEFds0I4mF{Y;R{GEFds0GAS$yARr(hB11t^QcqMOIv^-BH7p=8FfcGMEFd&BFf1S<Y;R{GEFds0GAtk>b8&KXA}k;<E;lJG3LqdLAR<&xO+i#oB03-_HZUw8F)%PNFf1T4Ffc42B5Y}HWnpq6EFds0G%O$@b8&KXA}k;<E;cDF3LqdLAR<##QbAWjMN(2(B03-_F)}bLATcm7Ffc42F)%PJAR=>da&#grATTaBEFdCmX>Mg<aw04sF)lVKED9hXARr=5MNCglB03-_GBq$PATcm7Ffc42GcYhLAR=sUXCf>hFfKAIAR=>dA}k;?E;cDF3LqdLAR<LaM<O~PC^axFATcm7Ffc42Gcz(QAR=sOZe?L|A}k;<E;KA4B5ZGGA}k;<E;1=B3LqdLAR<jkOiLm<ASf|5Ff1T3FfcGMEFdv5GAtk>b8&KXA}k;<E;cM6B5Y}HWnpq6EFdv1HYqF$ARr(hB3DmOOd>iUC^9fGEFdv3FfcGIATcmCEFdCmZ)YMbATTa6EFdCtaUv`rGcGbIED9hXARr<}MN(8rOi5ZrQX)DaC^0ZFEFdv3FfcGIATls8EFdCmX>Mg<aw04sFfKGKAR=sOZe?L|A}k;<E;K1D3VjM+Q%FxxUr<s{L{&pnQy@JcdkP>RARr<_K}$taSt2?hC?Z8iM<OgBB3DR7K~y3sED9hXARr=8Nm^P#UsFg=P$D`YC?ZWsOiLmxAR<&xO+i#oA}k;xS4c%cR3a%X3LqdLAR<CiRZc@lUsF&|R3bVcC?Z8iM<OgBB3DR7K~y3vAR<##QbAWjMN(2(A}K5iARr(hB3VIFPG3`0Pf|r9Iv^+_S5Hq&A}lE^3LqdLAR<XaMPEZwML|tpQ%FxxB03-_B2!dSL03XWQc_tWEFdCHNlZ&3EFdCRNJT+ZA}K5iARr(hB2YzCUqeAgMIt&NC?Z2aQc_P;A}lE^3LqdLAR<#uPft`xNkv~%NKa5AIv^+_Q&dtxS3*TnQduG_AR<jkOiLmuED9hXARr<}K~hacQd3_|K~hUaR3bVcC?Z!#ML|>|EFdC7K~hprR3a=OB2-UJK~zs7EFdCNR8m1#LPb(iSt2Pc3VjM+Q$<WnK|)MLAUz;-b#QEDC|^xMQcFctUr<3(K}}OB3SUe~QB_GqK~zakPG3(_L`6~{Js>CwARr(hB11t^QcqMOEFdC9M@J$oAR<OZQdCJyNm@lxA}k;xO+`#kP9iKIB27t5OCl@^ARr(hB2!dSL03XWQc_tWEFdCOPfbBoPa-TJB3DR7K~y3vAR<>!PfQ{#3MmR-S4BlcUsF^;R7D^?AbT)6AbWi*ATc^1dwqQhUspv%L|;-xP)tEtUsF^?P*Wg1AUFzNQ%FfhR9{n6K~zN`Js^7uARr(hFghT6B5YxEbYF9HWpE-oAT2R0AR=USWnXi2WpE-oAT2R0AR=USWg<EtdwqQ@3LqdLATc^1dm?OMb97&GbY*ZNIv_1EEFdCeb!A_3bY*ZNIv_1EEFdCeb!8$tAbWj%EDC)JUr<s-MNLptUqwzqLQF+OAUz;PVQh0{3SUrCMMX_eR9{9?K|@qYPfj2`ATlm63SUrCMMX_eR9{U&SYJXxR6|H0Js>kM3SUrCMMX_eR9{U&SYJa-PfkT&L`hRrK~6(OAUz;93SUrCMMX_eR9{U=PG3+`Nkc_nQbANnPar)YFfK3(Ur<s-MNLptUrk9)Uq)3_RZ>M?QB^@sR7q4>AUz;73SUrCMMX_eR9{n6K~hv8Js>eMFbZE#Qbk2gP*h)2R8LSKJs>tXFbZE#QbkQkRZSp0ASfbJR8m1#LPb(iSt2YTB27h1Pfj8%AR<jkOiLmxAR<>!PfQ{y3JMBjWo95>XJvFKc42IFWh@|TWqB+hWMyVyb!>D!PH%2yDLM)uARr)VW*}*EX>N0LVQyn(D0X3Nb!99dWNBk`DLM)uARr(hARr)eWps6NZXkAHY;|QWXJvFKYh`&XAY^4`VRdYDDGDGUARuRDbaZ8MAUz;wWprV5baE(mVQh6}EFdChWppAeAWm;?Whn|EARr)VW*}o>Y;0j-Y-K2CWps39aw$3rARr(hARr(ha%FUNa&91JWps39awuzMc`P7gWoBV@Y;-9KARr(ha%FUNa&91JWprV5baE(mVQh6}EFf!Tc`P7gWoBV@Y;-9K3JPRpW*}c<Z*X~EVPkY@Z*C}IV{~b6ZYeqnARr(hVPkY@Z*CwxAY*TEc`jsSWpHC}aCs<UV{~b6ZXj=RAbWi&3LqdLAaZ4Nb#iVXdkP>RARr(hARr=UVRCI{aw0k)Y-w|JC}CrCX>V>WXJvFKB4%N7ZDn#IDIjlhAX_3(K~qyAT`4RIARr(hARr(hB4}Z5WOE`qAX{u{b95+ga%5$4Aa8OYTOv?FQ&S>cDIjKVav*PVWMy(7X>K4WVPkY@Z*DGUWppSaXkl(-b0R4qZ*m}8T`65G3LqdLARr(hAR=vHa%*LDB03;jY-w|JC~tCPWpXJXW^ZyJZ*pX1av*7LAShvDbZKvHE@x$QC?aiPa%*LDA}Jtmav)n>DP1fIARr(heF_Q+WMyU`Uvp()bSQ6Pb16CsARr(ha%FUNa&90oAZcbGX>N2VUuR`>C~snOEFdCqY+-q2aw04sFexB!av(4%AU!=GF(74Zb7def3JMBjWo95>W?^z|C~snOEFg1bVRR`v3LqdLAZB55ZF3+!AZ%%KbSPhEWppTSVsk7YB4%N7ZF3?lAX{B2Aa8OYTU{v%ARr(ha%FUNa&91IVRCJATXSV$bX_26W*~EAVRRroAZ%rBC}v@DZF4CgWo&b0AbWiZ3JPRpW*}c-Y-wk1Uua=&WOFECV{~b6ZY&^gVsj}v3LqdLAYo&4X>V>IJs@9WZ*X~EVPkY@Z*C}IV{~b6ZYc^NARr)RcyMK7bY)~9Js@mlZYW=8WppTCW?^z|C~snOEFfQVWnpwEZ(?&PDJ&o&Xkl(-b0RDtTU{w2Z*m}8T`3A6ARr)UVQyq|AUz;#X>)WaVPkY@Z*DGUWppSaXkl(-b0R4qZ*m}8T`3A6ARr)VW*}^3ZYXGBZe(*QAUq&tcyMK7bY)~Z3LqdLARr(hAZTH3WOFWMcywiMWGGu(B2Yn7QzBg;W^ZyJUm$62AaY@DXJsg5cyMK7bY)~9Eg)=VZYXGBZe(*QDP1WFARr(hVPkY@Z*E&6Xkl(-b0S?JJs?|bX>)WaZ*pX1av*PVAX_3(K~qyAT`3@DZ*m}Sa%5$4AZczOXkl(-b6YxPcyMK7bY)~+T?!x|ARuyObairWAYo&4X>V={3JPRpW*}d4Xk}zyVPj)ub8{$jX?kTTItm~lARuUAY-S)mAaiMYWgssvATkOdARr)eWps6NZXkOKARr(hARr(hC}?49W*{vfF)ScxVQgj~Eg&%|EFdUoVQgkBAZTH1W*{vfF)1txARr(hARr(hC}?49W*{vfF)ScxVQgk8EFdUoVQgkBAZTH1W+^NRARr(heF_Q+WMyU`UvP47YGq?|Wn^D-Xk}z5Z(?&SAYo&4X>V>RItm~lARuO8a%~_zAYW!-a&0JYVsk7YUvp()bSQ6Pb15kbARr(haB^vOVRU66Js@9aWppTSVsk7YB5-nPc42g7A}k<#eJLPsav*zs3LqdLAaHVTYGq?|Wn>^dAbScRARr(hARr)XWqCRvZDDvQFf1TxZgePiVQh6}Aa8OYFexbtARr(hARr(hW^ZyJYh`&XAa-GFb!8xFZXjf7V{|BAXJvFKaB^vOVRU6IAR=>UWn>~OAbWi&Aa8OYdwnS`X>?_6b0{eaARr(heF`8TARuXOc4cmKZ*pm6b09q+Y-w|JC|_q~bSQ9gX?9_BWh@{fX>N99Zgg*QX=QUFEFfE5DIjlhAX{B23LqdLAaHMUX>@6CZgU_#AX{H&WppTJVRCIOAR=aAa&2XDA}k<VFf1T2T`4ReDr{+UbSPhEWppTJVRCIOAR=gCZe(*JEFfE5DIjlhAX{B2T?!x|ARu*aX>?y<V{~b6ZgU_#AX{N$bZKvHE@x$QC?aNIa&2XDA}k<VB2Yn7QzBg{EFdauX>)WaVPkY@Z*DGUWppSaXkl(-b0R4qZ*m}8T`64(ARr(hbZKm5b09q+Y-w|JC|_q~bSP$Fa&0UiB6MkNWpg4dAX{B2Aa8OYTU{v%ARr(hVPj)ub8{d)AYXH6Wn^DrV`F7=b0}<OZYXqVY-MvPAa8OYF)%3#ARr(hW^ZyJX>Md?cq|}wZfSI1VPkY@Z*CxIZXjiDb!}yGVRU6Eb#7^NUtwc(X>V?GDLM)uARr(hARr)VW*}*9WMz0DK0P38Wo{^NZ*ysMX>V?GDIjlhAZc!7Wq2SyJs@mlZYXJPc4cmKZ*pm6b16CsARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAaHMUX>@6CZXi7%aBp*IbZKvHb6aU{WMz0=3LqdLARr(hAZcbGZf|rTX>)0Ab97;DV`V6CZ*ysMX>V>UASi5Ub95{qbailSWhp5jZ*m}PWo{^NZ*ysMX>V>RAUq&4Itm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)kEFgIxJs@drbSQ9db7^#GZ*E&KT`4ReX>N2VaBp*IbZKvHTQOZJ3LqdLARr(hAZcbGD0nO&c_|=nZ*(AOZXjV}V`X!5Aa8OYZf|rTC@>&AJs^1?JRodkZYXqVY-MvPAYpD~ATS_2Js@}>JRodkZYXqVY-Mv>d0i<fItm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)VZgypEbZ>HbAUz;^Yh`&lAZ=lIC@?G_X>N2Vc42IFWgu^IATTK@AZBlJAZulLEFgAaY;|QIX>K57X=8LKX>N99Zgg*QX=QU;X>Md?cwHcGav*zsDK2SrWo>gPDSZkcARr(hARr)VW*~KLX>?y<V{~b6ZXjW9WFU2JX>?y<V{~b6Zd)*2AU!=GB1BS8P$D`CARr(hARr(hARr(hWMyz~b7^#QAUz;yZgypEbZ>HbE@^aSZF49o3LqdLARr(hAZ2W6W*~KLX>?y<V{~b6ZXjW9WFU2JX>?y<V{~b6Zd)*2AU!=GB2Y|0Lq#GWVQyp~Y-MgJb#7^NUtwc(X>V>RAU-`HGCB$%ARr(hARr(hARr)VbY*QIJs@>%X>?y<V{~b6Zd);33LqdLARr(hARr(hAarSLWgtBubZKm5b6a^`TX<axARr(hARr(hARr(hb98cbV{~<LWgtBudm=+mS0XwfB2Yn7R8>+%A}k;xQ%FTcP$D`YB2Yn7R8>+%A}k;xM^8^vMIt&NB12D4P$GRUXJvFKX>?_6DGDGUARr(hARr(hARuXGAaitbbz^jOa%CW4Ze$>7b7^jKbYX5|WhiuMY-KDUWNBk`DIj5PWFT~DY-KKIWppSaYiVv|A}Js}Js@**a&=>Lb#i4OVQyp~Zf|rTbZKm5E@x$QC?a8QX>DO_A}KlwARr(hARr(hARr(hARr(hV{dMBX>N683LqdLARr(hARr(hAarthItm~lARr(hARr(hARr(hARuyOadl;LbY)~9Js@drbSQOhX>?y<V{~b6Zd)>4DIjTPAZ%rBD0OaWbYEd(bZKvHDIh*QATuCkY;$EGF$y3cARr(hARr(hARuLUV`Xr3AShIMaAieua&K}hAXZ^)b!A0za&K}eItm~lARr(hARr(hARr(hARuFJZggpGb!7@5ARr(hARr(hARr)QWpHnEX>@ZSJs>D3X>?_6EFf)ZZYXVGcqlL|AaZ4Kb!BsOWn?KVAZc!PWo~qDa(OOiWppTMbY*QUATTK@DJ&@pARr(hARr(hWo&b0Itm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)SZ*m}MbY*QUAaQkJZggpMc_3+SAY^55Z*ysMb2<tjARr(hARr(hARr)eZ*OfNJs@picqlL|ATcm7AT1zsb!{kca&Ky7V{~O?E_PvTb!BrXDJdxmARr(hARr(hARr(hVQp`9Zge0$AZ=-GC~aYQC@?G_X>N2VadlyCbZK;XAa8OYFexc4AaZYSZ7B*MARr(hARr(hARr)VW*}j0Z*^{TItm~lARr(hARr(hARr(hARusZZ)#;@bY)~)X>?_6T_8OmaB^>IWn*+@WG-iAbSP<bWo;}VFexA_AYpB9b#8PDARr(ha%FUNa&91Sa&Ky7V{~O?3JMBjWo95>aCKsAX=7h=X=iR>bairNC}v@DZ7DhmARr(hYh`(JAUz-`3LqdLARr(hAR<>tML|>|EFdC7K~hprR3a=OB2-UJK~zs7EFdCNR8m1#LPb(iSt2YTB27h1Pfj8%3LqdLARr(hAR<FgS0XGRB2!33MNlFvAR<RkPg6xAEFdCKK~q#!Qbi&xAR<FgPf#K(AR<>qMMNSj3LqdLASntUARr)PZ*^{Tb09q+duwHRIv_A0W^ZyJYh`&LX>K5EWqEUb3LqdLAZBlJAaZYaAZczOC|_q~bSP$Fa&0UiB6MkNWpg4dAX{B2Aa8OYTU{wS3LqdLARr(hAZBlJAarSLWguy8AaZYaAZcbGX>)0Ab97;DV`V6EZ+9#pY-w|JDIjHRb7de~a&LECItm~lARr(hARr(hARuXGAZ~ATAZc@HZgX^DZewLAbZKm5EFffQV{|Dx3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuOMav)}DWo%?1X>K4WB4cuIa3U-qB4KW6ZDDL8EFdClX>Md9DLM)uARr(hARr(hARr(hARr)iVQh6}AUz;+baE(kX>4UKXJvFKW@%+?WGo;eA}J{@b#QQHawsVZARr(hARr(hARr(hARr(hX=WgHVQh6}AZczOV{dhCbaOfiARr(hARr(hARr(hARr(hARr(hV{dhCbaPvFVQh6}T_7txATbIcARr(hARr(hARr(hARr(hARr)Oa%Ev_3LqdLAaZ4Nb#iVXC<-7TARr(hARugIZYW=8WppTJVRCIOAR=gCZe(*JEFfE5DIjlhAX{B2ED9hXARr(hARugIZYW=8WppTJVRCIOAR={cY;R+0Wn^D*bzx+3VQzGDA}k<VT`3@Mav)n>DJ%*gARr(hARr)gb#QEDC}VGRZgg{7Yh`&|AZBlJAZulLAZczOb8m8VWn?I0Z*^{Tb15k-3LqdLASnt83S?zwAYWr_Z*FB@WNCABVQyn(C~snODLM)uARr)SVRCJAAUz;#X>)WaUuR`>C~snOEFdCgVRCJAA}k<VT`3@Mav)n>DGDGUARuXGAZ%rBC}v@DZF4CgJRmYU3LqdLARr(hAaZ4Nb#iVXF)%7BISL>kARugIW^^nda%pF1bRaz-UvPC|Y-wX(b7^O8VRUtJWhiE0a&2>4FkLAuAYX8GVr*$+Uvp_^ZeetFa%CuHVRCJATQOZJ3LqdLAaZ4Nb#iVXC<-7TARr(hARu94b0}<OW^`LHT_7zWa%pF1bXzc8DGDGUARr(hARsFsGaxD;VPbPAY-MJ2TQOZAEg*7fXJ~X=F<mJNARr(hARr(hD<E@qZ75-4b0}dTEg)hkAZBlJAYm*ZVjyX5AbM$VC~ReBbXziAEFf}eXJ~X=GF>St3LqdLASnt83S?zwAYXH6X=Zd^b97;JWhifAb1WcpbY*ZUItm~lARu#PVRRrpAYXH3VRR^OVsj}9ARr(hb97;JWgtBuUsFg)MpR!@R6$flTXSV$bX^J{ARr)VW*~EPWpE%pJs>b3Z*m}WbY*ZLJRoUqbSQIlVRU6KXJvFKB5YxEbYF9HWpE-aAT2Q|DLM)uARr(hARr)fbYXO5AUz;^B5YxEbYF9HWpE-oAaitOa4aAqWOZd<b97~JB03-~F)Sb=WOZdCIv{&}eF`8TARr(hARu2;NJ&OiUsF^;R7G2JWnpw(AUz;+bYXO53LqdLAaitKbY)v2Y+-YBUvqS2a3WnGJs@**WpD~0ARr)eWps6NZXk1XVRU5*3JPRpW*}d3WpH76Uvp?_W^^cTVsk7YVPkY@Z*D9gb97~JDLM)uARr)VW*}~FbRb_)Qbk2gP*h(<PC-IUMMOFZARr(hARr(ha%FUNa&91DV{~b6ZVDhEARu#eVRU66Js@9mXlZ72UvqR}bY&=SVsk7Yb97~JDGDGUARuXGAZc!ND06gSbY(7QWppSaWOZd<b97~JA}k;+F)1k^Aw3{-bY*Zl3LqdLARr(hAZcbGX>N2Vb97;JWiDrBbSNTZb!A_3bY*ZNEFdj0DJdX4AaitOa5@SgARr(hARr(hARr)fbYXO5TOwq2WnXi2WpE;0EFg1qVRU6%B4l-CB3&RoAT2R0AbWiZARr(hARr(ha%FUNa&91DV{~b6ZVDhEARuIQWgtBudueoKZ8{)rVR$GoEFfuabSQCkVQzG3ba^Q$AZBlJAZc`EZ7d*hbzyFFX>@rYX>K57X=8LKb97;JWiDrBbSNTZb!8$cAa8OYdwnS`X>?_6b0{f&3LqdLAZ=lCYh`pGJs?|M3LqdLAZBlJAaY@MAZczOVPkY@Z*DGUWppSaZDDe2WppAeAX{B2Aa8OYTU|N|ARr(hARr(hZ*pX1av(h*Y-w|JC~{$UDGDGUARr(hARuXGAZ%rBC~tCPWpXJXK0P2aAYpD~Aa8PHWpZ0ET_8O@AR<#mOiUsmVQyp~WOZdOXJvFKZ*pX1a$7N7EFdr`AU+^4Itm~lARr(hARr(hARuXUWo;lmAa8PHWpZ0FT?!x|ARr(hARr(hARuyOadl;LbY)~9Js@picqlL|AZc!NC~tCPWpZ0GT`4IFARr(hARr(hARr(ha%E(7V{~b6ZXi7%ZE0>Oa%FLKWpi|8WGo<Lb!A&=bY*Q_DGDGUARr(hARr(hARuyOadl;LbY)~9Ej=J|Wn^_@bZKvH3LqdLARr(hARr(hAY^rATWNG<ZCxNOJs@&rWOZY7X>V={ARr(hARr(hARr(hX=WgDWpQ<7b97~7AUr)FFggk#ARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hZ*pX1a$7Q8AUz;*WpQ<7b97~73LqdLARr(hAZ=lCYh`pUVQ_F|Ze%ELa%5$4DGDGUARu95bZKvHTOw^?a%*LDB3&RoAZ=lCYh`o_ARr(hb97;JWm_U-b!A_3bY*ZNT`VATbYXO5TOwq2Wg=Z5Js>SHEFgP*3LqdLAaZ4Nb#iVXVPkY@Z*B?-3S?zwAYW#6bairNUvp(_Y;!1cbY*ZUItm~lARuXGAaitOa3CunF(5uYAZ%rBC|^NCR7p=xQz<$MARr(hARr(ha%FUNa&91deF`8TARuyOb9HQVAUz;^eF`8TARuOMav*YHcOYqQASho!LsUsmPE%WRbY*ZLD<CmlE@x$QC?aiPa%*LDA}Jtmav)n>DLM)uARr(hARr)VW*}^3ZYXkLcPSt~Js>k6VQyp~a$$E{FkK)$Js=`eMNCX0AYpD~AaY@MTQOZAX>K52P*O!rNmWfc3LqdLARr(hARr(hAaZ4Mb!>E7a$$E{F<o6CJs@&rb9HQVE@x$QC~{$UTQOZMATTK)D<ExQcqlL|AZc!NC~{$UTQXfKDGDGUARuyObairWAaZ4Mb!>DB3JPRpW*}d1a%E+0aCBdDXlZ72C~snOEFfWHbZKvHEFg1qWpF7v3LqdLAZcbGZf|rTUr<s-MNLptUqwzqLQF+OAa8OYZf|rTC|^)gMMX_eR9{n6K~hv8JUt+DbY*ZLJRo0CQbk2gP*h)2R8LSTItm~lARr(hARuyObairWAYo&4X>V={ARr(hb97;JWgtBuUvp?_W^`Y3bYXO5C~snOEFg1qWpF79ARr(hX=WgEbYXO5E@x$QC?aHaWg;mcZ*m}CV{C73WnW}zb97;DV`V6BVsj}VJ|JIEQbk2gP*h({L0DfyOixZlUqne$R6$NdMLG%~ARr(hARr)eWps6NZXjV}bZKvH3LqdLAZB%Rb#i4OJs@9Zb#!%dWnXh;Y;1EVb97~JDGDGUARuXGAZ~ATAZB%Rb#i4o3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)ZVRCC_bRaz-Y-w|JC}CrCX>V>WXJvFKB5h%EYh`pIDIjlhAX{B23LqdLAZcbGY-MgJZDDe2WppVZK0P2YFggk#ARr(hARr)eWps6NZXjV}bZKvH3LqdLAaZ4GVQFq@Zf77pAYX8DZ)#;@bY)~;b7*B`C~snOEFfWHbZKvHDGDGUARuOMav*YHcOYqQAZ=lCYh`pg3LqdLARr(hAZcbGY-MgJa$$EVAU-`HGazAZWFT^3cUv%BAU!=GB2z_7Od>i8ARr(hARr(hARr(hX>?_6AUz;*VRu_GT?!x|ARr(hARr(hARuyOZDDC{X>Mm*X>?_6T_8OmZDDvQFf1TxZgePeWo=<;ZfS03E@x$QC~0(MZ7d)#DIjlhATTK)Eg)@ScqlL|AZc!NC~{$UTQXfKDJcpdARr)ca%p2_b09q+UuR`>C|_q~bSQ6Pb1WbtZDDe2WppAeAbWi&Aa8OYdwnb*B5-nPV`Xz9EFgP*DIjlhAbWiZARr(hb7*O1bY)~9Js^913LqdLAZBlJAZc`EZ6IlGAYV{YMNLUnO*#r7ARr(hARr)Sb#!%dWnXc1VQzG3ba@~>AZ=lIC@?G_X>N2VW_5IRa%C=OWppTMbY*QUATTK)Z*m|oDJcpdARr(hARr)VW*}yDbairNUvYI|ZggpMc_2I>Ur<s-MNLptUrk9)Uq)3_RZ>M?QB^@sR7q4>Itm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)OVRL0)aB^v5WgtBuW^8X^bSPg<K~hUaR9{d*QbA2qTWNG<ZCzV1T`3A6ARr(hARr)VW*}y4Z((#OUuR`>C~$IVV`XzJAZc`EZ7d)#DIjlhATTK)JRo9Wb7fy}a%p2_ASxhVP*O!jO;A){O-W8)P*O=lMPE`uR7p=d3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARu&Ma%W|9AUz;$X>KSAARr(hARr(hARr(hZDDvQFf1TxZgePeWo=<;ZfS03E@x$QC~0(MZ7d)#DIjlhATTK@ED9hXARr(hARr(hARuOSbairNUvYI|ZggpMc`OPbARr(hARr(hARr)LP*O!jO;A){O+i>+LP1nRNGu8<ARr(hARr(hARr)ZVR$GpEFfuabSQFfb#7!RW_5IRa%Ep}bzyFFX>@rYDj;7_Qbk2gP*h(=Qb9vhNl#8GDJd)pARr(hARr(hDGDGUARr(hARuXGAar4JXJvFCJUt*VAa8OYY-MgJZDDe2WppVZK0P2YFggk#ARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hZDDe2WpplKaByXAWGGuAQ$<WnA}k<jbY*QUAar4JXJvF<DGDGUARr(hARuyOZDDC{X>Mm*X>?_6T_8OmZDDvQFf1TxZgePeWo=<;ZfS03E@x$QC~0(MZ7d)#DIjlhATTK)Eg*DZa%W|9DGDGUARr(hARu#SX=Zd~WLs%;Wo=y`Js@;pa%W|93LqdLAZcbGb7*O1bY)~Z3LqdLARr(hAYo&4X>V>@B5h%EYh`pIT_8OmZDDe2WprCQF)&>UARr(hARr(hb97;JWm_U-b!A_3bY*ZNT_8Omb97~JAS)m-3LqdLARr(hAaitKbY)v2WOZdCT_8Omb7*O1bY)};ARr(ha%FUNa&91DV{~b6ZVCztWMyU`Uvz0~WnW=*C}v@DZ7d*gZ*ysMX>V>RItm~lARu&dc{&OpARr(hARr)kEFgIxJs@drbSQ9db7^#GZ*E&KT`4ReX>N2VaBp*IbZKvHTQOZJ3LqdLARr(hAaZ4Nb#iVXC|_q~bSP$Fa&0UiB6MkNWpg4dAX{B2Aa8OYTU{wzd0kt0T?!x|ARuLUV`Xr3ASg+0WMz0oa&m8SEFe^QaAieua&K}hAXZ^)b!A0za&K}eItm~lARr(hARuyObairWAR<gpLrX<OA_@u$WMyU`UvzR|V`X1qV{~tFUtwc(X>V>Qb97~JEFfWHbZ>GgItm~lARu&dVPj<=Js@8}LsUsmPE%WLX>KTOVR$HMZgePfbY*ZUEFdr`EFf%UZYW<tLsUsmPE#o$Eg&%|T_A6AAbWiZARr(hX=Wf{V{~tFAU!=GB4%N7ZDn#IItm~lARr(hARuyObairWAZ%%KbSQLkVPj=3XJvFKB4%N7ZDn#IDIjlhAX_3(K~qyAT`3A6ARr)UVQyq|AUz;-a$#d-E@x$QC?aTKZe(*JEFfE5DIjlhAX{AuARr(ha%FUNa&91OX>)WaXkl(-b6a6!bZ>H9AZcbGVPkY}av(e)Y-MgJXkl(-b15KYY;$EGTOv?FQ&S>cDGCY-WMyU`Uw36?WM6V+aA9e3Utwc(X>V>QZ(?&SAYo&4X>V>UAaitOa49+pARr(hVPkY@Z*CwxAYWl@X=iR<Xkl(-b0}eBbZKvHEFf=Ub14cSARr)fWnpw6Js@9mWnpwEZ(?&P3LqdLAZKB1WgtBuUspv%L|;=>K~zOsb7f(4T?!x|ARuXGAaitOa3DQBATS_rav*bbWpE%oAZKB1WiDrBbSNThVRLj}b97~JA}k;+F)2C<ARr(hARr(hXJKt+AUz;^B5YxEbYF9HWpE-oAaitOa4aAqVPkY@c4Z<uAbWj%3LqdLARr(hAYWHSMMPgyR6$flTXSV$bX_1lAZKB1WeOl5ARuR9ZDm^`Y+-YBUvqS2a3WnGJs@**WpD~0ARr)SVRCICJs@9ZVRCIKZ(?&SAaiA5bSVlTARr)cZ*ysMX>V?GAUz;kUuR`>C}v@DZ7d)nW?^z|WpW}ZEFdauX>)WaUuR`>C}v@DZ7d)nXkl(-b0RDtTU{w2Z*m}8T`64(ARr(hb#7^NUtwc(X>V?GAUz;kVPkY@Z*DGUWppSaW?^z|WpW}cAX_3(K~qyAT`4ReDr{+UbSPnCbZKvHE@x$QC?aTKZe(*JDIjlhAX{B2T?!x|ARu95bZK^FAUz;wVQpnwB4J~6X?A5IT?z^yARr)SZ*m}EV{~tFEFg4pVQzC_V{~b6ZXjuHAZ%%KbSPnCbZK^FE@^aSZF49oDLM)uARr(hARr)VZe(S6AUz;3AZcbGVPkY}av(iDAR=aAa&2XDA|PdKb7dfDZgePNV{~tFDIhB#F$y3cARr(hARuXGAZc!7Wq2SyJs@mlZYXtbX>?y<V{~b6ZgVL*3LqdLARr(hARr(hAYo&4X?A5UaBpxZVPkY}ax5TDZ*FBN3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARu99WgtBub97~JAT1zta$#<BVPkY@Z*E&6b97;HbRt~}ARr(hARr(hX=Wf{XJsHgJs>eU3LqdLARr(hARr(hAa!nObYEd(bZKvHb6aU{WMz0=AUz;#X>)WabaG*Cb75n2X>V>@B57`PWo~3;WFlQD3LqdLARr(hAZ2W6W*{;kJUt*`XJsHfJs>e4D<EH2MMXqkQbkZqL0Ml@R7FrzItm~lARr(hARr(hARu*aX>?y<V{~b6ZgX2{Ze(S6T_8OmUvzR|V`X1qV{~tFUtwc(X>V>Qb97~JAT1y<EFfWHbZ>Gg3LqdLARr(hAZ2WGWjYEVARr(hARr(hARr)NV{~bDWiD`Ua42D8bZ>GjAWm;?Whn{@ARr(hW^ZyJX>Md?cq||&aBp*IbZKvHEFfuabY*U2Wn?KJX>K58Zgp*Ca$$63D0*pdC~$9cX>@6CZgVUkb#7^NUtwc(X>V?GDJeP%ARr(hARr(hVPkY}av(h*B4%N7ZDn#IAZcbGX>Md?cpyDJATS_hY;$EGX>Md?cpxnxF$y3cARr(hARuXGAYo&4Z*m}MZXjV}bZK^FAa8OYZf|rTX>)0Ab97;DV`V65ZggdCWMyP5AZ%%KbSWTjav*MRbRcPNbY*U2Wn?-EARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZcbGX>N37Ze(R-TQFT9Zf|rTX>K4WB0^P3OhjK$K~q#!Qbi&xAR<sqK~7X6DLM)uARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hbZKm5AUz;obZKm5Utx48W?^z|EFf@ib7^#GZ*D0HARr(hARr(hX=Wg9Z*(AOb7^jKbYX5|WhiuMY-KDUWNBk`DIjlhAarSLWiDrBbSNTgX>Md9DIg&|AR<>qMMNSx3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARu95bZK^FTVZ2#Z*pBAJs^7`b97;HbRs$+b97~JEFdCjZggdCWMyO`Iv{Lmb95+aZggdCWMyP2eF`8TARr(hARu*aX>?y<V{~b6ZgX2{Ze(S6T_8OmTOveBM<QJc3LqdLAYo&4X>V>@B4%N7ZDn#IT_8Omb#7^NUtwc(X>V?GTQFT9X=WgGZfSI1VPkY@Z*FrSWo&b0AX_3(K~qyAT?!x|ARu95bZKvHTOw#-Ze(*JT_8Omb#7^NUtwc(X>V?GTQNFa3LqdLAaZ4Nb#iVXUtw%%XKr6;VQyq|C}CrCX>V>UAa7!GDGCY-3SU++H(ydUUrj+$OGQ*uAUz;zb8l`gY;R#?b0~UjX<{y9Wn*t`aB^jHb0}hAb7eL(E@C(}WMyM-WMwD{ARr(hB4aIZJ54KTb0$?aWI16_V?K6oZah<cXm3bEZ+c5{Dq~+aO)5!BPIZ1`L`O6*MI}K_d}va0Up#$wJ|!etd}eiMJ47)&O<EyZV_7yqM`bx7SZQ7~bxV1CM15CdZf$ZUI73TCZahdicp+a^Yb$$4V0318J}Yx8JxXv;Q+!_~X<<ieCQ&e6cS~7rR5B=FB~MaASYcmCKR0hqR7+MjaZo8UDMdX$D0negEo(70HdZM>KwnCIWKbn}cXUH~WN|Tkc1u)Id_rGpNPJT(W<D`0MomjGD0gc$Ln%rtOK?hgNO&VsBzH%4KU7pzOi3+3Ln$h8GfzcLby;dKb5%lhO*L>)US(cXY$`P*Xg_vOK4MH_Yb#hWV_`8UKv_^(BR5AOBY19bQ8YncAxd68MRh_`bZ9MDUQKgzFiJ2qc}0CudqzTYH9&G~enWOsS3hqdR7yNST6KF^FmZT2Q&@Z^Z8S+TWJgJEI7M?|L@g#)J#>CTN_t^-d{;z0Ej?OQSv5IRS#xb<dU-P?S371&H#}G{NhmlgVm45AVk<U!d}S~=QA9;WRC`cXBtKJ7d}%aId{8nmGj2sVJ6|?>ReC`&eo!@GOldeXOgvRFL@_2}a6D~GG(b5aZ9PLbBuhCzL3d4TCTwU(M>Q%xYiC7geN#I`DqbjMLu5X8LwhY%HBVklWM4ICQ9)E`KUYI=Lq%;kSXw+rd@E&CV^$_UJaQyiB`tbDW>+yjC_;8IaCj{>EoW~?X+v0hMm0xSa&9?jX=Y-3Pd-CCQf4hQb!RA5KVWl1cO+;nGBadqVm(?kZ%}V$L~(g?LvC6(b8=~KDr9jgO(;D?DLr~rYHnFYFit!nZgNXtNlRX1K{#hfL`7<4AwO|UNLp=ketL5?UwTn<L3&X(RY@j&NjWxXQYB<0L02|eIXzf7RdFd!ZB=GqW=2DFC3{tCKSq8(cymo>YkY55dwy0-OkO=EJ~et)C}So{RC7^jPAyt=H9|~HV>m@WY;Yu6V^uS3J5F9?B~e#;R##DFFep$*K2CO5Qbr^^F?%v_cQbQBDs@yiVMsJeU^zY^Dpx{CPkdu}Ic_pGJ4iW4T2n|#J1u2bK1w|`JuoCZMkz;5Dl;o@DrZ=6ZBt=zX=rgqdSf?UXIM8!Drj_mG&g-^DK}*(V?=c{NnuPfI7&+>Y-T$tOg2hxW;aMSBP(nvRYNIMV{%k=S!7N}ab-YvPH-xEWOqPqZ)i#*WoAA+c0F_|NNQMSX*5tMdPi?&U`c&GS0QXiQEGNsC^ab~GGa40c~Vk)MI=QkKy)ETO=dYnRB<#>eKA2ja9&nJe0y(fDt;wLH#JjnDP}xMX-`v0Y(9KZYeqj;dvA3$KsPvMB`HaGYj=GrYg9~YPIYEUdT(?zD0e7qVrV=hX*E_tYD8WsR&7XLdOc}*WnW`WDM(LZK}|MpJY!mSP+CK1YG`44U_Ue}St?^YHdI(WYg!{AEl@vBa(F&<Mr1^IDPl7rQ*Li6eQIe{KUXp`KrnPLK6pr2X+lCcJz{x!WimioDp6}ec``ygN<epSdS)^vLMClTcsnCdL_keoR4PF!Rd{MOcQ-IaUnC)PN_0I%B~B(sY;Q4cUOiAnGe<aRbxmw#XGvdvM^AEJeRm}-adbB|Oh8dGAy_vpd|E#^U`97XLp4J+N<b)cNKkc3U^^&QNkA<?W-}`xZ$2h5COIQVcWFp!Ur<>@XHrdLQ8QyWSZhWkLqBgYOKe$8a4C07W>Z9UDm6o1Okg>6WP3$qEmwRzRcl^5dP6f&es@_ZV`WQgc3EOQcQ|QSOLaDND^PrNL@8i4b2w8WT3%UgVK7NoM`%z;RA@D9dNp`bb7Em6I8=LWCP6A8G)yvKQA1#9Fd-yZH+Oh-MsQ3pXia8&a5y3gDJd>wWn*t-Whf$bbY?9$A}J{fUsf?UUs5$+NmNBmQy@JcC?`!tOixZHEFdRMNlZ&8EFdRSR8m1#LPb(iStl$YCs$8TOeZM{Usf?UUs5$+Mp8jTR7p=xAUz;4E-(sTRxvkUQZ-*wR6$flAUz;^3LqdLATT;0dm?OMb97&GbY*ZNIv_1EEFdCuVRC0>bRs$+MqzAoWqm9PARr(hF*+c7B5YxEbYF9HWpE-oAT2R0AR=^Oa%W|9B03;OVQh0{eJl!n3JMBjWo95>b}=_!ay4IbX=iR>bairNC~snODLM)uARr)fWnpw6Js@9mWnpwEZ(?&P3LqdLAZB55ZF3+!AZ%%KbSPhEWppTSVsk7YB4%N7ZF3?lAX{B2Aa8OYTU{v%ARr(hZ*XvLZe?zCAUz;vVRCJATQMLlAaiA5bX_26W*}^3ZYXA9a&2=dAU-`HG9YDab7dfVeF`8TARuFJcXJ>;AaiJCWpE%pATSCbARr)SZ*m}VZ+9SRZXj%Fb95+QXJvFKZ*XvLZe?zCEFdCuX>4V4A}k<VT`3@Mav)n>DLM)uARr(hARr)SZ*m}XX>4U6X>K5FX>)Waa&LDaZ*m}8T`4*WARr(hARr(hARr(hX=Wg9Z*(AOb7^jKbYX5|WhiuMY-KDUWNBk`DLM)uARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hV{dnJAS*o}X>N2VbZKm5E@x$QC?a8QX>DO_A}Js}Js=`OPgf!-3LqdLARr(hARr(hAaiJCWpE%XJs@drbSQLbY-KKIWppSaVQy({VQeBPAU!=GB2!33MNlFs3LqdLAaZ4Nb#iVXV{dnJEFg1eWo2**3JPRpW*}d7F*jdnb6;{bUuI!#X>55YZ(?&SAaitOa49+pARr(hb7f(4AUz;ob7f(4C~snODGDGUARu#eVRU66Js@9JF*jdQHD6OyK~zOsb7f(4T?!x|ARuXGAaitOa3DQBATS_rav*bbWpE%oAZc!ND06gSbY(7QWppSaY+-YBUvqS2a3U-qEiox6Itm~lARr(hARu#eVRU66Js^7`Y+-YBUvqS2a3VS&b97~JEFdCuVRC0>bRs$+MqzAoWqk@DARr(hARr)LRxvkUQZ-*wR6$flTXSV$bX_1lAaitKbY%)4ARr)fbYXO5TOw>>b97&GbY*ZNT_8Omb97~J3LqdLAZcbGZf|rTb97;JWiDrBbSNTpVRC0>bRsDrVQyp~b97~JAU-`HGBi30ARr(hARr(hV{dnJEFg1eWo2+6Js@9pF*jdwHD7aSXKrD1b#i4WZ(?&P3LqdLARr(hAZcbGb7*B{a3DTCAT%IhZe$>1Z+CMbJUt*YItm~lARr(hARr(hARu#eVRU6%B6MMLXJvFET_8OmRC0A?3LqdLAaZ4Nb#iVXVsCG3D06gSbY(7QWppSabYXI5WppAbDGCY-WMyU`Uv@D!UvzJGZeL_&ZDDR?Utx48Z(?&SAZc`EZ7d*jbY*ZUItm~lARuIAZDDR?AUz;4AZcbGX>?_6AR#><B1T10R7p%pT18SKAYpD~AaitOa3CchGBhAPJs>b3Wo&b0ATSCbARr)VW*~EPWpE%RAT%H$Js>bT3LqdLARr(hAaZ4Nb#iVXWMyq(Ze$7|ARr)gZ+C7WJs@9aWppTSVsk7YB6M$eZXzrodwnS&Z*m}eeF`8TARuOMav*bPZ*U-KZXj%Fb95+QXJvFKbZ>WVEFdCvZftL3Yh`3#b7*gHb0RDtTU{w2Z*m}8T`4*WARr(hARr(haB^>Cbz^jMAUz;oQ%FxxUr<s{L{&pnQ!Zy^bSQIZZ*VLiC@ColARr(hARr(hX=Wg4bY*QIX>K5Ja&KgHV{~&m3LqdLARr(hARr(hAY^52VQyp~D?K1GAZcbGY-MgJaB^>Cbz^jMDIh&PATc0iY;$EGF$y3cARuyObairWAY^52VQyp!3JPRpW*}d7F*jduX=7`3a9?s|b7gXNWhh}|bZKvHEFfugWo;=s3LqdLAaZ4MWpZ|9AUz;33LqdLAa8PHWpZ;MJs?|QV{~b6ZZ2nKbSNTbVRCI{aw04sTOv?FQ&S>cDJ&o=Y-w|JC}CrCX>V>WXJvFKB4}Z5WOE`ZAa8OYTU{w#3LqdLAZBlJAa8PHWpW^CZXj=RWMy)5Itm~lARr(hARuXGAZc@HZgX^DZewLAZ*pX1ax5SyY-w|JEFg4saBO8MDIj5PWFTy1ZYXbZWMy(GAU-`HG9Y1YWFT*HWMy(&FkK)$Js=`bNkdCjP$D2<Ze$>Da%5$4TQOZAJv|_4bY*Qi3LqdLARr(hARr(hAaZ4MWpZ|9AS*o}ZDDvQFf1TxZgePba%5$4TQXfKDIjTPAZ%rBC~tCPWpXJXK0P2aAZ2WGWgsyMARr(ha%FUNa&91UWpib6c4Z0*3S?zwAYXPdH(zo!Ut@1|ZggdGC~snOEFfWHbZKvHEFg1qWpF7v3LqdLAZcbGZf|rTUv@D!Uukn+ay4IOVQpz_c_?pUb1WcpbY*ZUItm~lARr(hARuyObairWAYo&4X>V={ARr(hW_5IRa%CVrAaitOa3CunG72CdARuXGAZB%Rb#i4OK0P38Wo{^6RxvkUQZ-*qK~hUaR8uKB3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)gVRC0>baNm*AbWiZARr(hW^ZyJZ*pX1av*7LAYWE7H(ydUUrj+$OGQ*uTV{21b#i50Itm~lARr(hARuXGAZ%rBC~tCPWpXJXK0P2aAYpD~Aa8PHWpZ0ET_8O@AR<#mOiUsmVQyp~Z*pX1a$7N7AZczOUsf?UUs5$+NmNBmQ#uMDARr(hARr(hARr)gVRC0>baPv8a%5$4TQOZ-AUz;-VRC0>baO6eWppTSa%5$4TQOZMATTK)D<ExQcqlL|AZc!NC~tCPWpZ0GT_A6AATTK@3LqdLAZcbGZf|rTbYXI5Wpr~o3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)NV{~b6ZXi7%Ut@1@d0$~;bZKvHC}CrCX>V>R3LqdLAZ=lCYh`pGJs?|bX>)WaZ*pX1aw#BYZ*m}Sa%5$4AZczOVPkY@Z*DGUWppSaZDDe2WppAeAX{B2Aa8OYTU}iWARr(hb7*B`AUz;tX=8LKUuR`>C|_q~bSQ6Pb1WbtaB^vOVRU68EFgP*DIjlhAbWi*AR=>UWn>~OAbWi&Aa8OYdwnSiARr(hW^ZyJX>?_6AZczOUsf?UUs5$+NmNBmQ#uMDARr(hARr)cY+-J0Wn>^dAar4JXJvGAE@x$QC~0(MZ7d)#DGDGUARr(hARuXGAaHD9Zf<2{AUr)FFggk#ARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hBOp>WK_G2!c4c!QbZBXFAYx&2Wgv55Y-J#HZy<AYWpFDoAa8DLc_4RaWo{sJZ+C7WWMyq(Ze$>2Z)I~JZf|r7ARr(hARr(hBOr2RW@&6}AarPDAaHVTWOZY7AYx@^Z*pZIX>=fAV{~&aARt3;b#8QJav*PRY<VDbXkm09V{Bz%ZXjb}b7d|HARr(hARr(hX=Wf_b}=_!bZ>WVUu0!%VQyq!VRR^OVsk7YX>?_6EFg1qWpF7VJ|Hk4Z*m}Cb}=_!bZ>WVUu0!%VQyq!VRR^OVsk7YX>?_6EFg1qWpE%XATcQ*J|HkU3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuLUX>)XGZf77pAaiwXC<-7TARr(hARr(hARujFcqlL|AZc!NC~tCPWpZ0GT_A6AATTK@3LqdLARr(hARr(hAZBlJAa8PHWpW^CZXj)8a%*LD3LqdLARr(hARr(hAZcbGY-MgJZ*pX1aw#A_Js>k6VQyp~Z*pX1a$7K6AU!=GB2z_7Od=p*Ze$>Da%5$4TQOZAJv|_4bY*P{ARr(hARr(hDGDGUARr(hARu9OVQFk(Vr*p~Js@picqj@WARr(hARr(hARr(xED9hXARr(hARr(hARuXObSQIZWn?aAWppTMbY*QUATTK)Z*m|oDGDGUARr(hARr(hARsLuWq4_GbZKs93LqdLARr(hARr(hAT1zYb}=_!aA{*}b#PyDWpib6c4a7GV{~b6ZY&^abY*QRED9hXARr(hARs9UARr(hARr(hadlyCbZK;XAUz;$X>KSAARr(hARr(hARr(hVRm6@Y++(-Wh@FHARr(hARr(hARr)ZVR$GpEFfuabSQFfb#7!RaBN|2Ze?U3Dj;80F*jdQHD5+jK|@qYPfjT*DJ%*gARr(hARr(q3LqdLARr(hAZcbGadlyCbZK;XAUr)FFggk#ARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hV|8+JWo~pJJs@sncyuTVARr(hARr(hARr(hC~tCPWpW^9Z*m}Sa%5$4AZczOZDDe2Wpp5EW*}^3ZYXbZWMy(GAU-`HGazAZWFT*HWMy(&FkK)$Js=`eMNCX0AYpD~Aa8PHWpZ0FT_8O@AZc`EZ7D1YARr(hARr(hARr(hPH%2yED9hXARr(hARs9UARr(hARr(hX=Wf}b#iiLZge1Nb0BVSbRbS|Ze=<OARr(hARr(hARr(hV|8+JWo~p^GF>1&AZ=lIC@?G_X>N2VV|8+JWo~p^GF>2Vav(4%DIhB#adlyCbZK;X3LqdLARr(hAZ2W6W*}^3ZYXVGa%*LDDIh!`F)%s`ARr(hARr(hARr(hZDDe2WpplKaByXAWGGuAQ$<WnA}k<jbY*QUAaQkJZggpMd0i<AARr(hARr(hWo&b0Itm~lARr(hARr(hARuFJZggpGb!7@5ARr)NV{~b6Zd)R4VRCC_bRu0KJs@pia%*LDTRJf?T?!x|ARuyObairWAYo&4X>V={3JPRpW*}d4Xkl<=C~jeGWh@|eVQh6}DLM)uARr)iVQh6}AUz;$VR$GoE-)-0W^8X^bSQRVY;|QRDGDGUARuXGAZ}r8WgtC0AR=sOZe?L|B035nARr(hARr)eWps6NZXkAHY;|P{ARr(hX=Wg9VQpm~Jv|^Ib8#X%3LqdLARr(hAaZ4Nb#iVXc42IFWgsdbc42IFWeOl5ARuXGAZ}r8WgtC0AR=>da&#g(3LqdLARr(hAaZ4Nb#iVXZDDk1E^~2mbSQRVY;|QR3LqdLAZcbGZeeX@AU!=GB5ZGGB035nARr(hARr)eWps6NZXj)8bZ9PYZ)Y)ZD0X3Nb!90EARr(hX=Wg9VQpm~Jv|^IY;R{VFd{k%ARr(hARr(ha%FUNa&91PVRUFNY;R{VFeouDFd!=+c42IFWhn|EARr)eVQF(^AXZ^)b!A0za&K}dZeeX@DGCY-WMyU`Uu|J>Yh`p_aB^v5WhiNMWo;}VX>N99Zgg*Qc_}&yARr(hVqtS-EFfiZb!lvAVsdG9Z7d*jV_|G%EFfZKY;Sj8W_503EFfZKY;Sj8bYXI5WppecVPbD~WnX4>ZeuJUVPbD~WnXk*a%W|9AUz;oO+iviMO0r<K~h0YQ(I|tWo=ywARr(hX=Wg4ZgypEbZ>HbAUq&tadl~IX<~9|b!|EdARr(hARr(hVQp}1X>@gDWgtBuVr6V^cVBd2a%W|9ASxhYVRL05FCbrYXkl<=C}L%7Z+Bm2b#7xUAai43Y-K45ARr(hARr(haB^v5WgtBuVqtS-AS)nYZE$R9baiB9ASxhVb7*03Whi20Y;Sj8W_503EFfiZb!lvAVsdG9Z6GZmX>N99Zgg*Qc_|7YARr)RY;$Eg3LqdLARr(hAYpBAY-x0LWMv>dAYo!}c4c35VRC0>bRa4qVqtS-ATJ<ab7*03Whh}{Z+2y0W_503EFg1ZVQgh73LqdLARr(hAaHVNV`U&cAYx&2WgsmeVQp}1X>@gDWgsdbUvp?-aAhcAVsCb3UuJb~V=N$PZgypEbZ>HbAT1zeadl~IX<~9|b!{mMARr(ha%FUNa&91PVR$HCP*O=lMPEitPft=TAZc!NC~|LgZe%EMa%p2_DJdxm3JPRpW*}c_b6<01Y-}iRa%5$4DLM)uARr)eWps6NZXhTMARr(hARr(hX>)0Ab97;DV`V6Ba%5$4EFdUsX>)WeAar$bY-K4a3LqdLARr(hAYpD~AZ%rBC~tCPWpXJXK0P2a3LqdLARr(hAYpD~Aa8PHWpZ0ET_8O@AR<#mOiUsQARr(hARr(hVQyp~Z*pX1a$7N7AZczOUrj+$OGQ*)P(e~bO;ZXWARr(q3JMBjWo95>X>D*}V{~70V{dY0C~snOEFf=kWMy(GItm~lARuXGAZ~ATAYW;7Uvp(_Y$$JXWMy(GItm~lARr(hARuyObairWAZBcDVRR@WEop9MA}I<WARr)VbY*QIJs@**awu<dWMy(&F<mJNARr(hbaHt*3LqdLARr(hAaQkJZggpMc_2L?ZDDvQFf1TxZgePba%5$4TQXfKDGDGUARuLUV`Xr3AShIMaAieua&K}hAXZ^)b!A0za&K}eItm~lARr(hARuyObairWATTa43LqdLAZ=lCYh`pGJs@9aWppTSVsk7YB5h%EYh`pIEFgP*DIjlhAbWiZARr(hX>N99Zgg*Qc_2L?UuR`>C~aYKYh`pSAR=jQc4cmKZ*qAeEFgP*DIjlhAbWiZARr(haB^v5Wpf}sAYW%?bSQ0Ma%*LDEFdCqa%p2_b0RDtdwnS&Z*m}eeF`8TARuFPa&l#EbYE$1c4cmKZ*qAcJs@drbSPhEWppTMZgypEbZ>HbEFfugWo;}VF)%PNFexB!av(4%3LqdLAY*lMa%FCGUvYJBbY&ntAZBcDVRR^8XJvFKaB^v5WpgYbX>?_6EFfQPVRCC_bYF0CX=7z5X>?_6EFfcba&l#EbYE$1c4cmKZ*qAlDIjlhATTKkARr(hY+-a|a$j+EZ**lKJs@UmZ((#OUu|J>Yh`p_aB^v5WhiNMWo;}VV|8+JWo~p|X>N99Zgg*Qc_1qwadlyCbZK;XDJcpdARr)eWps6NZXjlCZ((#OadlyCbZK;XDIh8!ZDDvQFfK4GAY*lMa%FCGUvYJBbY&ndAZ%fDWpZC}b#HWKDGCY-WMyU`Uu0!%VQyq!aAk5|WMO$IZ(?&SAY*TCW@%@2a$$67Z*D9gX>?_6DLM)uARr)gZ+C7WJs@9aWppTSVsk7YB6M$eZXzrodwnS&Z*m}eeF`8TARu#SZ*X%UJs@mpb95+QXJvFKbZ>WVEFdCvZftL3Yh`3#b7*gHb0RDtTU{w2Z*m}8T`3A6ARr)gb#iWVUvOn|Uu0o<AUz;yZgePLXJvFKV{dL|X=inEVRUJ4ZY&@obairWb5Lb+L}7U%EFdy8DIjlhATl&53LqdLAaiJMa9?R|bY*gOVQe5hAZ=lIC^0M`X>N2VUuR`>C}VGKW@%@2a$$67Z*D9gB6M$eZc}J)a8qS$Y)Ni(WpZ|5Y$7ZmG$|l&av(G*DGDGUARuIAZDDR?AUz;3E-(rpARr)SZ*m}WXm4;JX>K5MXm4<HItm~lARr(hARusZZ)A02baNm*AYW5RPf%Y_QcpxxLsU~PXJvFKb7*gHEFdT;DGDGUARr(hARuXGAZc`EZ6IlGAaHVTWOZY7b2<tjARr(hARr(hARr)QWo=<@WFRX&ASiToa&B{9aAk5|WMO$AFCcShZ*X5}ZggdGc42HOASxgzG9YPYAZ%rBC~$IbWOZY7b15J_Js>e4Wo&b0ATcQlARr(hX=Wg4bY*QIAw3`>MnzIoNlZyvMN%R<3LqdLARr(hAY)~2bY*g1X>N37a&}>CAUz;$VR$GpEFfuabSPhEWppTGZ*FF3XLWL6bZKvHEFdCuZ+C7(Wo~q3a#Lk&Y)Ni(WpZ|5Y$7ZmGBhb5Z*m|qG$|<xARr(hARr(hWMyq(Ze$=UJs@;-a&B{9aAk5|WMO$AFCb%OZggdGUukZ1WpZ|5YziPCARuyObairWAY^52VQyp!3JPRpW*}d0a%5$4Uvpz`a%CuQVsk7YV{dL|X=inEVRUJ4ZY&^ga%5$4DLM)uARr)fV{dY0AUz;oX>D*}V{~70V{dY0C~snOEFf=kWMy(G3LqdLAZcbGb7OCEWgt8~ATS_rav*MRbRb`8b6<01Y-}iRa%5$4DLM)uARr(hARr)eWps6NZXk1GZ*pY{ARr(hX>?_6AUz;+baE(fa%5$4TQOZJ3LqdLAaQkJZggpMc_2L?ZDDvQFf1TxZgePba%5$4TQXfKDGDGUARujFa%*LDAUz;oXJvFKZ(?&SAR=vHa%*LDA}k<#eJLPsav*zs3LqdLAZc!PWo~qDa(N&<AYW%?bSQ0Ma%*LDEFdCjZgypEbZ>HbA}k<#eJLPsav*zs3LqdLAY*lMa%FCGUukZ3Wo~qDa(N&<AZc!NC|_q~bSP<Vc4cmKZ*qAoAZc`EZ7d)$FfcGMDIjlhATTKkARr(hWMyq(Ze$=mAZ=lIC@?NEH7p=sWMyq(Ze(9@WpZC+VR<NTVsk7YV{dL|X=inEVRUJ4ZY&^abY*QRDGDGUARuLUV`X!5AUz;$VR$GoE-)-0V|8+JWo~p|X>N99Zgg*Qc_1qwadlyCbZK;XAT1y<FfcGMDGDGUARu*eXJu|<c_2L?ZE0>OF)lDHASh*cV`X!5ATJ<fWo=<@WGNsoATcm5FewTkARr)eWps6NZXk1GZ*pZIDj+B^E-)Y~AYVj9O+ijXUqMVzNI@VfAa!zQWo~16DGCY-WMyU`Uvgn?YhQC^Y;0e1Y;SaPC~snOEFfWHbZKvHEFfcVZf0p`b#h^JX>V>RItm~lARu95bZKvHAUz;oV{dSIUtwc(X>V>QVPkY@Z*D0HARr(hZDDe2Wpp4tAZ%%KbSPnCbZKvHE@x$QC?aiPa%*LDA}Jtmav)n>DGDGUARuyYcXJ>;AX^F`ARr(hARr(pUvF|`WpZD0V{dY0C~snOEFfcVZf0p`b#h^JX>V>UAa8PHWpXJjAT4QbWMz0PAZ%%KbSQ6fWMy(GDGDGUARr(hARuOMav*7LWMz0PAa8PHWpW^CZXjiDb!}yGVRU6EZDDe2WppVDARr(hARr(hX=Wf_X>(t5Wo&FHZ*pX1aw!TRARr)J3LqdLAZcbGY-MgJa&LEYDIh!`GCB$%ARr(hARr)eWps6NZXjV}bZKvH3LqdLAaZYab1rjla&#ziWp-t9b7ehLa&=`X3LqdLAaY@DYh`30Js@dxWpXHTZ+BZVT_9#}av*YVcOYqQAaZYab14cSARr)NV{~b6Zd)R4VRCC_bRu0KJs?|dWq5Qba$#<3Wn?KJX=Wf_X>(t5Wo&FHZ*pX1aw#BXY;$EGZ*pX1av)}Jav*PVWMy(7X>K5GVRCC_bX^J{ARr)eWps6NZXjV}bZKvH3JMBjWo95>bY*gFX>MU`Uu<b{b!lW_bZKvHC~snOEFfWHbZKvHEFg1qWpF7v3LqdLAZcbGb97~JAUq&9F*Z61ARr(hARr(ha%FUNa&91DV{~b6ZVDhEARu95bZKvHAUz;oV{dSIUtwc(X>V>QVPkY@Z*D0HARr(hb7*B`AUz;oXJvFKUuR`>C~snOEFdCqa%py9bY&teAbWi&Aa8OYdwnb*B6DbEWFjmedwnS&Z*m}eeF`8TARusTVQy|^WFS2tdueoKZ8{(@AZBlJAZc`EZ6IlGAYW5OOiV#SOhtVPARr(hW^ZyJZ*pX1av*7LAYo&4X>V>WXJvFKB5h%EYh`pIEFfE5DLM)uARr(hARr)VW*}c_b6<01Y-}iRa%5$4DLM)uARr(hARr(hARr)cY+-J0Wn^1(baE(fa%5$4TQOZJT_7txAZ=lIC@?G_X>N2VZ*pX1a$7Q8DJcpdARr)SZ*m}MbY*QIX>K52Oi58yNkl<ZNl#8+Pf|ohQaTDCARr(hARr)Nc4293VPb4$AUz;$VR$GoEFfuabSPhEWppTWXk}z9AZc`EZ7d)#DIjlhATTK@3LqdLARr(hAZ2)Ta$z7nAYpc4X>4I)Y-J#6W*~EPWpE%qJs>wRI3Q(gb7dfHVR$GoEFfWaVQFk(Vr*p~Eg*1gVQy|^WLs%;Wo=z43LqdLARr(hAZcbGWq5RQVIW~{WFTy1ZYW`6bZKvHTOw^?a%*LDB3&sUJRmVJItm~lARr(hARr(hARu95bZKvHTOw^?a%*LDB3&+FaByXAWGGuAQ$<WnA}k<jbY*QUAZ2)Ta$#L53LqdLAaZ4Nb#iVXVPkY@Z*B?-3JPCVF*jdLL|;uoQcFctQy@JcYIARHE^KdMWOFEbY-wUHWMyM-ZE$jBb8{$SVRL0RG%jK|HDqODZ)9aC3LqdLAR=QeaeZ|rGDuE4G%-&xC1pk=VK_@?Sw%M{d~t0kcttB^Mm1w(IahWwHF`uUWME!HaZO`)b2NQ^aU?cqC}(F*OEF7yW;S<hK4n%mCNW=gU^yyxKSNhFKU!gIH9S2}H8My~bS+R`H$+f*N-0oUZa;b}GCd_^Xm)!%d{;_PC}cxKGk1J`a#&v@SYjkVIVB-^dQDI$KzmDddrUt!H$PHPI9EGOX?!wpL?J0)Z+tm)d3_~xM|*H%WqW;fN@GM#V?kbELs5QmZ!%dyYIsaxbwf8~UV2p~Eq+#7e0fbIXHQc%LRxhxD>8RqMtM;wF-T@?Au(P+Qae*bBtdK~IA&5aePm2*LQY^UXG=0iHA!+&W_ws-Vq!;1Z$nplR$f|OUtvaPV>?MoY;Isxa4<VhBW*QPXkceWOLSRRS5`7VWJD-ALSrOlSAK7ES~)5;JtZwpF(FnkOmsVbF<w<QT1{7OOmb;iAtfPTOEpp^T0d=5JwQMudS7ioJ7Pk3X<0l)IDR1}Urka<S2ap%I5{#wRC-@oDmhR_YcozvV|-OYKqzuOLNrBIN+fi7F=%Z>GD1ImHbrPzXjD0Mcr8t6Xd_oUXL3g)WJF3*b24dsVK#I<YFH$DGeKHAY$YRWMmTXqdVFzkN_##ePfjIKG(AlyFfx20PkvZyN>?;lJycpla9%l0CPjNxPG~+gRX-^-el~JHF>p6PaYaUID<v&hN;hyJG<;?;HDYf`bT)2uUpZDuJ|#jwVmC}`WMEc)c1SsCH+f4oay)QqbvSNtN<}MUd|5U*d}M4sCN@<jc1}r0SVALxQ$TTkWp`<JXe~BlHgaW1Dt>Z5D^f{MZfijyF?dsNR5N8>Z9+^_FlS?NW@Bz`W;I_*aClfec79J;D>)@$J|rtgOkaB`a3phYc{gY`Us+mIN=hhMM{jRPVoo7wX?ZDAH%2x!Y)VmnUm+znQg?lFOJY}5KtXSLad2lmF)?#wOE@!5D=1G{Qd%KSSXNIqZ+t-{C~7k~IVwj?XIM>DXh&pYW=dsNRVi#!c}_PgQ*U>BFl~5vUnOc;BXlWMQbutzR&*wDRZ%!@V?RJzQ(sV6dtzi&c2PKFR$f3vJYqO&ay?E^WkOI#ZcKG@dSh==c`{9BU@B>PHCJGBDN|}BYdA|tK|o<?T18_^J!@V?HELNZR4{8hKy+kHJz7;-VOUW%CL=<7U|CN-FfuJQK6FbqLt-jdFi~M_Xi;KvVk%N)PBK?xbVxKmZYVQNFhOiWWpO58Ja}(qc2H@4HG5%NYJ5~wczz~LS7AgsWLQ2XM<i=RS#@PVP<>B5U^YHxcQs!|bX0tKDr+-ybuc9|WoUMMdO%B9cVIDQR3T$bd|qi|OL9a?RcA&$R47w@K50lMF=tL*J6U>3aBwC+Y;!hsOmtCJGdpoLY;blqB{E}rcy?YmYFBewK{8=UOeiXIC}&S%Hb_KjZY458IbeQHKPpC2Mrv$uA!}@XP-;O)S0rO8JyaoMV{}MhV0uPuM<g*pICynHLsoBRRcKCVWJ+3CBwsv6VK5<NDKuDhNqkyzUU5KrUQT3kL}g!MetT{_Q*t<BIZAd@a%FfnbtP$bV__<LcXnndZ)|NLS~)d&cy~N&Ku$+}az1r=R$p3YHcmHICPy@3S2SKBcQb1}aArp^IaY5<W+f;%HF05Pcwk~pQ9CGPL^x+zcVua1CVW6zG%<R7ZDclDKz<=*Sa*JPGk0QoZb3|AML1<pAx1@Vd{tOJF;Pu-Sv*I3Noh`eEogQ|PHQGBUUD}~Qao*1BuzL{eP(hhODT42Zge+GZy{%FCU{q4Lw!$IM>|=1H&tp$J$OxSC1ZYlWMFtrJ~LNRRXA5(GB|rqc|&hAI4engBr!!WRy{LgBX4ssD|AmSabr|kN+d}|IA%mEbyG-RB|S-SEh|ZKT7G+PP(NTdDp@}{eS2_sJv>e-V?9}TawK|QX=pererPFbaeHw<aaU*|Y;8zzbZ%=%Lm@w7V_-vNFj{g?He)0}W>X_=X>&O+UqpE_dwgeRWO`LAY<xp~du)DocWz&DSUG(rCNgbrBw=k%a8GPaCUZwBKx%I!LvBz|YCcFcb1F_dPIh2RKV^PxDRon3VR&a>BY7})R%A?9Sy4biLSHgoBur;3WOOhwDra;~Q*SvxPHlKeYgQvYQzbn}Mq^(*JRx9GUU)-EH%m8CZz@1acTGW9Z)QC(Y(g_fdqaCEbXh)7L{~{NLs(d4Z&Wo=Rd8TaQ({VfVqj=OS8*yfW`03xLrW!nK5%+dMrtxkb~rpWJ!xtp3MnZrWMyM-WMwEKb#!JeI3g)23SU++H(yOeUq(_vLsUsmP9QxXGA=L*Usf?UUs6v`O<zY<K~h8@Js?zab!7@)RxvkUMny$LUq@9zQbZs<AVy(qb7cx&RxvkUO+;TwR7FiwAUz-`B27h1Pfj8%AR<jkOiLmxAR<##QbAWjMN(2(A}k;xS5Hq&A}I=ARxvkUO+;T)R6$flAUz;^3LqdLATT;0dm?OMb97&GbY*ZNIv_1EEFdCuVRC0>bRs$+MqzAoWqm9PARr(hF*+c7B5YxEbYF9HWpE-oAT2R0AR=^Oa%W|9B03;OVQh0{eJl!n3SU++H(y3YMMPgxMN>mnMPE}?K~zN`Js^7uARr(hFghT6B5YxEbYF9HWpE-oAT2R0AR=U8c_KO>Eio)0B4J~6X?A5IIv{&}eJlzfARr(yIv{%@Y+-YBUvqS2a3VS&Eio)0B4lBCB03-~F)Sb=VPkY@c4Z<uAbWj%EDC)JUsf?UUs6v`O<zS;K|^0tR6$flAUz;^3LqdLATT;0dm?OMb97&GbY*ZNIv_1EEFdCeVR<4tAT2R0AR=L7bZK^FB03;WZ*FCMED9hXARsY1AbTQgVRLj}b97~JB03-~F)Sb=WMO$CIv_1EEFdCbV{~bDWg<EtPH%2yeJl!n3JMBjWo95>b}=_!ZDe0_X=iR>bairNC~snODLM)uARr)fWnpw6Js@9mWnpwEZ(?&P3LqdLAZB55ZF3+!AZ%%KbSPhEWppTSVsk7YB4%N7ZF3?lAX{B2Aa8OYTU{v%ARr(hZ*XvLZe?zCAUz;vVRCJATQMLlAaiA5bX_26W*}^3ZYXA9a&2=dAU-`HG9YDab7dfVeF`8TARuFJcXJ>;AaiJCWpE%pATSCbARr)SZ*m}VZ+9SRZXj%Fb95+QXJvFKZ*XvLZe?zCEFdCuX>4V4A}k<VT`3@Mav)n>DLM)uARr(hARr)SZ*m}XX>4U6X>K5FX>)Waa&LDaZ*m}8T`4*WARr(hARr(hARr(hX=Wg9Z*(AOb7^jKbYX5|WhiuMY-KDUWNBk`DLM)uARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hV{dnJAS*o}X>N2VbZKm5E@x$QC?a8QX>DO_A}Js}Js=`OPgf!-3LqdLARr(hARr(hAaiJCWpE%XJs@drbSQLbY-KKIWppSaVQy({VQeBPAU!=GB2!33MNlFs3LqdLAaQkJWO89{baNm*AZ%rBC|_q~bSQ6daBps9ZgealB6V(TZ)0m^WM6T0VPtY)Zgg`ZEFfE5DIjlhAX{B23LqdLAaZ4Nb#iVXV{dnJEFg1eWo2+IAaQkJWO89{baM&{3S?zwAYXPdH(zOUUu|SxW?^k<Y<VbeVsk7Yb97~JDLM)uARr)fWnpw6Js@9mWnpwEZ(?&P3LqdLAaitKbY&ntAYWE7H(yOeUsF^;R7G2JWnpw(3LqdLAZcbGb97~JAU!=GFd%PoAaitOa3DM&X>N2Vb97;JWiDrBbSNThVRLj}b97~JA}k;+F)1lJ3LqdLARr(hAaitKbY&ntAbTQgVRLj}b97~JB03;*bY*ZXAR=^Oa%W|9B03;OVQh0{eF`8TARr(hARu2>F*jdLL|;=>K~zOsb7f(4T_8Omb97;JWeOl5ARu#eVRU6%B5YxEbYF9HWpE;0AUz;+bY*Y~ARr(hX=Wg9Z*(AYbYXO5E@x$QC?a%Wa%W|9A}JtYZe$>HbY*ZLK0P2YHZVF0ARr(hARr(hV{dnJEFg1eWo2+IAaQkJWO89{baNm*AYXPdH(zaJUvp_^ZeetFa%CuQVsj}9ARr(hARr(hX=WfOadly2a$#<Cb09uFATl6fZe$>1Z+CMbK0P2bAYpD~AaiJCWpE%oJs>hEAa8OYV{dnJAU-`HIXVg;ARr(hARr(hARr)fbYXO5TOxE}a%W|9B3&RoAXIX7WeOl5ARuyObairWAYyNCY$$VdVRU6KXJvFKB6MMLXJvFEDJcpH3S?zwAYXPdH(zaJUvOz-Yjtp6a%FR6a&~1XVPkY@Z*D9gX>?_6DLM)uARr)eWpib6c4Z(vATSCbARr)SZ*m}Sa%5$4AZczOTVZ2#X>V>WXJvFKB4%N7ZDn#IEFfDVP(f2uB3&sgAS!HWb95+SV{~b6ZZ2nKbSNTdVQyq|A}Jtmav)n>DP1}WARr(hARr(hX=Wg4b7^jKbYX5|WhifQWMy(JASi5Ub95{qbailSWhp5jVQyp~Y-MgJZ*pX1aw#A_Js>h5VQyp~Z*pX1a$7K6AU!=GB2Y;~OI1)JAYpD~Aa8PHWpZ0FT_8O@AZc`EZ8{1dARr(hARr(hARr)eWpib6c4Z(dJs@picqlL|AZc!NC~tCPWpZ0GT`4IbX=Wg7Wo{^Ma%5$4DIh*QATuCkY;$EGF$y3cARuyObairWAaZ4MWpZ|93JMBjWo95>b}=_!ZDe0#Z*^{TWpXHQVsk7YVPkY@Z*D9gb97~JDLM)uARr)VW*}cyF*jdLL|;ZyK|@qYPfj2_Js>b3Z*m}RZ*(AEb}=_!X>(s~WM5`sZE0+IC~snOEFg1qWpF7VZ*m}WbY*ZLD<Cl-K0P38Wo{^6RxvkUO+;T!K~hUaR8uKB3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)gVRC0>baNm*AbWiZARr(hW^ZyJZ*pX1av*7LAYWE7H(yOeUrj+$OGQ*uTXS?}a3CunF<m+eARr(hARr(hX=Wg7Wo{^Ma%5$4DIh*QATuCgZe$>Da%5$4TQFT9Jv|^IQ$<WnA|PRIWFT*HWMy(&F<l^OZXjP)F*jdLL|;i%MNLyW3LqdLARr(hARr(hAar4JXJvGATW@k?WpZ0FU0onOAar4JXJvGAE@x$QC~tCPWpZ0FT`V9lDIhB#ZDDvQFf1TxZgePba%5$4TQXfBZ*m|oDJcpdARr)VW*}~FbRcwLa%W|9b2<tjARr(hARr)eWps6NZXjV}bZKvH3LqdLAYo&4X>V>IJs@9WZ*X~EVPkY@Z*C}IV{~b6ZYc^NARr)ZVRCC_bRaz-TWo1_bSQ6fWMy(GAZBlJAa8PHWpW^CZXhUOV{~b6ZZ2nKbSNTiVRCC_bRsDrZ*m}8T`64(ARr(hb7*B`AUz;tX=8LKUuR`>C|_q~bSQ6Pb1WbtaB^vOVRU68EFgP*DIjlhAbWi*AR=>UWn>~OAbWi&Aa8OYdwnSiARr(hW^ZyJX>?_6AZczOUsf?UUrj_`NmNBmQ#uMDARr(hARr)gVRC0>bRaz-bYXI5Wpr~cXJvFKX>?_6EFdr`3LqdLARr(hAZcbGbYXI5Wpp4sJs>bT3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuLUX>)XGZf9R{bzyFFX>@rYJs@*+Z72#LARr(hARr(hARr)ZVR$GoEFfuabSQ6fWMy(&GF>2Vav(4%DGDGUARr(hARr(hARuOMav*PVWMy(7X>K5GVRCC_bP6CKARr(hARr(hARuXGAZ%rBC~tCPWpXJXK0P2aAYpD~Aa8PHWpZ0ET_8O@AR<#mOiUsmVQyp~Z*pX1a$7N7AU!=GX>?_63LqdLARr(hASntUARr(hARr)Nc4293VPb4$AUz;$VR$GCARr(hARr(hARr(hFf0lnARr(hARr(hARr)VZgePfXk}zBXJvFKX>?_6EFdr`Aa8OYFewTkARr(hARr(hARr(uAZ2)Ib98BLXJ2u3VQzG3ba@IOARr(hARr(hARr(uAYXPdH(zaJUvOz-Yjtp6a%FR6a&~1XVPkY@Z*D9gX>?_6DJ%*gARr(hARr(q3LqdLARr(hAaQkJZggpMc_2L?ZE0>OVRm6@Y++(-Wh@|VVR$GpEFfuabSQFfb#7!RbYXI5Wpp4aAYWE7H(yOeUq(_vLsUsmPAMrVDGDGUARr(hARuXGAaQkJZggpMc_2JJATT-#ARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZ2)Ib98BLXCOTwZe@6MC<-7TARr(hARr(hARs7ja%5$4AZBlJAa8PHWpW^CZXj)8a%*LDAZcbGY-MgJZ*pX1aw#A_Js>k6VQyp~Z*pX1a$7K6AU!=GB2z_7Od=p*Ze$>Da%5$4TQOZAJv|_4bY*QRED9hXARr(hARr(hARta}Ze=VAARr(hARr(hDGDGUARr(hARuXGAZ2)Ib98BLXCP^FAZ~ATAWm;?WjYEVARr(hARr(hARr)RcxiKVX>Mm*GF>1&AZ=lIC@?G_X>N2VWq4_GbZKs9TQXfBZ*m|oDJdW;AaQkJZggpMc?uvPARr(hARuLIX=Wg7Wo{^KVRCC_bSWS_ATcmH3LqdLARr(hARr(hAZ=lCYh`pUVQ_F|Ze%E1B2z_7Od>2GX>?_6EFf`pVQzG3ba`DV3LqdLARr(hAZ2WGWjYEVARr(hARr(hARr)PZ*FvHZgph}ARr(hVPkY@Z*E&6ZDDe2WppB4AUz;$VRCC_bXz(xFkK2DARr)eWps6NZXjV}bZKvH3JMBjWo95>b}=_!ZEtpEUvzJGVRB?BaBp*IbZKvHEFg4Ya%W|9DLM)uARr)kEFgIxJs@drbSQ9db7^#GZ*E&KT`4ReX>N2VaBp*IbZKvHTQOZJ3LqdLAar;vAar>kJs@drbSQLTa%W|9TQFTIEFfuabSQLTa%W|9TQOZJ3LqdLAZcbGcpy9=ba*-nARr(hARr(ha%FUNa&917B1J({R3cppARr(hX=WgJAU+^;csdFoARr(hARr)eWps6NZXjDCS4C4)B3%j~ARr)VW*~VWJRo#=Itm~lARr(hARuyObairWAX_3+PgPV%B3%j~ARr)VW*~VWJ|J{?Itm~lARr(hARuyObairWAX_3%Pf}D!B3%j~ARr)eWps6NZXjDCP(f2uB3%j!3S?zwAYXPdH(zFDWn^DxbzyR3C~snOEFfWHbZKvHEFg1qWpF7v3LqdLAZTxOav(h*X>N2VUuR`>C~snOEFdCiZ*_7aEFdr`Aa8OYFewTkARr)QVR;}uAZc!NC|_q~bSQ6Pb1WbtWMO$CEFg1qWpE%bFCa2BDIjlhATTKkARr(hX=Wg9Z*(AERxvkUMny$LUq@9zQbZtcav*4Lb#fp)ATc;P3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)NV{~b6ZXi7%Utw%%XKr6;VQyq|C}CrCX>V>UAa7!GDGDGUARu#PVRRrpAYXH3VRR^OVsj}9ARr(hb97;JWgtBuUsf?UUq(emL|;-xQ$tlnUsF^;R7G2JWnpw(3LqdLAZcbGb97~JAU!=GFd%PoAaitOa3DM&X>N2Vb97;JWiDrBbSNThVRLj}b97~JA}k;+F)1k^Z*m}HVR;}SJs@drbSQIlVRU6KXJvFKB4lBCA}k;+F)1lJ3LqdLARr(hAaitKbY&ntAbTQgVRLj}b97~JB03;*bY*ZXAR=U8c_KO>WMO$MAR=L7bZK^FB03;@eSHcbARr(hARr)LRxvkUMny$LUs6R=LsdmzQ&d4zMO$-aVRT&}Js@**VRU5*ARr(hb97;JWm_U_VRLj}b97~JB3&RoAaitOa0(zGARuO8a%~_zAYW!-a&0JYVsk7Yb7f(4DGDGUARusZX?9_BWgtBuUuR`>C~snOEFdCqa%py9bY&teAbWi&Aa8OYdwmKZARr)cZ*ysMX>V?GAUz;kUuR`>C}v@DZ7d)nW?^z|WpW}cAX_vnAT(VmEFdauX>)WaUuR`>C}v@DZ7d)nXkl(-b0RDtTU{w2Z*m}8T`64(ARr(hX>N99Zgg*QX=QUDJs@mpb95+QXJvFKaB^vOVRU6IAR=jQc4cmKZ*pm6b0RDtTU{w2Z*m}8T`3A6ARr)ba%5$4b09q+TVZ2#X>V>WXJvFKB4%N7ZDn#IEFfDVP(f2uB3&sgAS!HWb95+SV{~b6ZZ2nKbSNTdVQyq|A}Jtmav)n>DP0N*ARr(hbZByAVRUmKJs?|M3LqdLAZBlJAbBhxa&LDaX>K58Zgp*Ca$$63C~RqSbSPhEWppTJVRCIOAR=^WY-MvIEFfE5DIjlhAX{B2DLM)uARr(hARr)SZ*m}bEFg4gY-J#6ZXjiDb!}yGVRU6EY-w|JC~|LiAa8OYTU{wBItm~lARr(hARr(hARuXGASenTARr(hARr(hARr(hARr)Vb7^jKbYX5|WhiuMY-KDUWNBk`DGDGUARr(hARr(hARr(hARu9GWFT~DY-KKIWppSaVQy({VQeBP3LqdLARr(hARr(hARr(hAYpD~AZc!ND0FFTWiDrBbSNTYZ*FsCV|8?Cc4c36Zf0d<A}k;<DIjlhATTK)K0P2Y3LqdLARr(hARr(hARr(hAYpD~AZ~ATAarSLWiDrBbSNTbWn^D;Z)9P4A}k<AVQh0{DGDGUARr(hARr(hARs9^3LqdLARr(hARr(hARr(hAarPQWnpx4E@5zRWo~3BD0nO&c_}FhARr(hbZByAVRT<}Wpp4tAaiAOD0FCYWnpx4DGDGUARu95bZK^FAUz;+bYXO5E^}pcWMyVyb!>DfB4J~6X?A5IEFgP*DGDGUARuOMav))2bZ>GjAar4JXJvFCX>K5FX>)WaVPkY@c4aPUbY*RGC@CpA3LqdLARr(hAYo&4Z*m|#AZc!NC}CrCZ*nOLARr(hARr(hX=Wf{V{~tFAU-`HY-MgJaBp*IbZKvHb15Kiav))2bZ>GXK0P38Wo{^GZgypEbZ>HLWpgPYZ*m}Xb#QEDD0E?RXJvFLAZ~ATAZczObZByAVRT<}Wpp|UARr(hARr(hARr(hVPkY@c4aPbZ*VAKV{~tFEFeyAZe=M7ARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZc!PWo~qDa(N&<AY^G{bSP<Vc4cmKZ*pm6b6a6!bZ>H9Aa8OYdwnSiARr(hARr(hX=Wg4ZgePVZgypEbZ>HbE@x$QC?Z!#ML|>|EFdr`Aa8OYFexBBJs>bT3LqdLARr(hARr(hAYo&4X?A5UaBpxZVPkY}ax5TDZ*FBN3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuXGAar$bY-K2LZ*ysMX>V?GTVZ2#Z*pBJAU!=GbailSWhiuEa%W|9DLM)uARr(hARr(hARr)ba%5$4b6a6!bZ>H9AUz;kB1T0;L?T@ZARr(hARr(hWo&b0Itm~lARr(hARr(hARupYWMy)5TVZ2#Z*pBAJs@9pF*jdrZ+2y0bZ>WIa%3oQZ*ysMX>V?GTVZ2#Z*pBMAar4JXJvFL3JM?~ARuFGVQFn;WFS2tdvtYhY-K2PVRC0>bSWTaZ*m}XVRC0>bRcPNAYo&4X?A5Uc42IFWpgMgeF`8TARuyOZDDC{X>Mm<VPkY@Z*FrSJs@picqlO}ATl%{Eg)!bb#f^RARr(hW^ZyJbYXI5Wpp5EZXk4Ma%Ew3b2<tjARr(hARr)VW*~H7a%W|9AZczOV{Bn*ZDnLS3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuXGAYpEKC<-7TARr(hARr(hARu&gaBO8LaBp*IbZKvHDIh&PAar4JXJvE>ARr(hARr(hARr(hVQyp~VPkY}av(e)Y-MgJZ*pX1a&svPARr(hARr(hARr(hVQyp~Z*pX1a&uc@V{~tFT?!x|ARr(hARr(hARu9GWFT*HWMy)5TVZ2#Z*pB*FkK)$Js=`RMMXp+3LqdLARr(hARr(hAZBlJAYo&4Z*nXkaBp*IbZKvHAZczOWo~tCWpZJ3WhiiOb7^#GZ*Frb3LqdLARr(hASpTuARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAY)-}WNBn!bY*iOJs?|M3LqdLARr(hAZBlJAYo&4Z*nXkaBp*IbZKvHAZczOWo~tCWpZJ3WhiiOb7^#GZ*FrbItm~lARr(hARr(hARuXGAYo&4Z*m}MZXjV}bZK^FAa8OYVPkY}av(lEAZ%rBC~0nXWo~qDa%p9ADLM)uARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hX=Wg4ZgePQX=8LKX>N99Zgg*QX=QU;VPkY}a$O*Aav*zsDK2MabSNTMNJT+ZA}k;<DIjlhATTK)JUt*VItm~lARr(hARr(hARr(hARuFJZggpGb!7@5ARr(hARr(hARr)QX>)X8ZewL2Js@FXb0}$UbSQ9db7^#GZ*E&KT`3?fAar4JXJvF-FkLAiD<ENFb0}$UbSQ9db7^#GZ*E&LT`3?fAar4JXJvF-F<mJNARr(hARr(hARr(hX=Wf~X>)X8ZewL2D<Cl-JUt+CWo=<;ZfS03Utwc(X>V?GItm~lARr(hARr(hARr(hARuF5Ze(d>VRU74E@5zRWo~3BC}e4KbYX5|Wh@|JV{~tFDJcpdARr(hARr)VW*}~FbRc74Ze(d>VRU74Itm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)QX>)X8ZewLEAYo&4Z*m|#AZ=-GC}Ux6WNBn!bY*iX3LqdLARr(hAR{0|Zy;`ObRctOX?kTKVIX&Ja%*LBAZ1~4Y<W8%b97;HbRchTY<VDIbRcwSWgu)}b95kcVP<6@ZEtO5ZgegRARr(hARr(hX=Wf~X>)X8ZewL2D<Cl-JRovqZDDC{X>Mm<VPkY@Z*Frs3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARu95bZK^FTVZ2#Z*pBAJs@mpb95+lVRC0>bSVlTARr(hARr)PY+-3_Wn?a4WMn9GVRC0>bSVlTARr(hARr)ba%5$4b6a6!bZ>H9AUz;kB1T0;L?T@vX=Wf~X>)X8ZewL2Jv|^WAZ2WGWguU6F*jdrZ+2y0bZ>WIa%3oQZ*ysMX>V?GTVZ2#Z*pBMAar4JXJvFL3LqdLAYo&4X>V>@B4%N7ZDn#IT_8OmZ*pX1a&ucST_9;@Aa8PHWpZ;MWo&b0AX_3(K~qyAT?!x|ARu95bZKvHTOw#-Ze(*JT_8OmZ*pX1a&ucTI$a7NARr)eWps6NZXjV}bZKvH3JMBjWo95>b}=_!a&K>KUuAY-V<>N8b1WcXV{~b6ZY&^kbY*ZUItm~lARuXGAZ~ATAYWE7H(yduPfcG(RY6ijAa8OYb97~JAUq&8G&nj6ARr(hARr(ha%FUNa&91DV{~b6ZVDhEARuUOb#fp*AZc!NC|_q~bSQ6Pb1WbtXm53LA}k;<DIjlhATTKkARr(hWMO$AJs@drbSPhEWppTSVsk7YB4lBCA}k<tbY*ZLFE1c6G$|l&av(4%3LqdLAaiA5bRaz-Uvp()bSQ6Pb14cSARr)fbYXO5AUz;oRxvkUQcq7!Uqx0yLtj%=K~zOsb7f(4T?!x|ARuXGAaitOa3DQBATS_rav*bbWpE%oAZc!ND06gSbY(7QWppSaY+-YBUvqS2a3U-qEiox6Aa8OYWMO$AAw3{zZgePfbYXO5E@x$QC?aHGc_J(zEiox6Itm~lARr(hARu#eVRU66Js^7`Y+-YBUvqS2a3VS&b97~JEFdCeVR<4tAY@^AEFdCbV{~bDWg<EtPH%2yeF`8TARr(hARu2>F*jdQPftx>MOHyWUsF^;R7G2JWnpw(AUz;+bYXO53LqdLAaitKbY)v2Y+-YBUvqS2a3WnGJs@**WpD~0ARr)VW*}&9b#fp)ATlvJ3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)NV{~b6ZXi7%Utw%%XKr6;VQyq|C}CrCX>V>UAa7!GDGDGUARuO8a%~_zAYW!-a&0JYVsk7Yb7f(4DGDGUARusZX?9_BWgtBuUuR`>C~snOEFdCqa%py9bY&teAbWi&Aa8OYdwmKZARr)cZ*ysMX>V?GAUz;kUuR`>C}v@DZ7d)nW?^z|WpW}cAX_vnAT(VmEFdauX>)WaUuR`>C}v@DZ7d)nXkl(-b0RDtTU{w2Z*m}8T`64(ARr(hX>N99Zgg*QX=QUDJs?|TX=8LKc42IFWgu^IAbWi&AZBlJAa-GFb!8xFZXj%Fb95+QXJvFKaB^vOVRU6IAR=jQc4cmKZ*pm6b0RDtTU{w2Z*m}8T`64(ARr(hZ*pX1a&sU(AX{N$bZKvHE@x$QC?aNIa&2XDA}k<VB2Yn7QzBg{EFdauX>)WaVPkY@Z*DGUWppSaXkl(-b0R4qZ*m}8T`64(ARr(hb7*B`AUz;tX=8LKUuR`>C~$IVc42g7EFdCtXk}y~EFgP*DIjlhAbWi&3LqdLAarkZVQe5hAaiwXC~aYQC@?G_X>N2Vc42IFWgu^IATTK@AZBlJAa-GFb!8xFZXk1LWn?aPVQh6}b0{e(AS)nqb!{jLARr(hARr(hZDDvQFf1TxZgePiVQh6}Aa8OYFexb@W^ZyJX>N99Zgg*Qc_3+SAZc!PWo~qDa%p9AAZBlJAa-GFb!8xFZXjuHc4cmKZ*qAqc42IFWpgMg3LqdLASntUARr)NV`F7=b09q+Uvp?>WM5%pV`X!5C~ReJC|_q~bSP$Fa&0UiB6MkNWpg4dAX{B2Aa8OYTU{w2Z*m|pFewTkARr)VW*}&9b#fp*Js>hMAYpD~AaitKbY(7QWppSaVPkY@c4Z<dAZc?TPH%2yAYpD~AarkZVQe5iATcm7Itm~lARr(hARuF5Ze(d>VRU74AUz;kT?!x|ARr(hARuOMav))2bZ>GjASiHeb7^#GZ*D9gX>N99Zgg*Qc_|=iZXjiDb!}yGVRU6EdTDSdaBp*IbZKvHb1WcfZgypEbZ>HLWpgPhItm~lARr(hARr(hARu#LY-M3$Y-J!lAaiwXC~aYQC@?G_X>N2VX>N99Zgg*Qc`j#VbSP<bWo;}VFexB!av(4%DIjKVav*7RWo;m7ZXjP%MNCXVLQF*|3LqdLARr(hARr(hAZcbGb75>{VPb4$AUr)FFd%PoAYo&4Z*m|$Js@mlZYXbZWMy)5DIjlhASiEgWMy)5TVZ2#Z*pBAVQyp~Z*pX1a&uc@V{~tFU0X0+AR#><B2Yn7Qz9uk3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARu&Ma%W|9AUz;$X>KTCV`F7=b1WchWqCbpVQpe$VIXjCX>N2nAYo#2C~0nVC~$9cX>@6CZd)*2DIhH%aBpdDbXzc8DIhB#VPbPAX>N2VaBp*IbZKvHTQOZJAT1zpZ)t9HTQOZJDGDGUARr(hARr(hARuIEb97;DV`U&cAYo#2C~0nVC~$9cX>@6CZd)*2DIhH%bYXI5WprCGT`3?dAYo#2C~0nVC~$9cX>@6CZd);3DIhH%bYXI5WprCHT`3A6ARr(hARr(hARr)VW*}r~b97;DV`U&bJs>hV3LqdLARr(hARr(hARr(hAY)-}WNBn!bY*icVQ_F|Ze%DZWNCABVQyn(EFdj&VQgh#Vr*qBAYo&4Z*nXkbYXI5WppVi3LqdLARr(hAZcbGV_|M&X=Gt^Wpg?TARr(hARr(hARr(hUo0SBEFfWHbZ>GjAar4JXJvFCJs@pqZYX16Ze(d>VRU74DGDGUARr(hARr(hARu#eVRU6%B4J~6X?A5IT_8Omdm>?DbZ>GZIv`<VbZ>GjAR=^Oa%W|9B03;!X>)WabYXI5WppWh3LqdLAYo&4X?A5GJs@**VRU6KXJvFKB4J~6X?A5IDGDGUARuXGAYo&4X?A5GX>%Y>Z*FBe3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)NV{~tFAUz;yZgePNV{~bDWm_U)V{~tFB3&s8ARr(hbYXI5Wpp4tAar$bY-K26V{~bDWm_V2VRC0>bRu0T3LqdLAZcbGVPkY}av(lEAZ%rBC~$9cX>@6CZgVLhZ*m}EV{~tFAU-`HY-MgJX>N99Zgg*QX=QUMItm~lARr(hARu#eVRU6%B4J~6X?A5IT_8OmPH%2y3LqdLARr(hAaZ4Nb#iVXVPkY@Z*B@8ARr)VW*~HRaBO8LaBp*IbZKvHb6a6!bZ>H9DIg&|Aar4JXJvFc3LqdLARr(hAa8PHWpZ;{VPkY}a$O)jAYXPdH(zaUc4c35Z+BsGWGHZNb7^#GZ*Fs2VPkY}a$PJSbYXI5WppVDARr(hWo&6?AZTxOav(iDATl#L3LqdLARr(hAa8PHWpZ;{VPkY}a$O)jAX_3tQcqAKT?!x|ARr(hARujFa%*LDAUz;kY-w|JC~tCPWpXJXW^ZyJZ*pX1av*7LAShvDbZKvHE@x$QC?aiPa%*LDA}Jtmav)n>DP0O6ARr(hARr)RcxiKVX>Mm<b75>{b09q+dwmKZARr(hARr)SZ*m}Sa%5$4AZczOZDDe2Wpp|UARr(hARr(hARr(hX=Wg7Wo{^Ma%5$4DIh*QATuCgZe$>Da%5$4TQFT9Jv|^IQ$<WnB035nARr(hARr(hARr(hARr)RcxiKVX>Mm<b75>{b6an6WMy(&F<o6CJs@RxX>)XGZf9R}VQgh{E@x$QC~tCPWpZ0FT`V9lDIhB#ZDDvQFf1TxZgePba%5$4TQXfBZ*m|oDJcpdARr(hARr)aWo2Y#WFS2tZDDvQFf1T+Z**a7AT1y<Ffb_!ARr(hARr(haB^vHa%psVAUz-`B3DmOOd>2GB27t5OCl^FB1K0>A}k;xO+`#kP9iKIB2!dSL03XWQc_tWEFdCOPfbBoPa-TJB11t^QcqMOEFdCAMN(8rOi5ZrQX(uMB3DR7K~y3s3LqdLARr(hAZc!PWo~qDa(N&<AZc!PWo~qDa%p9ATVZ2#Z*pA<ARr(hARr(hW^ZyJX>?_6AZczOaB^vHa%psVItm~lARr(hARr(hARu9OVQFk(Vr*p~Js@picqlL|AZc!NC~0nXWo~qDa(OOiWppTMbY*QUATTK)Z*m|oDIhH%Wq4_GbZKs9Uvpt>WpgfPWppTMbY*QUATTK@3LqdLARr(hARr(hAaQkJZggpMc_2L?ZE0>OZe?X;Wn?TMVRm6@Y++(-Whn|EARr(hARr(hARr)VW*~8OVQzG3ba@~=Js>bT3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuLUX>)XGZf77pAZ}%NbSMfSARr(hARr(hARr(hARr(pZ*pX1av)}Jav*PVWMy(7X>K5GVRCC_bRcPFAZ%rBC~tCPWpXJXK0P2aAYpD~Aa8PHWpZ0ET_8O@AR<#mOiUsmVQyp~Z*pX1a$7N7AU!=GX>?_6DJ%*gARr(hARr(hARr(hARr)4Z*FBQ3LqdLARr(hARr(hASntUARr(hARr(hARr)VW*}vFX>)XGZf78Ab0BVSbRbS|Ze=<OARr(hARr(hARr(hARr(hWq4_GbZKs9TQXfBJs@drbSPzbX>)XGZf9FET_A6AATTK)D<E-oVQzG3ba@IOARr(hARr(hARr)RY-wg7Y-MgJZDDe2WppVZJRmVJItm~lARr(hARr(hARr(hARujFa%*LDE@5zRWo~3BTOw0MOiUsyAZc`EZ7d*hbzyFFX>@sADGDGUARr(hARr(hARuLIb7eXTARr(hARr(hARr(hARr(hV{dMBX>N683LqdLARr(hARr(hAZ}%4WMyO^Ej=J{bzyFFX>@rCARr(hARr(hARr(hX=Wg9Wo2Y#WFR~}ATT-#ARr(hARr(hARr(hARr(hVsd3+YYHGBARr(hARu95bZKvHTOw^?a%*LDB3&RoAZ=lCYh`p>Ix#R^3LqdLAYo&4X>V>@B4%N7ZDn#IT_8OmZ*pX1a&ucST_9;@Aa8PHWpZ;MWo&b0AX_3(K~qyAT?!x|ARu95bZKvHTOw#-Ze(*JT_8OmZ*pX1a&ucTI$a7NARr)eWps6NZXjV}bZKvH3JMBjWo95>b}=_!a&K>KUuSh;a%3oPVsk7YVPkY@Z*D9gb97~JDLM)uARr)VW*}~FbRb_=F*jdQPftx>M^!;mL?CZ+AaitOa3CchGBh9|Js>hOItm~lARr(hARuyObairWAYo&4X>V={ARr(hVPkY@Z*CwxAYWr|aCu*0V{~b6ZYW`6bZKvHDGDGUARusZX?9_BWgtBuUuR`>C~snOEFdCqa%py9bY&teAbWi&Aa8OYdwmKZARr)fXk}y|Js^8)WqCRvZDDvQFf1TxZgePiVQh6}Aa8OYFexb@W^ZyJYh`&XAa-GFb!8xFZXjf7V{|BAXJvFKaB^vOVRU6IAR=>UWn>~OAbWi&Aa8OYdwnS`X>?_6b0{f&3LqdLAZc!PWo~qDa%p9AAUz;kWNBk`D0X3Nb!8xLav*zsDIjKVav*kLY;|QIX>K5FX>)WaUuR`>C~$IVc42g7EFdCjZgypEbZ>HLWpg4dAX{B2Aa8OYTU{w#3LqdLAY);2a%p8`AUz;+b!{kZVR$GoEFfuabSQRVY;|QIZ*m|oDJdXkZ*m}MZgypEbZ>HbAZczOX>N99Zgg*QX=QUDW^ZyJc42IFWguy8AZc!PWo~qDa(OOxVQh6}b0{e(3LqdLAZB55Z6G}$UuI!)Z76SIb1WcVb7f(4C~snODJcpdARr)cZ*ysMX>V?GAUz;kUuR`>C}v@DZ7d)nW?^z|WpW}cAX_vnAT(VmEFdauX>)WaUuR`>C}v@DZ7d)nXkl(-b0RDtTU{w2Z*m}8T`64(ARr(hZ*pX1a&sU(AX{N$bZKvHE@x$QC?aNIa&2XDA}k<VB2Yn7QzBg{EFdauX>)WaVPkY@Z*DGUWppSaXkl(-b0R4qZ*m}8T`64(ARr(haB^>Cbz@~@AUz;sZ*FsSZDnL2Js>a&ARr(hW^ZyJVPkY}ax5Tka%5$4AZczOWo~tCWpZJ3WhifQWMy)5DLM)uARr(hARr)VW*}i>bZ>GXK0P38Wo{^NZ*ysMX>V?GDIjlhAZ~ATAZc@HZgX^DZewLAZ*pX1ax5ThX>)WbAa8OYZf|rTZ*pX1aykkiARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hbZKm5AUz;obZKm5Utx48W?^z|EFf@ib7^#GZ*Fs2VPkY}a$PA3ARr(hARr(hX=WgAa%5$4TQFT9Jv|^INI_CoMN?EFAYpD~AZc@HZgX^DZewLAbZKm5EFffQV{|Dx3LqdLARr(hARr(hAaHVTWOZX@WFRX&AZ=lIC@?G_X>N2VbZKm5E@x$QC?a`jWo%?$b#7^Nb0RDtFexB!av(4%DGDGUARr(hARuLIX=WgAa%5$4TQFT9Jv|^ILr+XhMMG3yMnzIoNlZyvMN%RlVQyp~X>)0Ab97;DV`V6GX>4UIAY^G{bSWTVZe$>IX>4UKXJvFKB4%ZBbZKm9dS!B7VRm6@Y++(-Wg;vfMqzAoWhpueARr(hARr(hARr(haB^>Cbz@~@AS*o}F$y3cARr(hARuLIX=WgAa%5$4TQFT9X>K4WB1T0;L?SF8B1T10R7p%pT16r$Itm~lARr(hARr(hARuFJZgX{QWn>^LJs>d(ARr(hARr(hWo&6?Aa8PHWpZ0ET_8O@AR<sqK|@6%AYpD~AZ%rBC~tCPWpXJXK0P2ZAYpD~Aa8PHWpZ0FT_9<0ASfb7Pft@tA}k;xLr+&CEFdCNNJT|ZA}KlwARr(hARr(hARr(hV{dMAb!}y2AS*o}F$y3cARujFa%*LDAUz;kY-w|JC~tCPWpXJXW^ZyJZ*pX1av*7LAShvDbZKvHE@x$QC?aiPa%*LDA}Jtmav)n>DP0O6ARr)cY+-J0Wn^D-Wo&G7AUz;^eF`8TARusTVQy|^WM5)+d2=8=ATSCbARr)SZ*m}Sa%5$4AZczOZDDe2Wpp|UARr(hARr(hX=Wg7Wo{^Ma%5$4DIh!`Gdc<&ARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hadlyCbZK;XAUz;$VR$GoEFfuabSQ6fWMy(&GF>2Vav(4%DGDGUARr(hARuXGAa8PHWpZ0ET_8O@AR<#mOiUs=3LqdLARr(hARr(hAaHD9Zf<2{Uvp(_Y;#+0a%5$4TQOZ-AUz;(Y+-J0Wn^D-Wo&G7E@x$QC~tCPWpZ0FT`V9lDIhB#adlyCbZK;X3LqdLARr(hAZ2W6W*~2JWMy(&FkK*NZXhTkLRDE`P*P7sRYO!FEFdC6Rasv_PDxEcOd=^d3LqdLARr(hARr(hAaHD9Zf<2{Ut)E6b08}{AaQkJZggpMc?uvPARu95bai2DUuAe{b98BLXJ2z=Y;1ENJs@*+Z76MNZYXnTWn?aAWppTMbY*QUATTK`AaQkJZggpMc_|=fZ*m}MbY*QUAaQkJZggpMc_3+SAaHD9Zf<2{Uvp(_Y;!JYbY*RGC@ColARr(hZe?X;Wn>^dAZ=lIC<-7TARr(hARsU-3LqdLARr(hAaiwXD066KWG;4LY;|RGC@Co*D<ETGa&l>9WFRXbaB^>Cbz@~@AT1zcZ*FsSZDnL2D<E)eVQy|^WM5)+d2=8wAYo&4bzy8@Wq4_GbZKs9Uvp(_Y;zzjATcm7ED9hXARs9UARr(hX=Wg9Wo2Y#WFR~}ATT-#ARr(hARr(ha%FUNa&91DV{~b6ZVDhEARr?kMrm$ob7*B`AYpE4ZDDL6aB^>Cbz^jMAYpD~Aah}CWi4-RY<VDKa&K^RAYpQ4Aah}6Wpi{ObZ;PRX>oOFWMOn=E(#zZARusZX>W3Aba@~>ASfbNPftuDEFdCHNlZ&3EFdC9M@J$oAR<jgOixZCEFdCNR8m1#LPb(iSt2YTB2-UJK~zs7EFdC7K~hprR3a=OB1T10R7p%pT18SKEFdCRNJT+ZA}I<WARr)SZ*m}MbY*QIX>K5Ja%pdJX>@ry3LqdLARr(hAYp8BWnpA_AUz;(Y+-J0Wn^D-Wo&G7E@x$QC~0(MZ7d)#DGDGUARr(hARu9OVQFk(Vr*p~Js@picqlL|AaiJCWG-iAbSP<bWo;}VFexA{AYp8BWnpA_DGDGUARr(hARuvdVQzG3ba@~>AZ=-GC~jqCWMyP5AYpc4X>4I)Y-K45ARr(hARr(hX=WgCbzyFFX>@rYJUt*VItm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)RcxiKVX>MmAJs@sncyuTVARr(hARr(hARr(hC~tCPWpW^9Z*m}Sa%5$4AZczOZDDe2Wpp5EW*}^3ZYXbZWMy(GAU-`HGazAZWFT*HWMy(&FkK)$Js=`eMNCX0AYpD~Aa8PHWpZ0FT_8O@AZc`EZ7D1YARr(hARr(hARr(hPH%2yED9hXARr(hARs9UARr(hARr(hX=Wg0cxiKVX>MmAX>%ZMZ*(9|Z*FBe3LqdLARr(hARr(hAZ2)Ib98BLXInB|AUz;yZgePRcxiKVX>Mm*GF>2Vav(4%AS)nobzyFFX>@rCARr(hARr(hWo&6?AZ%rBC~aYKYh`pPAUq&3Fggk#ARr(hARr(hARr)ZVRCC_bS`0VaAj^}C|e>^MNCX0EFfugWo;}VadlyCbZK;XT`3A6ARr(hARr)RY;$Eg3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARusTVQy|^WM6Y-Y;1E|X>?_6T_8OmVQg|`VPttAD<E-oVQzG3ba@IOARr(hARr)aWo2Y#WFRd)AaQkJZggpMc?uvPARr(hARuXGAZ}%4WMyO^JUt*VItm~lARr(hARr(hARuCLWnpUyARr(hVPkY@Z*E&6ZDDe2WppB4AUz;$VRCC_bXz(xFkK2DARr)eWps6NZXjV}bZKvH3JMBjWo95>b7f^@Ut)D+XJvF>XLVt6WGHW9b1WcXV{~b6ZYeqnARr(hA|fJGa%pWKZ*FXPAW%#}PE;Uoc4cxcb9G{KV{&P5bZKvHAarP9bRc(cb!=oHVRUb8X=7n*Y<VDKVQyn(Y#?-KWgug6Z*VRmA|eVPARr)NV{~b6ZXi7%Utw%%XKr6;VQyq|C}CrCX>V>UAa7!GDGCZ8ARr)ca%py9bY&ntAYW%?bSQ6Pb1WbtaB^vOVRU68EFgP*DIjlhAbWiZARr(hb7f^@b09q+WNBk`C|_q~bSQ9gX?9_BWh@{fb7f^@b0RDtdwnS&Z*m}eeJKhcARr)SVRCICJs@9ZVRCIKZ(?&SAYXH3VRR^OVsj}e3JM?~ARusWb7^#GZ*FrSJs?{OARr(hARr(hUuR`>C}v@DZ7d)nW?^z|WpW}cAX_vnAT(VmED9hXARr(hARsDiX>)WaUuR`>C}v@DZ7d)nXkl(-b0RDtTU{w2Z*m}8T`4RIARr(hT?!x|ARu*aX>?y<V{~b6ZgU_#AX^F`ARr(hARr)NV{~b6ZZ2nKbSNTbVRCI{aw04sTOv?FQ&S>cDJ%*gARr(hARr(rY-w|JC}CrCX>V>WXJvFKB4}Z5WOE`cAX{B2Aa8OYTU{wE3LqdLAYBRyARr(ha%FLKWpi|MAUz;^eF`8TARuOMav))2bZ>GjAa8PHWpW^CZXjiDb!}yGVRU6Eb#7^NUtwc(X>V?GDLM)uARr(hARr)VW*{gEARr(hARr(hARr(hX>)0Ab97;DV`V6Ba%5$4EFf%Yb95;RARr(hARr(hARr(hVQyp~Y-MgJZ*pX1aw#A_Js>g)ARr(hARr(hARr(hVQyp~Z*pX1a$7K6AU!=GB2Y|0PE;ZaARr(hARr(hDLM)uARr(hARr(hARr)eWpQ<7b98eqb7gd7WoBV@Y;-7dbaE(fa%5$4TQOZJEFfE5DK24faAj^}C}CrCZ*nOL3LqdLAZBlJAY*cGa4aBUV{~tFb0BGMAaZ4Kb!BsOb1rFgWo>gPDLM)uARr(hARr)Ob!2B{bRaz-ZDDvQFf1TxZgePfWo2Y@E@x$QC}VPOa4aA&DIjlhATTK@3JM?~ARr(hARuXGAZ%rBC}CrCZ*p@fAUr)FVs&I^Wpp|UARr(hARr(hARr(hV{dMBX>N683JM?~ARr(hARr?kP;zBvWpW^LWpQ<7b98ecbZB98AYpQ4Aa-GFX=EU8ZXk4MWgu{MVr*$+AaitKbY&oNX=iA3AZ~AWE(#zZARr(hARr?kRA^-&a%FRLAZul1a3FJZVPb4$AYo&4Z*m}Sa%5$4AYpSLVIX8>bY*gFX>MtAbZKKCW?^h>Vqs%zE(#zZARr(hARuyKZfj*^AUz;+Z*p{HWGD(CARr(hARr(hARr)NV{~tFb1VuVARr(hARr(hARr)XWqCbpVQpe$VIW~+bZ>GxASenTARr(hARr(hARr(hARr(x3LqdLARr(hARr(hARr(hAZcbGC<-7TARr(hARr(hARr(hARr(hARu95bZ>GXJRodkZYXeXb7^#GZ*Frb3LqdLARr(hARr(hARr(hARr(hAYpD~AYXK8Y-L|zbSP$Fa&0UiaBp*IbZKvHb6a6!bZ>H9DIjTcAWm;?WeOl5ARr(hARr(hARr(hARs9UARr(hARr(hARr(hARr(hWo&b0ATcZoARr(hARr(hARr(hARr(hVPkY}ax4lUARr(hARr(hARr(qED9hXARr(hARs9UARr(hARr(hYh`6{AUz;+WppTVVQy<>WLr97b!2B{bX_S53LqdLARr(hAZBlJAYo&4Z*m}MZXjV}bZ>HVItm~lARr(hARr(hARuXGAYo&4Z*m}RZ*(AOZXj!AWpFwQARr(hARr(hARr(hARr(hb#7^NUtwc(X>V?GTVZ2#Z*pBAJs?{mP(f2uB3%j!ARr(hVPkY@Z*E&6W?^z|WpW~2AUz;;ZfSI1VPkY@Z*Fs2FkK*NW*~KLX>?y<V{~b6ZgU`IY;$EGTOv?FQ&S>c3LqdLAYo&4X>V>@B4}Z5WOE{2AUz;;ZfSI1VPkY@Z*Fs2F*;oeARr(ha%FUNa&91DV{~b6ZVCztWMyU`VP|D-bSQ6Pb16CsARr(hbaHt*3LqdLARr(hAaitOa3DP(ZE0>OZDDvQFf1TxZgePLXJvFKZ(?&SAR=>gWpE-aATTK)Z*m|oDJd)<Y-MgJUqM4uNl#8wDIhH%F)0clARr(hARr)NV{~b6ZXi7%Uw36?WM6V+aA9e3Utwc(X>V>QZ(?&SAYWr|aCu*0V{~b6ZYW<tLsUsmPE%WRbY*Z|DJ&p!bY*ZU3LqdLARr(hAYo&4X>V>IJs@9pF*jdkWo2YvXLVt6WGHW9b1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9pF*jdwZ*Og1Wp-g>C~snOEFfWHbZKvHEFg1qWpF79ARr(hARr(hVPkY@Z*CwxAYXE2aAA30b7*O1bSQ6Pb1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9lVQy<*b7gF7Uvq44baN<gVsk7YVPkY@Z*D9gPH%2yDGDGUARr(hARu95bZKvHAUz;oaB^j3ZE$p7b7*O1bSQ6Pb1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9pF*jdwHD6<Ib#8QJawu<Nb1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9pF*jdrWM5-%b#8QJawu<Nb1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9pF*jdwZ*Og1XLVt6WGHW9b1WcXV{~b6ZY&^kbY*ZU3LqdLARr(hAYo&4X>V>IJs@9mWo2YvVs&I^WprO>bzyR3C~snOEFfWHbZKvHDGDGUARr(hARu95bZKvHAUz;obY*gFX>MU`Uu<b{b!lW_bZKvHC~snOEFfWHbZKvHEFg1qWpF79ARr(hARr(ha%FUNa&91BVQgt<ZeM6&Ze(*PVPkY@Z*D9gZ(?&P3LqdLAZ2)CWpH#LMR;RnaCB*JZaNAeARr(hARr)SVRCICJs@9ZVRCIKZ(?&SAYXH3VRR^OVsj}e3LqdLARr(hAaZ4Nb#iVXdkP>RARr(hARr(hARr=UVRCI{aw0k)TOv?FQ&S>cED9hXARr(hARr(hARr=WVQyq|B03;jTOv?FQ&S>cAZBlJAYUM9ZXhUMXJvFKW?^z|EFdCiVQyq|A}k<VT`3@Mav)n>DP1fIARr(hARr(hARr(hB5h%EYh`pIIv`tJED9hXARr(hARv7T3S?zwAYW@?XJ>3>UvqV0ZE16JX>V>{Wo~qGd2nxOZgePbVsj}v3LqdLAaZ4Nb#iVXVP|D-bSQ6Pb14c63L_v*VRCC_bS-0VZe(e6X>V?2WFTdDaAk6IAaiAGWn*-2axMxZAWTnJIv_%CVRUR|WgtRKEmko%EmAQpQbRH<AaHVJb7gXNWn>^>VsCb3AYyrRWi4iJaxG$cbY(6IBOpjgM@Tv#OJR3mb7*O2X&`WQVr*$+AZ2iAb8lp2AUQHMGBGnVHY^}>Wnpw6FfIxsAVP0+XdrTLb#!GQb98cLVQq6DVRB_4X=G(?bZKK@Y#?-Ka&L8KXdrTRZggpFWgv5OWpE%dHa9K`Uqv!EIbTdqS6@LxR7p=xQy@JcUqM4uNl#8w3SUJsH#uKONk>RuK|@qYPfk-HJs@gxZ*DGZZ((F}C<-7TARu~dX<{y9Wn*t`aB^jHb0}hAb7eL(E@C(}WMyM-WMwEPV=Z!PJXJDkU}<4{O>a;tGC6QsLqJA0ZEibKQe`+bc_Tn0L0LjFGDb{(XlFB8M|E0WIAdi=Nj+e7En|0iCMhFzQCLHFX>2q;W?Djccz$v}N@hYrAu?t{S9C*qc}aO-Y*}h!JUnVLaZXx$M`t@ba%pf%CUzw~SYTgqBz1gDb2VySeQ!)tCVEDBSbR)*Zg^jPXi#QgR99kjd0BouPjoqaC`(>RcQAN+V0LILDn(;Aa6@)dKuJR*R7Gh%c~ECRZY?(<AzxoAF*|#4aCC4jQbBGZBtJ7LNIrUBHePo>LuhbCUr%OsVMA$2ZC+3{Y;AgHSXpX(OK(bWbwEQ>czaPYdsKU9N^U56S93UYEk-ItKPW*;Eki#(Kw?Q>UszvDUsPv&OG`CuO>tjHQaK}fctbl#Y*k58J5Ot4Vo66pY(qF^S~gBODnLkNS$AYNBt17aDo#&(F?4A~Xl{3KD{f$GUU_CQWOGz`Y<5dgUNkp5V{TM^U`1*vC^u{^Sv+!2dP7ewS3p&FN;^nkZgyZmFj_b&a$ia`Vl{eKHf&ULKQ%l?HdjS*JWM7uIYKo=U~_O<J0({+L`H5zIcPmdUtuvxL48PLNj_tFeKl`<Zf|WyVR$VidQ^BgV0mCvJatEMO>rhfUSeooPbo}#G;v@|Vn|OWR8LSVL26AcBQtJkDnC4NbVP7%cv($DOJh8FeONzKG+$71Wl48>XLUeLPH0wFKT=p#YHo9PGCwIwP(x~aFhwCmKU8~nK}|zOZ)0aoG&DIXK`nGVK`nVaUn*cFXnrL)Wo2b!bxUh#c4B2JH)tVVb0#BIHbx^zUQlZ#WNRrlZclJQL@i@`c5^;zSaoqrZfhx4O(r8sVRk`zNkDvAes^DVIZbP2Kw?=VMj=yecz8=~LU|!_NhWYbXh0?{a&|CdZ6$qiDkwrwMtfFlFm_5xLViOvP;o0XKuSJkWlu>}Jt%83RdFMHc|37@aXmdnK2kt5Q+j%MUp91oW-(4?H%59YU~6GwYjAF7JW4k<W=J$cF)?l=Eix#2WmiWnX-g_2c5o$2D>86zNku_gG<YF<Mn+aaYEp1USxY%bGe2=xBP2F!NF+aGNh3m5R8KKCU`1wjIcRkza4UOPU}SDrNMm(IG$DCCLODV(OiM6OYbttHIa+>bX-PpVeSK3)Nk2&{b6G7zPg*H^YdlLeXiH=~VLLE0P$fA~S9E(;Z)tQqS3^x`V@_^NGB_xFJzq0zc}7uKKS@zPHDpqFBqd)pGh{1RB}85}c~o^jOhhtdXg6bOUR5(yRwGp<QB6T)WlJVzXfY{MIW~GtZ);OtMRrYTLr{HFS4T`ZRbp#cY;Ss6G;&Klb~aaVNn&7AKzVITXenxYdv!8POF?#TEqg^aDlkH2O(`}<dO#{NbW<}+CR9>vLozEiZedzsX*44yR4_GTC{Z_XL~JQ#Lqu08S2IL4L?tw2D@8G7M0s9fb#XgHW>9xySy)PJb3sZyIb=IMdr5J1HcfS7Hhn2*C{i<DCPO@DH+*(Ld2?89Ry<#IJ7p<8M=EzeaCK)+eLy@dN^WCGM0G1xJ$gthWno`Od{IqNF(qbvWJ5ksc}#pXbvb26K6xu7HYqb`G*e1+eMly0Fk(MbZ*xg0RA5g&U~wu!Pd#&FNLM*2MOsF6EhA|_K1*?Ab~kNmOEp+_LQ#8SVI^XGVPsWRep5F#L`^kRJ}WdkZ$fA#VlpL4bXI6{AtZJ|IWaLfbz*EeL_=RtbR{ZUD0Nb7Q8Z3hR5nyUIc<JeJ5zi=O=%=|QDSg&O*J=BVtai*MPzC+K1*0lG)inkdVN1mRckm>WO-90RyHzkdR0LwPHT5lb!Q|*Y-f32Ku~B!UvyYMN?9vRT6T0XUt};@X)#JsUQI<vOeQ3CR#`AmKsjkNVMItoes)njHBcrpLVIi~c2Z3@MovjhWLI@2C2U_jcUovrW+Ze$Wk-2^W_WsOEkq@8JUM(!Dkw8+R$f0%WH5RwczSSbcr|V`W++!9Yba_nP%uh%Z((yPJ$6YZZ!K?gXfrBVVJbFEdrDYKYkh5fNi{Y^c`I5sH*{lZF?&dTWMCmnb3$HWbyYb#S1KuGWjQ}QU~4sNQ#D^ZQFlZpcQ-9VUu<q9LN{t!YCB;`P*pN`SZZQ&R$@(McwaRkZD2+=ReUH*Yfo21B`RehM|m(mbTdy$MoBP3eMEdwDK|%Pa7tQjJUKs4VqtGLbvZdva5gqdRcmEDcT{R-KQ<+JF->zwNm+GANo*=5F<Em`UvGLLNIxYrZ#{2GNN8X?dsQ<<NIh{hKUH5aFl{4KcYJquN;GX*R4_9)V<dhiHhpquWGOi-J#t?vWngVQMq@TKM`KrRO;#j3N;O!0MSN^EPGn6yEj3YCBqMraF;Ht$b24{zHBKaRXedBxKt6qDdon&NZe&SEdTu*TCU1N;a&jt2Lwad2At_ZrM?_vXQFu0ER3lR}Y(FzQKT$AhdR{V3LO4x3R8KoFW=ljeVL~HwG-PT;A!<uhH8NN|ayMj2N;p7XSu<fXdt*FkWi&M_Us_c(Us*smQ$0U*Ls>{wNg-K0SWiP&WNds!baq~CST#FGbY3t+Njz>XUw1NYAvJSKBPmcuPCQ~WP&r0Der-EWY%xS*QEM?ZY)vRtOkg~4DL!ChB|9`sQ(sYhI7mEDQ!QswS13MidN@>LOE73yK6X_%Y(qwMZ&WueQ&oCdV^mT}ZgOoyGh=yVHfU-gDLy@AICOL-a8PbRW?6c5C@3{}XC^a6Rc&QOJ56FoSTcB2Xg@SEd~I_}ZZ~%&ByBZUU{q6KbU|N5IZ!ZEGjvUMR8?zHL`Xp^SvV^*dTCQwH*jT6R!L1%c3@U|aW+F$M|3<eepO3sS3h)0XfrcRMLlRdQBZDQXnku^BRDH@b2c$SR6!<9J2+z@cX@d@J!nUGbto%CW<_CqDLq(KeO@GWZAwB{FfueoOh#`tGiWFwGe$EiHc>xTBs_8<LPS@3Vq#}ma8zJwS#n}(A$f3ZRx&qmBW!FrVKqiYUQS<CD=K|3F=jtaOd~2&QE_)dY&k|{NKGgtKVeyHAvQ2JAxU8)IWcZDBO@(-b6H?hIYBu<G-zu|OhRooWnN}`NGnBla3onrGB#OhFe55cWqNdFQ9&{_OFS@NOg&UtHbOLJB|J$+YgJ5eDmZ(4YASF(YF9E?G9fZMPgPJVS3fm)JwP)iNPc}wQB+TTcs^!3dr3A(YcWGHdox;NVqbf8WPUqTG(|B!JX14HOISlCWiVE0C3`|WYJPN2cqTtqIB8BzWHU}RBy=%SQ7UF~VofPSYFbb{K1xe{bVg}MQglj3F;RU=I9_F7XE1F{Vtz||Au}{3JxhIXPD*%pSwdBHQBq${DJe@%cV2ZgRx)-lAu2_EcXDlCK3O$-Rc%j8LS=VPK2v>nMLkzzKzcc5YC}CqL1ao|YAH^6a6dpqSwtvwQGGKvQdKomEp&HGbUk5aSV=;3K{ji2D`RX$N>6-6RAG8mPAFF<DK>g=c0naSHhC#bS1EZ&Z6kAPUrQlsGHGK*d?r#eB{EiMejzt%byHw^FhVVBP-H`BenfOWZgw^_Z)H_rbs>66N>wB~RbnYMM@MpGZDmq3P%U0^bbB~GL}g)nXfk(oU`0woO-OTAa87YSOEYO;QEYi5HaBEvPf>4EU{WP<bvr3IRV7ADJu`S~L~lWHd~14pdS4_|YcgR_U{-d0Wi5MXZ8Re?P9!T?PIM|fR3Ta>MpjN{dOR~=Sza_qLnVA!dpBrCMp<EdBt&XaL1johOG|u2HEw=CUpH7Oa9(0oY&JYLSUxd#Dm+3hUOZPSWI|4QH+3{dM14Ufdr)RsY(aWuRaizdMImxhBvChZQ&~S}X(VG~QCcWuCU`k5IC@D!JSkRaJVHK9Yb|tYDs3u8M_6Y$U_3@~GfpWfFkx(XGkJJ(XecvxZ&+<+aY8FYGc$2gV@gC#Sa>l^KTB?FDmW%_aBWLGH%NJ9Y%pwaOG<A=aea1tQbt8_KWRu$e0paoLqlUmZ9gd_NKYkjL10dEWKL2xCTCT6Oju@VV_8ypQ&%=QcSTNdSyOjUGG%0EH%w|uPAW!1A#rO}AxBSQHD!HES2l7#FgHg?W_>$OGG=pAC{is`F*I~Rdr&}OPFgf;A$3}NN>5F9MSD?BIBaBaV>DD{S3*l9bvIWvJViZJT5eflBSJxQB~e0iB~&*iNmfvJLoqlZeq(f3bv`LbV<9$SSWqfsNIohzNJ>&bURh@?Lm_fCT2Ep;Un+7YC3k#sN<d*hP)TWXF(ph(S$AeBH)&XKWI!P|a#~1oQZP<tM{0IaSAIQfaw988G*vkvVsB-0IAuvlIXNpkWm<V@Y+fZYS4dQ6d_gL4Xh3XpCMjQ6U|(5vPH<B{Ju!D_LpM7=SVDGoLP%ImVn!=oA$n6@dN4jFG+8?{b8c!;Xf$_9I8{(fG+<LueN1mua3*XjLqs+~NlkV_Ej~3wC4F;HBUn*!G9xK`L{dX^PCk2RSw=`HGcql6d~r`GUTY;{CO0%aCR8JFMLQ@-NknF0HEn7+C{l54H$rSZMp!XdQDS=~JathpMK>^NZ!%aiIbcgtM?fYjH+Mp1HBdn|DnM*vF->k-NI4-PO>=TuVMt*`d39nWZYfYnbV*+=Z(4b6duwB4P<wD`crbWaD@9UqLvu(&JS96)C3|l@Wo&JBQAaH@aaTPvdo@8RS3p8YaAY+zYjq}KGITO?Q8{o@Y<MYEQeq=&GhbRZd2vO0N^eYiZZk|#a3f@BD0(4EG;dN;OJzK3ZBR*cbYn9kCU!|ZN+UxgMrvU|W-u~5R(@kqL1$!qRU{;HbAD%7awJ+;aad$SRZdD(LUtuJAv|U%Mom>kP%>m{Z+2BmIax_;Xgx@1U^_HRS5h}5RU>doSSU?oM{8GYNKGMZYb!r=Gc;9vOkXfwAuw!kc~nw-aZx;0Ek{&*MsaO$J3uK$Zc0K>S4D0)b!#MIP$hFJYk5I?C^lj=Lt0pDGBs&@XemWWT3#_?c|uYoLS8C-L2W5YGD=5aXjN=PQ&lx4M{`eCc0X7+A#x~JOj=i9EkJ%#bwg1yG;?KCR%k10Z#OqUZ&xdGLQF|$QB7<>ZDmL;dt+21J70K3J3=9DU~o`ga$|8;KSy+XL}yY@dNFf3N<e5tRD5u0enf0oR4QV8U~pPdaz|!BMnOq<BS~>WVRTG!Dr$XcZa+3^DtT=|YI<u<b!uWXCPhgyKWHf>PE9L6AvJbTKYcttY9%9nOE*YobY(?hElxakNI-0Ecus9Sem`e*COC5?SZytKPB3pnSW`$dX>dGpL3C+rG(Ay1Jzz#)Q*R+iQciSJeq$+Gdt_Q~GgM=8Kqz)NU`#<rB_S<Ba4}U)P)=@DKQvxXdV4`bM<h{UO?qQSKr1{vD{DS>RZ%&9c1<ByK_NvYGdCkzSz0+kI4faZeK$oZbVOrpdq6cKacXc%JZ3E;U^7r(Ms+eUHEA+?M@vasb2%_tKVU*gdwX$GQ!Q_5URXY2I6rDqLUvk9JvdoYLMu-;Fj81WB{*h6a4~E@L40a$VtztYLtZ>FYHdDmOM7;HSSe|4Emd(}dsZ|&EmwCXHClK!a3&~cDrhNwK4WrrHDz^nZgh7vGH*X-W-&EIcRo~mLqH{2VR<|^GBZVSX=gHHOiVCTC}vb9Q8H~pLt1WOZh26BZ&OKgMp-B;UV2nxC`@=wGb&heb6{|IB~~bVOHp1!eR@$#GkG;`PeCYICRs&tIUy=Ua7bu2C_ZpbR8M_*HcLQQNliIXT5)w=Nnu_@COJ?uMrds&OF?dVCMYUietlVWd2vlURVXq}G*e(EFfCwVWp!0JNI@z?RWL$hXKzDbZEQUzGHP~eBxO`dM`a{XcX?VnCNWAuUpanFWJ@+ZOgU3gNhnE3elsR#S}8<iNNZC&Z%t|_epqQyWkh>CBuIB&bb2ajB_k^;Q8YbKEoVDvI8I)Eay2PLCPOlNPe5uROFm{MbR|N5UQ#t`M^7a-dPP)IK7Dj)WMLyUNOV+fVM=jrPiRVGerGTyH#90XQ8#HeIaotJS9f|iGFNa=etvRLOL9acHA7c=Ra!S!d1qc;VI@dxT1jP2BzQ4wJ2`iCb9*%<J48H2Zc1S>KQmG*Xnajha8P1QPj59yVt0EpO*UhBQ+p~gR!?d?JSt{ZVJc%(C?;8DSSwmKYc^|5RApCSYAZ!DelsgsKyz4LYiDgwQ%NOSKzToXY)exyd^9^ZKu|YtVoFXVF=AF(UukxAQ%Y!2d31J6Lt1WGc0DapMoL*QYkoH^V?I@DK36MQcxFN>F?md5NJ(chXj3q7MNDO1EjwUYZg^2yZh1gjC`)B%B|%6xJ#RTPb45a8b4h1LKvgL)cV9P6Wk6;{QFVAEQcqtYIBhLnK6OtcD@izBYd};+H9{mwNM~<ULqcgxWGPo}A#i;{V>VuHN^@yFHd<g!SZXwAenV(7ReDxtR98P@JWFCIVl!`HC}~epZ)#UCPb+V4dLv79J5gFpa6chwYh)o{T4Q2FW=3H@J$rH^Ktoz6bu}_oBS?67Vl7}%IW$O0DOE@{Kw4TtCVFl?d_72dUvwceGCN6cM?*qPZX;k`b}%(7Z*wwyet9KJG--8GeQ#MnMM_^JWJX?IZbfizZ#+tURarDyJ9&9AHCRnFXg+FDNLfH$BVtZKac^-uP)vScI9geJcxEv;P&{8*Ja%hYMQm$*V@gpgP)Tc0T5@1(b3<@cEopp2dNEIMVP-sIVkKEjXF+-<XJ{l<eP(fTF=t*dDM><ZRyRsLM0P+UD0D$bWl1tdMpz^*NKqwpG*~J)GFL%uZCPnGV>wA$eoIU!XDw|iNk~dPV0JfmK0qy2BtS!LL0@KYQ*u{gG$~6ZLUCYrc_dbBWIanuJbEotB{FnGVtq&<Mp|t)L`Nh*ZYCo}S9?ZUaDHQAGe&AxBX&uCOD%Y0S9K+0BT!ydJTY`sJV0%CLpeA+DQ81zNkM*keMC=iXiG|NY&b$ta#?pzMq^PwK4x!WRW(#pbVEBPS8_*JOkp)gUt?lCGEgIXKs+@=aZF$>bxk~CC_z$1cS%@&JVQZ9OJQ?HBR+9SD`#|gV0k-NJ~mQ*LRNNDX;o4yM^RLAcV9$DIbtDUQ9W=yKVxx0Xen7UR&sh?b#ZokX+JP~V|HG2PiaSGO-?;`c293nGd5*bDoJQrWg~DzT3BT*H%nD}c|lM{bXqGrWqm|yW^*z^B|JlBI73rzVSGI{Bwsv7G;?byB|9}HS~6}{MNu<lStdt8dsa$cP*hiTPf=@DCTw_SQY3XSX<m9#VM1*+WnghkLqcF;b$5L<c3v|{VKQS-A!tTrDn>?7S!FzEKPFK|d@DvVC3ZGNb!{nFdRJ#YGFd`XZc<hwB`R)eM<_%}K3OA6G-y#sJ2^8|dLcYbRzzfPD^PN8F=I?)aWq(FJ60%1GEzAuEpmNhHC1qMUsG2}BRpP1Ln>t<KR8lsBWWsEM<q~SR#I+MZD2)CGelQsQzck;H!Dv&MKo7dQhaT2B`Ya-VnAOub18m7M0{&sYixZ;SXOLEP)2@9J}@M6WmP<OGeJpad?R8eQ%GkyOCu<HXL)Q#bXGwzaAH|^T5ncxd`LoTS9xM=S!8Q<LuO(*GIC&TJVPo+YglG#cX4_-Hc>%id1o^yK|54uI3{gSJzptzL_jxjQFBvjR!2ZrUUWPrJu*giF+);ndtN4DH*GCxDq?F#HY9ydM^H>=W@}e-Vm@+KbUQL6DKdIXRY*Q9cv@ORHCbj=JA71rV{<8Ca3e}_S9CZjJxEe0KQld1Lp?@3S4=l0OkQSjT688cR&8r{QdKo;U~O%7cw=}zV`hCsGkZfpdUQ51cxQPqEptU#D?(x>DJo%ND1J&kW>+&vGG!!9S3`SwUQ%akUwTS0Qf5F-B|&{PC2J~HV`DaaQZ*@cb4);MYFa5pK}LO4P$g3-D0ysEQaeUFW^8pOeQIucFmPWuW^g-LL|9QyJ0&4;R6j~6ZzV8VOl@aGStN8-VJJs5MNM!~P(W=wNMlP$G(|jQGjd^2G$>VKM|x~>Ur|bEd1rQBa$0UUa&kjaN+fn=T2O3hRBA0#Q%Y!NPCz|DF)2GtNGc&kZar{wKznslc5-l7F=1<FVQh9obbD!bLrXPwJy}mjT7DyCb}&RwPbG6qY9lmYB_lB=HZf^qQ6n@+RaHQ4F)M8-VM`=SPgP7XYH@mJR6#;1J6A+RYDYeBXL>MTAvi>GRaQ81Y+qL>DSK{gGCN~9T5T;MXhU%|ZCE6AK6FGvK1ox3HAZ_mej!ADF(p`cOmt!-ds0$VGizB;S}|09UN$RXJaR#8G(l5WaVBIyDMU7YH6>zbLM3t|Jb5>LC1q${R5DO*d17}mK}K<BDl<<+B~3|1HAp;hX+|?lQf(nwN>glCXfSb5F)dSHFhnJ8N+~olODRb{WkF9eFjQ|pJU44`a$+WIF=S>_Q)_p5U~Dl&IZQufW;8`dRdPvqHY83pXEI4^W<^0ZRCiH5V177rJ9KJAK4Lh1DPm_;c5!e@AworDAy9H9LO3Q(DS3V)Z(mwWUpqZ{J8^GmNLFk>Bt3gtXEH=yWO74uIY2>UJ!C6BbTn2{M{Z1YL24*5DMmDOGe}8zPIOpnWm!CGHZ>tdWiezUKw(QuZzg70P-Qn|b1_kBG(c)TV_rEhO-E=wJ7g(qL1At-J5W7*Gk8E`PJ1S3Lq1?VW^-00eQ7FcZdGSzM@4T%U_^RJLSk56H+^#^QcZm*N;^w4Z!Kh3J!DfmG(vk?LTq3$OFeF0a!7I^R7_D|L3l!7Jxw%eX=NrMRcC!uGdyieOi6WNKSeEcKYT_xR&Z){V`O7!VPj7ta#M0YJ3DeuQgeD@d2n-MICfq<aUp3?a(O{MY*Al7cwcsGIA%{%LQ-f^Z&y(%W=3=+Y)D5qBT*zPEiqDRYG^A{UTAMxS7tL>Z$)EAC2(gcGG!qpbwYVXcp)-UWO!I%WP4?3R3&9sXhlanaB4;~ElY4pcXCBUN@ziANm*7#Dn&ObW=&UKPG~@BPj^OiK`D4*N_9$SL_2pQZFw?JHfMZ9dn+?bG+;|%YGEWKNii{faZDq5YIkm8Ej@87W>i>vaBOI3X;>?0eOfkhS$J0}C~{{iV0uPHXH;ToIaxSFF(g`9GAcn)UuH^aBtTG7X+Lr#U_Vt*W>s-xM0YVmRC6?ZJy1?YB{)Q0X;msmS1?0wY#}f-NGdI2b3sNhPhWCFFl9nLbwDIVePmftUN>(cGiE_TYARPkZccV7W<yVRK66KPWGZz;YB+vLZANE2XjpM>P<m#3cW^i+D{gOaAtY~jU_&=>N^B*1czSGZO-Cz6IVfmwHa9mUUMoLrC1^x-dU91jKTbj^C~#SKLn&u4XJl+}Z#!3PEj>A6PkK*KH$-_-L{(2iQc6xnZ+uT;BTOS>G%-<aBSUvfQZ_MbSt%tuX-#Z4VogtZa8@&AIYBv0ZX;4<eko=&S6EnMJw#_!IVfmxX*+viF(GA1XL5QiYj|}uV^U5zZ7@)9QD<c&VtyuCGe9sgJ2XaVDPc5JcxY-hPEBK3GfF@-PfTJpaCbvgdp$yUd`47vSWj0yL2^P?Hhz0WUvoZvT4h95a7t`rc575`Jy2(0KUF3(IbnP<Wo}_qJZxeuSVL1tMI&}dGkGmOPa{Q6d`n+nY%M=;S94x?Zel@raU?@FW+ORMDrzccVRKSwIBqt2WN%VUWNUt7c5rrQC~QuCVJ${yD`|dCV`3<Fd3-fmJT`7VP)ArKHhU{=Mq+7qJ!M38bxb@#Z#XDda#cw^a7TGdNl00FAwE)lM^rOTW=DP|KT0EMba_)INLW8vRa9*?CTwIsNlZmIZ9PkGEpSOTR7p&IN-%RICOdV0O-M^BJY`W*eQ{tjX<%nWWnfh|PD(H&V{v?OOKeJIb!H<}Y)UgYa!XPpDL!*#BTh$2a9UO+OjTZYb1OJPLSA!0OFdzJA!&V3aw|VnX>eLtU`#=0X+u&scRV&~PCZdaKyF1TR(5k~S2rVAUU*M8KUPgNBsMZ;bVPPeXCx_bZgNslO*>C5Ha2;7V{l1(a3wQWDNcEDL})#EPd0OCb|q&*K}SPaH)u^lV`?~6C1X`kGe#|VM_5@Xe0Og*JtIkRby{IEPIoqTD>7{<d{byOH9uZ!RX1WVby+_xHA!V@RBUNQP%U&?AvtwVMkQ)6PcuF=Xew_rJWNSRVP`TZS$l9oa6&6HHYsT-CL}{qY+rM1dNEE#a8hY+MsHJeKS)wCXmV|PB{E`SHat^BN=JQIBPDYsctd(`Heh@-L~33mBxXi9ICoStA#-y-NmfNxcX@4KYG-joBtl<qWNsrVX;fG>Eh|xHS4Kl5bt5Y{d_s9=Xn0d;NNP=1c~m7}dv0+paeOs4JY+*SLwt5?Za6S8Do%A;KSoY*RZus4L?(4pbtPd?Bs6qOEkb8)G&Md<XHIiMb0K?DHA6~iDl$-0K3{q;d{<{rCP{QXYH~I*aAhrhO+RsERC`uyDK}|tR7+Q5XGeTQI9XU#Vm~=nRdi-=bvsx|Fj8$tCSp`IaD8n!LO@JFN+CN|ZZ}UVLqu?Ib8t6mD<~yxHBf#xMQU|gWHe=0b9r-FFhWv7V|+6tHDM@2Pc3v!L?J_ZHAPcmXE$zPPjGusaaCbKWHD7RMmTm;COtcQDJ6AQa9)0LDlvFMYCbSfY+rF;VpT{{WqMF%HfvLJM>KvvJ|;;jML#zrGDbcnQF%;EF-A3ONO?<bVJk!?DnLVVGkaPka(HEPP<(J;M0a&aBVa~HXd`PYO>0kDOHo8rPi1O&IBIt*Pi99=Dt%^3GIn@4b4DgaJXK0QM1DkbKyD;aU}jQIWNJ-0YD+&?ZgM$7X(LH6cta(8KS?osR%k_8YI8<;A!8;oc5FE_M08O~El@{hdPrtwQ)zW}Fmo_)ZC6TqXj(#EQA}52Mp|J>eL`PTHdAa*VR>sQadk&ZR%s(pL?KdHF*0g#EjURdaakxac1uuMUp*vkNkvjSXDCKIb8${kHD5<ZNl-p>Sz#?rMP_nYe05MjNI^<ZS9CK~V|GY#OKo6ILQ^?7YF<ZJLT65RH$7l(C{%D~YbYdsX>E5<MN2(5PghB8K|E4OQF>WDQ7tWeNjO+7YhXz~bXQU;W;Rt(K5l4mYjSCHQYKVjVs~d{IXhlNPB=v+cu;g;aCSXVEjwyZb#qE9X-g?2Uq~c+M=DS}R6<Q@epN<BZZvsscVTT+F-m(YQYB4Rc}i44JxXYBH#uTLP&PX$Ax$GtUotpHQfPE3HCJzLcS|EZAznKrH%T&bRc%sNFltwCKUY38J19^>T1Z}3UwA(yeKdAyH#anVRdZe|aZe*{bU8arUSBXZVry)6cX)nOJa{!{V^lXpN-bV;AxUUbG($v4F>-cIU^gj9KT|>~DOpBUBy3AbG*e$aXKiR~CQd6?L|!FEI9hLZF*R9WV|F1>F(GGFJy%L&At+ZNWJ+N-CNniraA8zYL@{1OQcHYpS374tb$miYQ%XBfDkXVuR%vNfMMiHsWPWBbY&mjsSx-<fXgN_yH*#@5R7FKjH!(6`GfZ?>cx+@@ZB=PwEofv>LwYiLb5~VhV|FEAUO^@?b#+2>O*cz!Nl0c@K}IrkVNz*qYjQ?Gcr!_CQ9?;DP$*X|FnMHWbWDCRb!cQ-Pk1pnW-&{0GbC$PNpxs6OL}-RG(1#ycP4N~Wlu&kJ0wCSen?VuB{olMYHL(sGc<8GDt$FAYDZ&3WK}9oOnG@QQ8iOENmX7_b0%d*P;g3FDNc8IG;Jj{LsnNdUTSw^a5E!2UqnPrCVL|=VkL1UA$=+^b4Vy@S|nLmbs;@>Wj}5yDq}5PW@vI`C@LjuXGv-(SWGJ_d^tTWRBuT#Z6-8zb#7rtCOuDHJ4H7^CPZp}MPW;EPIqQ%K0PvJLNk0rVPiOIdV5H3Qa3(+YjQGPd}U={bxdAmEps?zQEeo0HDpaPBx_brH*RS-USl|OL{oTQY*u}9WLipAKTvlzV0KV8H)%;!d}m*BBtA2GIBs-#JXvQ_QYv^<ByU=JT6|AsLOozRG&y&2c{M(Ea&AOoV<>TWG-P2mW@lM$c5_iiD|mNgC0AA?Hfm^JT1{?EG$vFsYC9-GaVcw4MNxiWQAJi@bWvDCUO!hhYdm#!U_xqaK5KDGIe1h>Wl&*SKT}>gH7h1SK_NR{cSmGrV0b=gYE^nmR()C|JY_R@Vst|?Q9yW5NL6)aOfXPlT6HsJL1b=MF>Fawd2nbuT0v@iMKMu+V|hthell};SZIDocq?*YduT~Dbznz9a(Q||J1bs1KXySRbya+KaZYSwA#7GjLoh;6ZfPlAHa$pQG%{5~SS3|xT53!@YeZT|JVhoUMsQ?Ib39*HZC_}1R3?5kZ);0%d?7tYPggu`D|mV+Gg%>Oa4JkVXJ|JnEmS~pG-ym;YBEk!By%NHd`@#gBynMMM|E>odNXf%bvI!#PCqz8LrXhtRY81sUprz#Ep>83R$4btPDf37Q#B}dCQWWIZZb9@d2ds6eOEznaYa3MU`uImL~n3BQ8-K_Ic;P^DM5Nfcz8T(eRFqAF+N30d@3U=Vs&&#XJ0UFZhB61JuydYbYv?}V|q+?Q+`rjG&U(;XLn~*GJHQvaBNX|d2lIuQEovsXE;=3bSiIAeN$FNA!#cxG$A*8UOsMUIBqI3IA<$naA;9QW+_ipWKL67F(pcCax#2-BTi&sdnh#|N=|AcF*HphMnGA8OnXs$YeX?MD^o&JDj_joF=#e8XDeP=CN@}TJ2QMnaZ+G<X+cVQU~MKTY%zFxO*t)WH+4Q>KTkPwb~r{UG&fB#YjZLrJ4{JlV^2q8CSPq}RdY6Vb~#pKW+8fcDL_zecuRF}d3SDDBUVCHDq>+XXEro`S20FMDp+A)Hg-5FCNMNIUO8zsSz&c(KRY&hJ#|uADs?zyLQqwHGc7k*ZFeeSQA${5S9eHJV?j?*dT?5IMj>fQKUr5Zb5~JJUp;qfOm|IHEod@YOH6HAK3PX|UVK7WT5(o#Pf1BUa5*b*PGCkmMn7tGOJhZHY)(9Waxq|FKw?!uOh`3VdTmEqS|(sbMoS@WQcFlgKWigLdrnGcYDGDCc};hGbxU<2VPs)*M>k?AByD;nb0u_nQ(7%ELQg|CN_crUYdAqcDN;*tMOkiZU_e)GB_?J@WKumlPJ41}b4pe!MJ8V;N_<o%Lqb6%L0N20UT|<`Nj^YvPEjf~U~q6|XEH%eD@JxICVf6FZf!CxUNT90R!DAUNilqROekq?X;OY~O*tq{dtgd)C1x=zCU!w2QD<m5UtV-rK6@&3SW!lCZBQeAYdj$(WlKhDPh}==P&-#NUo&w>DrP23N-IJ^V>mlbD`rACPEcNHV`+O-MJ;7nZdXT8PDgBJGctEeP9`BGEh$fOW;b$LNn}V`K|3a7BR6+4X+upvc2qbqAu%vSWhriaR#hfJWpH*RJ1HhZKVD`(N?$;8Zbf-_P)KJ!O;vF>PktmTB|UshM|xsfWKd#pJzsG<cUd8MBRDrdB}ge@WoscNWqEZwadK=oF*0jbcxHWKD?d>&Sa3#lX*_siY$-TNZ7NDJQXz0sODkY?NIoP~dU|1DIcR-nFiw6nNKRHnPkn1bWld*rX*5P;Q%_?~Rd6PKRyA;8DnUaqDPCVrGc7xAK0;+zPGN6zM>J3=OGjWkPk3W`JaKDIW?*<tUr$eMCn+f|WMyM-WMwEKb#!JeI3g(uDGFajGB-J2L`6eMQ%O%wUsF^?P#`@ZF*Y~~Uqv!EIbTy$K~zN`Js^7uARr(hFghT6B5YxEbYF9HWpE-oAT2R0AR=>UZ*X%WIv^-1EFdCfcyMKMbRs$+PH%2yeJlzfARr(yIv{%@Y+-YBUvqS2a3VS&Eio)0B6DbOaC0I$ASfv;AR=XWaAk6IB03;WZ*FCMEDC)JUqv!EIbTp!LQF|RUqMGjPE;U0AYo@^ZgdJ?Uv_13b7^mGUtb_SAR<LFI5aJDWo2Y7Vs&I^WppiPbzyR3A_@u$WMyU`Uu7~kIbU;hWpF5OVsj}v3LqdLAZ2)PY-wX@bRaz-UuR`>C~snOEFdCtbY*ZNDGDGUARuXGAZ2)PY-wX@bRcPSAZ~ATAWm;?WjYEVARr(hARr)eWps6NZXjuHbSPzbaBOK~X>=fOav(4%3LqdLAaZ4Nb#iVXX>N2VUuR`>C~snOEFdCeVR<4fATTK)Z*m|oDIh8!GBhA7AZc!NC|_q~bSQ6Pb1WbtXm53LA}k;<DIjlhATTKk3JPRpW*}c>GB-J2b7*gHb0}|Ob16CsARr(hbZ>WVAUz;oXJvFKZ(?&SAR=^ccWxpqAbWi&Aa8OYdwmKZARr)eWps6NZXk4ZaBO8Lb98bjc42IFWho$LZ*m}ZVQh6}AZczOC|_q~bSQLhcWx{oB6V(TZ)0m^WM6Y=Z*X%WEFfE5DIjlhAX{B2DGCY-WMyU`Uu7~kIbU;SY-M9~Wn^DvcyMKMbSQ6Pb16CsARr(hb7f(4AUz;ob7f(4C~snODGDGUARu#eWpE%pAYWxNH#uK(bY*ZTZ(?&P3LqdLAaitKbY&ntAYVl?H#uKZR6$flTXSV$bX^J{ARr)VW*~EPWpE%pJs>b3Z*m}WbY*ZLJRoUqbSQIlVRU6KXJvFKB5YxEbYF9HWpE-aAT2Q|DLM)uARr(hARr)fbYXO5AUz;^B5YxEbYF9HWpE-oAaitOa4aAqb7*gHb0Rt*C@Cx;B4v1RWpZ>PIv`GOZe@K6ARr(hARr(hUqv!EIbTy$K~zOsb7f(4T_8Omb97;JWeOl5ARu#eVRU6%B5YxEbYF9HWpE;0AUz;+bY*Y~ARr(hX=WgEbY*ZLJUt*^MKU)zUqnSiNmEHrPG3`0MNm2lARr(hARr(hb97;JWm_V1Xm4<HB3&RoAYWxNH#uK(Xm4<HC~snODGDGUARuXGAaitKbY(7QWppSaWq5F9a&#goAZc?TPH%2yAYpD~AaitOa3DTCAYVl?H#uKKMMFtbNl#8+Q&dGzItm~lARr(hARu#SZ*X%UJs@;-aBO8Lb97;JWiDrBbSNToXm4<HA}Jtmav&%vDGDGUARr(hARuIKZE0>{bY)~9Js>CwARr(hARr(hARr(hY-MgJb7*gHb15J`Js>g)ARr(hARr(hARr(hVQyp~b7*gHb6YT7AU!=GB1uC<UqezwK}}y%NKa5A3LqdLARr(hARr(hAYpD~AaiJMaC2KRT_8O@AR<{oQchn}R8LYxA_^cNARr(hARs9UARr(hARr(hb97;JWm_U;cyMKMbRu0KJs>CwARr(hARr(hARr(hB4}x6Xd)nKW*{P2K~hd%Q&dk<MIs<+ZXk1LZ*X%UVQyp~Zf|rTWN&S0Zees~WFTd1b7deRY;SiW3LqdLARr(hASntUARr)eWps6NZXk1Xawv0jVRU6KXJvFKB4v1RWpZ>PDIjlhAR=sUcOoeY3JPRpW*}i_Wo~pRZ(?&SAY*TCW@%@2a$$67Z*DzKZ*FBNItm~lARuIAY#?KAZf0p`b#h^JX>V={ARr(hXKZg`VQe5@K|@qYPfk+`ARr(hWq5F9a&#a)AYWxNH#uK(Wo%_*bY)~;Wq5F9a&#zfVsj}9ARr(hUqM4uNl#8wAUz;oMKU)zUr0$uNMAuiR7p=xQy^((AZ2)PWpZ>NJv|^IXlZ9?A|PdKb7df3MKU)zUrbL|UqM4uNl#8w3LqdLAaZ4Nb#iVXUqv!EIbTp!LQF|RUqMGjPE;swVsj}93JN12Q*>csY-J#FVRtQTZ((F*av)`HbaHthaBpdDbS?^CMKU-uUrbL&Nkd;jM@3FlAUz;qXJu}53JPRpW*}=}XJ>M0V|8qFb#i52WimK3UtwouZgePbVsj}v3LqdLAaZ4Nb#iVXUqv!FGha+kM@d6pK}SVSR48v^b14c6VP|D-bRaz-Yhh<+a%p3AY;<*UWnX17I5S^iXJu}53JMA%AXqdxGCCkdGB`9Rb0B1Ia&2L5bRckYWo>D7Z6I)RWo2z}bZKvHEFf}ab98cPV{~O?AarjaadmHWWpf~5bRchXAYo!}c4Z19AZ1}=XdrNMZ)A02bSHBlVqtS-AaHVNV`VNNAXI2+b0BMFWpHyKbZBKDa&L8XWgua0WFTdDaAk6IAaiAGWn*+{Z*CxIZggQ|bS?^CSTs2@UqV4sMPETjMNU*8Js@FcWo~o|Ur<s-MNLptUqwzqLQF+OAUz;da&=`2Ur<s-MNLptUrk9)Ur<s>Lq%UwK~zakAUz;4E-(rT3S?zwAYo@^ZgePbVsk7YV{dL|X=inEVRUJ4Zaq$KZe=Mt3LqdLAY^51AY*TCW@%@2a$$67Z*B@8ARr)eWps6NZXjP+G&wR~LP1kSUqMGjPE;swVsj}93JPCec4cyNX>V>{Um!goB3LvzGA(5?I5aJBb#HWKEoWhLWn?XIa%E+0aCB*JZXyZ_3L_v`H8e0fAaiAGY#?KAZgXXFc42gBc4cgNAZcW5Wguo{a&&2IX?kUHAYpVMbZBKDWMO$NWo~33V`yb#YjAIAZgegRWMyU`UwAb%FkfMCaAj^}Uuk4)WnX4xa&&2IX?kUHUvpt>WhifAb1WcXV{~b6ZYeqnARr(hX=Wg4ZgePbVskEMWppSaXm53LA}k;<DIjlhATTK)Aw3{6Gdc<&ARr(hARr)eWps6NZXjV}bZKvH3LqdLAZ=lCYh`pGJs@FYbZKvHE^}pcWMyVyb!>DfB5h%EYh`pIEFfE5DGDGUARuXGAZ%rBC~aYKYh`pPAU-`HF)%s`ARr(hARr(ha%FUNa&91DV{~b6ZVDhEARusZX=7z`AUz-`C~snOE@x$QC?aiPa%*LDA}Jtmav*zsDK2MabSNTla%p2_b0R4qZ*m}eeJKhcARr)VW*}*9bSQ9gX=7z`E@x$QC?ZBhQdCJyNm@lxA}k;<DIjlhATTK)JRmYPItm~lARr(hARuyObairWAYo&4X>V={3LqdLAaHVNc42g7AUz;&VskEMWppSaaB^vOVRU68DIjlhAbWiZARr(hb7*B`AUz;(a%py9bY(7QWppSab7*B`A}Jtmav*zs3LqdLAYpc4X>4I)Y-J!lAZ=lIC@?G_X>N2Vb7*B`E@x$QC?ZBhQdCJyNm@lxA}k;<DIjlhATTK@3LqdLAZBlJAa8PHWpW^CZXj)8a%*LDItm~lARr(hARuXGASenTARr(hARr(hARr)Vb7^jKbYX5|WhifQWMy(JAZ%%KbSVlTARr(hARr(hARr)NZe$>AWo{^Ma%5$4DIh*QATtUeARr(hARr(hARr)NZe$>Da%5$4TQFT9Jv|^IQ$<WnA_^cNARr(hARr(hARu9GWFT*HWMy(&F<l@%Js=`RMN(8rOi5ZrQX&c<ARr(hARr(qItm~lARr(hARr(hARu9OVQFk(Vr*p~Ej=J@VR$GoEFfuabSQ6fWMy(&GF>2Vav(4%DGDGUARuOMav*hXX>=fIZXjD>V{~b6ZZ2nKbSNTbVRCI{av~`#ASx(fV{~b6ZZ2nKbSNTdVQyq|A}Jtmav)n>DP1}WARr(hARr(hX=WfO3LqdLARr(hARr(hAZc@HZgX^DZewLAb#7^NEFf%Yb95;RARr(hARr(hARr(hVQyp~Y-MgJb#7^NDIh*QATkOdARr(hARr(hARr)NZe$>JZfSH|FkK)$Js=`bNkdCjP$CK-ARr(hARr(hARr)NZe$>JZfSH|F<l@%Js=`RMN(8rOi5ZrQX&c<ARr(hARr(qItm~lARr(hARr(hARu9OVQFk(Vr*p~Ej=J@VR$GoEFfuabSQOhX>?mMT`4IbX=Wg7Wo{^SZfSHWAU-`HGazMbb7deg3LqdLAZcbGVRm6@Y++(-Wgt8~ATT-#ARr(hARr(ha%FUNa&91DV{~b6ZVDhEARujFa%*LDE@5zRWo~3BTOw0MOiUsyAR<OZQdCJyNm@lxA}k<nX>KSnGAtlrc4293VPb4$DP1WFARr(ha%FUNa&91DV{~b6ZVCztUsyFXFkeDJQ$=4vM@3FlAUz;qXJu}53JMBjWo95@XJu}5C~snOEFfcVZf0p`b#h^JX>V>lPH%2yDLM)uARr)QWo#g0Z*FF3XLWL6bZKvH3LqdLAaZ4Nb#iVXUwAb%FkfMCaAj^}Uuk4)WnX4xa&&2IX?kUHUvpt>WhifAb1WcVST!^-UqV4sMPETjMNU*GZ(?&PDGCY-Ute}*a&u{KZeL#@Js=`jH8e0Scr-aOEoo$IWi4i9a&&2IX?kUHEpuUPWi4iGVRRx2"""

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "x540_output"
output_dir.mkdir(parents=True, exist_ok=True)
main_path = output_dir / "main.py"
main_path.write_bytes(base64.b85decode(SOURCE_B85.encode("ascii")))

actual_sha256 = hashlib.sha256(main_path.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (actual_sha256, EXPECTED_SHA256)
assert main_path.name == "main.py"
print("Wrote exact competition entrypoint:", main_path)
print("SHA-256:", actual_sha256)


Wrote exact competition entrypoint: /kaggle/working/main.py
SHA-256: db2707d6a349521dd165de280038440cfcfdafc50a9739ffe1bb70d5b62dfdf7


## Copyable source

The next cell displays the exact emitted file. The downloadable notebook output is the canonical competition artifact.

In [2]:
from IPython.display import Code, Markdown, display

display(Markdown("The exact emitted root `main.py` is shown below for inspection and copying."))
display(Code(filename=str(main_path), language="python"))


The exact emitted root `main.py` is shown below for inspection and copying.

"""Kaggriculture E284.

Base: the supplied ~3000-score market-conditioned mixture-of-experts policy.
Added layer: a conservative seed-budget guard for atomic PLANT validation.

Attribution from the source notebook is preserved.
"""
import base64
import copy
import json
import math
import zlib


_ACTIONS = json.loads(zlib.decompress(base64.b85decode('c-rk<O>Z38k^C<_^PpyT^J8xusl5`+8448ThCLt#1K0}#hW9YLw}t=v%4C03)r*Xb%zQ<11m0TBR@M7{nURr^KmXs!fBpK~KmYdI$$$KO^266pHy?iaeEsGA>)qz$;q>J1zy9}M|L5CZzJ2`XufP4{Z~y!4^Uo(AKRy0c`|!ipKmT(5)2AP=Z%$56-rsIdPS2XJzkJ+mJ`euzWwZJ4?d$Ew&Gr4s>BZ#hA2&C*Kb@Q|4nO~Vcl+V%`}^bnIDdHf*XgihpFh3-<JZr}H!TKz`}t(M`Evi*)}L<g?ms?$I(#+xFdm36o12^CTbI+f?jJXJ6==xtwZ~7>sXz^wyw03G*uz6h9_M5+>g(=T<XxX{uHSF0@kIUE{|E54NxR8gcmHKLo=rO*zx(O97)E{F&6M#ocZ4_B)At{j$MyZ@Zn}u3-;GxfT)Jn|MfBzN>vR#di}Mfv-Wj8BCcR@**$&QlfG4AL?BDzA-O}9u=xJvTx*nR#<8ZYv-H*cXSMGFy{f8z8?1W|oleg@~9*o&wIGP!2f1}UXZrth6jh;K*dB-7ar^#5C3*m4Bo54I<`Pnk+f;O_~(D5g4-%@=n<!}6X1VgwxVZa=D^QI5t;T?w$-_G7I=tFGaj^kc=@a~s%()&K2PI#9N?EmlJO<kYset3b$PHvT@VNE)RY2X6s^VI3t8ri<j-h!z;LVntq5q(<l{`Tf(^Zx#qKWy&qKiz!#%lJ&_G<fNk1eQqp9W%|r{?;C}$J|2)M`ZG2<0@Z21T4T;z5WC9JMH5t@7=ogU(qH3=3Qex4vcWHa5H`eFh<~>z`fcn?U0$w`!MXS*GG2%fny&qNSUhwKY0&iV}U-o4`d#JXg?PGQM<`W2g)8)$@Wz?5cSRd`6r%E&Gl7)C--sCTMjr6z_>p?vNZ<%&EEni#J2R?7kZp)suJAnnGNf=r}clDeD4DrYNdkQdBecA723mj45Ke5u=uxA@9q{MHPUg&u3G7k%-9cyw+;@h_}wYCz0$eR5F%u~bSKdJwPbA2i#9VX+>SA!$cWSAwLf4sQOkpv3<-OTF8U+t=VGG-y>bS_hYTYJ?-a`Vet@gD$G$!GclcNx!0KV@*pYV_!gnF3br?V~Lh|jq8xNMba~i&q^cn+rN-Y5CSwtBSB!)^w+E1eDRY#Ux@W$ADyuSOB)v<mzegG{Hqu6Mu4t+@u(Qqs(6oYnf+8E@4Oi%(+_@Ey;_Vm`+pd+K|Fet;3^5GDGuZ))6agFW=<rwjl2mSa&bkz*sH!#q33}(vFp!XSgLngx9KDebxy_pSfkFA14YtM4p{qFK{ySK*F8WSHE5z}gC#C*BGyWak=xx4!_uw+sQliMNl?U06f)*WtQ4Ky0Dc$fi=dPWc`baw{K$SD++y-RImAqzasWnvAjlOfg=Lzp<IQd%E}4~Hw=f1HN1?Qi74rp4sfj-k$ycQPG;<SMZG9s2sInOTbveR^tWCdAs_a)j_EaJ3_k%fN)Q(ed5_TQ3zRJC9&T>!Mw=d`KLhLcGke5s0CJj#tH*roOrd#$>J)hE@zN!R_ts&0|^&G_7`j+)dE8^YOz;+SWVz^SHNxucf0?GY1((q7!FjI@H#!ARD~nSqZP@!$gRn9E>Gj2<#t_snpw0NKFy*L$vsqdS6R0REZv@`z|$hR2%(tkunLrZ9b*(&W)8H5o;oVO{d}5SQ8P-IN(f;6AQ|_Akg`Cy3x@$JuNcZfDJSHqz?)>by@&<nmUtXd;y<iV&2YWMV$w;Wf!T4-8eS3@+~izu^KkVM9P%r=*r=(39ljsh@k9UkZo$T0bIZAI@8e##h>;D$oRUw$&DX!NWnNSmXXPZ8M}UZcFaPM?roZpXH&ftNPz)lhsaU@?E~|F?POFl;`kx>L^7J4{L+f`z;+N#FxB(PSq^J>1;YPOG=NQZ`Kc3+Y*9h~mL)y}+i&f9>1CU>A?NAJf*<^mX7<FP7r+28ZH>@4w_Rq>HTxGcc__$DI@koBK^AUiX)CW}qk&-D<T<Cb0E~I!Y>SMu0b7IN&m6K~Ky|KXeE3;P|NfCmOEP#h<q^mU-!G60-UrEDgnf`Y3-Ahi8zF(5A)&Cf3E<B{o<|~m1Kh4PgRN7p-{Hy{qQREw4~#nE%u)FxO(=vDQ@Q<$SxVVFkAQ0cuFJy&?+xA`Ki&LsxAH7Msr{v&=~n^CcU$`DeG3sfkdXCZyk-@hVzB7KumT5fD^4GH*<mGuEiZ`Mgz|HOVJIQ>bWS-caAY6N^r11_041$8&!Le_LJx6ahDV7{Iu&!HLSwn~0n^rPLlRixTEvHE%{c>e-J(YAIgX@(TKTosC{4y(TiPtUWcXgOEp(rP8^f8wJGHOOzvY}0$?;8m$R?b7u94}R^=cj226Xmf9<AOUw&KBs)jC#vhX+mzg;v~&Bu-)HW!Ao+g><H_c+68SIExza6n7tMI&8-in;hnqR#kap5UKCo9x2tV9uPG6*&s%kc60?Vd}W;|<ae4v)g10QFgrZ&oX~S)kxkuMcimNH#fTY_ey*f%@xweKY-P%xY$$cGl7z|p_;wW^Ls;#P^i)cntfT8JKe549cqM_fLJ8)f-a{3GIJ7v=q8eT=<55t+9_kB+qq;=>d)KIlqbw(jpslazf&^kifN$3L#h@X`A$6ohTB|q1<8xjtMea#xlc;gzsZik5Y2xN+Xu;03wXoS!s~)sAtWaOOwT=^XGl3+|@=jsGOehu;kx~IkD#TZStcpYsOtG62(rM4(Z77pD(}&_TK0~YWr_O5_L|}rwhGxAD*4y@+2bqw4{<QB4>Y%)uuzA}y8a5X~)-hifbwjszl?b}rN)`vn1duKsYC^}!+$>=Fs~m?MsGf2{V+?WXLuL4Efc!{fPCQ(P9#Yl~fUFAJCO=gYgUSVYTA!F<-L$m~lgiB{_iKl55o4{@OfCLscHZD-TMN_0Zm$9!ual$7$h}A+lPos_M)z$j;_jlY=xiW7)-b^kw0H6DA;F$y=wACHfG-@sf<brO)XO$RvGOk|8a;)l6hk34iim{V{Iv;KE*AcpXY0t7eZ}*_a6GwJ=MS0iKY_+TX4RhSHmuIrZ*`3x>vXQ&FQZIM(oCAVN(g%-1WkM3&D+9WjqAWZlkRaoGF)=7bGMg)oknH_OkEqO_RM)WEFg<n?(q7dqnH%UmV)T8S<Mm6VH@Jksmt3UN~c`Q2(iOx)fuVHfIP5quM=x6G|2?0&34YUzlKsB%7Gga&YK?fC>a9nqyd$h<VqhjOtSgWBjLMr-U3HuUN(oJrETi&v+JOnzGNrv1c=ZI^^8MUMl|p*;5U~M;gB|v2dT{6XQ--!b4-(fMVHZ+netzwGgZGP%~hXExM7qdZ{S&yVX7L3l-1hCq2qiTZd**)k{S8|a1ak2@P<{ROtZVXl7VFn!bV`k*B69&?r2toR!EHJA%K3Nhq$VhrDa3d5HxG%HLe(D6Vp9xB4%>sghc=~V5&#lAPy`FU+dIP&5%0C=z?v(@RId#El81Ji^~!wbGXL@*LFLg=&KkH;YUW+cZi9WDGFph8bW$kmM_!RueGRH_x!x7tRRZJXtbu%!VKY+au!Ld<FaoW=Euf7e21}snEQCzUb%xMRcrl35r;X}lO$`j(5fj25}2o%${Nhb6^sKmM{uStF3}TnIea;6&!q#=%#XnU#)BFxH6h@f6{=^Y<QXy|_M&J^ybC6&Q2RLUsDo&KNO7QD07<S<U1Yhc<HS~wj)j

## Entrypoint and package checks

The emitted root file must compile, expose a callable `agent`, and remain the primary `main.py` output.

In [3]:
import hashlib
import importlib.util
import py_compile

py_compile.compile(str(main_path), doraise=True)
spec = importlib.util.spec_from_file_location("x540_emitted", main_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)
assert callable(getattr(module, "agent", None))
assert hashlib.sha256(main_path.read_bytes()).hexdigest() == EXPECTED_SHA256
print("py_compile: PASS")
print("callable agent: PASS")
print("primary output: main.py")


py_compile: PASS
callable agent: PASS
primary output: main.py


## Full-season official-engine smoke

This checks execution reliability from both seats for 720 turns. Its rewards are packaging evidence only, not a competitive claim.

In [4]:
from kaggle_environments import make

smoke_results = []
for x540_seat in (0, 1):
    agents = [str(main_path), "starter"] if x540_seat == 0 else ["starter", str(main_path)]
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 540540},
        debug=False,
    )
    env.run(agents)
    final = [(state.status, state.reward) for state in env.steps[-1]]
    assert all(status == "DONE" for status, _ in final), final
    smoke_results.append({"x540_seat": x540_seat, "final": final})

print("Official Kaggriculture smoke: PASS")
print(smoke_results)


19:00:07 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [Errno -3] Temporary failure in name resolution. Falling back to local backup.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

## Reuse checklist

1. Download the notebook output `main.py` from the root output directory.
2. Preserve the final `agent(obs, configuration=None)` entrypoint and verify its hash before submitting.
3. Compare any modification against this exact X540 source and the unchanged X492 control with paired seeds and both seats.
4. Treat the technique as a hypothesis to test under the current meta, not as a copied opponent policy or a leaderboard guarantee.
